# CS781 Capstone - Phase 1, Submission 03
## Single-stack prefix-hybrid legal language model

This notebook is an independent Submission 03 workspace. It preserves the reliable official-data,
checkpoint, recovery, prediction, and Codabench packaging infrastructure while changing the model
integration and training objective to a **single-stack prefix-hybrid language model**.

The prompt prefix uses bidirectional self-attention. Continuation positions attend to the complete
prefix and only earlier continuation positions. Unlike Submission 01, the prompt is not encoded with
a purely causal mask. Unlike Submission 02, there are no separate encoder and decoder stacks and no
encoder-decoder cross-attention.

**Academic-integrity disclosure:** this notebook and `prefix_hybrid_lm.py` contain AI-assisted
implementation and experiment settings requested by the user. The architecture-origin metadata
records that fact. Technical training and packaging can run automatically, but the notebook does
not set the student's personal ownership or scratch-training attestations on their behalf.
Review the course rules and declare this assistance truthfully before submission.


## Safe execution map

- `auto_train` (current Run All mode): start fresh when no latest checkpoint exists, or resume
  a compatible checkpoint and print the restored optimizer step.
- `fresh_train`: explicitly refuse an existing latest checkpoint.
- `resume_train`: explicitly require a compatible checkpoint; never restart at step zero.
- `inference_only`: restore the frozen tokenizer, student source/configuration, and selected model;
  only the official test prompts are required.

`Run All` is configured for a 10,000-step training and delivery pass. It preprocesses the
bounded corpus, trains/reuses the 25k tokenizer, trains on both T4 GPUs, generates official
predictions, validates `submission.zip`, and exports the complete model/recovery ZIP. The separate
representative GPU smoke test is disabled; automatic model-contract checks still run. Attach the
official data and `prefix_hybrid_lm.py` as Kaggle inputs first.


## 1. Environment setup and reproducibility

In [ ]:
from __future__ import annotations

import ast
import csv
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import platform
import sys
import tempfile
import time
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError as exc:
    raise RuntimeError("tqdm is required; install it in the Kaggle environment") from exc

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
        temporary = Path(handle.name)
    os.replace(temporary, path)

def atomic_write_json(path: Path, value: Any) -> None:
    atomic_write_bytes(Path(path), (json.dumps(value, indent=2, ensure_ascii=False) + "\n").encode("utf-8"))

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

print({"python": sys.version.split()[0], "torch": torch.__version__, "utc": utc_now()})


In [ ]:
def _discover_attached_tokenizer() -> str | None:
    search_root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    candidates = []
    for tokenizer_json in search_root.rglob("tokenizer.json"):
        directory = tokenizer_json.parent
        if (directory / "tokenizer_config.json").is_file() and (directory / "tokenizer_metadata.json").is_file():
            candidates.append(directory.resolve())
    preferred = [
        item for item in candidates
        if "submission01" in str(item).lower().replace("_", "").replace("-", "")
    ]
    choices = sorted(set(preferred or candidates))
    return str(choices[0]) if len(choices) == 1 else None

def _discover_student_model_source() -> str | None:
    filename = "prefix_hybrid_lm.py"
    local = Path.cwd() / filename
    if local.is_file() and not Path("/kaggle/input").exists():
        return str(local.resolve())
    search_root = Path("/kaggle/input")
    if not search_root.exists():
        return None
    candidates = sorted(path.resolve() for path in search_root.rglob(filename) if path.is_file())
    return str(candidates[0]) if len(candidates) == 1 else None

@dataclass
class RunConfig:
    experiment_id: str = "phase1_submission03_prefix_hybrid"
    seed: int = 781
    execution_mode: str = "auto_train"  # auto_train | fresh_train | resume_train | inference_only
    workspace: str = field(default_factory=lambda: str(
        Path("/kaggle/working/CS781_Capstone/phase1/submission_03")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "CS781_Capstone/phase1/submission_03"
    ))
    input_root: str = field(default_factory=lambda: str(
        Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
    ))
    train_files: tuple[str, ...] = (
        "train_acts.json", "train_high_courts.json",
        "train_supreme_court.json", "train_tribunals.json",
    )
    test_file: str = "Phase1- Test.json"
    test_filename_aliases: tuple[str, ...] = ("Phase1- Test.json", "test.json")
    hf_repo_id: str = "Exploration-Lab/CS-781-Capstone"
    hf_allow_patterns: tuple[str, ...] = ("*Phase_1_Pretraining/*", "*Phase-1_Pretraining/*")
    official_file_overrides: dict[str, str] = field(default_factory=dict)
    training_schema_overrides: dict[str, dict[str, Any]] = field(default_factory=dict)
    test_schema_override: dict[str, Any] | None = None

    # Keep these matched to a declared control unless your own experiment changes them.
    validation_fraction: float = 0.05
    source_mixing_ratios: dict[str, float] = field(default_factory=lambda: {
        "acts": 0.25, "high_courts": 0.25,
        "supreme_court": 0.25, "tribunals": 0.25,
    })
    # STUDENT: choose before preprocessing. The official document permits a compute-bounded subset.
    corpus_document_cap_per_source: int | None = 8_000

    # Submission 03 intentionally trains a new 25k byte-level BPE tokenizer.
    # A restored tokenizer in this workspace is still reused on resume/inference.
    tokenizer_import_path: str | None = None
    student_model_source_path: str | None = field(default_factory=_discover_student_model_source)
    completed_notebook_path: str | None = None

    architecture_record: dict[str, Any] = field(default_factory=lambda: {
        "family": "single-stack bidirectional-prefix / causal-continuation hybrid",
        "substantive_difference_from_submission01": (
            "the prompt segment is bidirectional rather than globally causal"
        ),
        "substantive_difference_from_submission02": (
            "one shared Transformer stack and one segment-aware self-attention mask; "
            "no separate encoder/decoder and no cross-attention"
        ),
        "attention_semantics": (
            "prefix queries attend to all valid prefix keys; continuation queries attend to all "
            "valid prefix keys and continuation keys at or before the query"
        ),
        "parameter_range": [50_000_000, 300_000_000],
        "student_model_config": {
            "vocab_size": 25_000,
            "max_sequence_length": 1280,
            "hidden_size": 384,
            "num_layers": 24,
            "num_heads": 6,
            "head_dim": 64,
            "ffn_size": 1024,
            "dropout": 0.10,
            "normalization": "pre_rms_norm",
            "activation": "swiglu",
            "position_representation": "rope",
            "tie_word_embeddings": True,
        },
        "student_exact_parameter_derivation": (
            "For V=25,000, H=384, I=1,024, L=24: tied token embedding = V*H = "
            "9,600,000; each block = attention 4*(H*H+H) = 591,360 + SwiGLU "
            "[2*(H*I+I)+(I*H+H)] = 1,182,080 + two RMSNorm vectors 2H = 768, "
            "giving 1,774,208 per block and 42,580,992 across 24 blocks; final RMSNorm "
            "= 384; vocabulary bias = 25,000; total = 52,206,376 trainable parameters. "
            "RoPE and tied output weights add no trainable parameters."
        ),
        "student_design_rationale": (
            "A 384-dimensional residual stream reduces the information bottleneck while retaining a "
            "deep 24-block shared network within the parameter floor. Six heads give a conventional, "
            "tensor-core-friendly 64-dimensional head for full RoPE. Pre-RMSNorm, SwiGLU, scaled residual "
            "initialization, mixed precision, and balanced two-T4 data parallel batches are used "
            "for stable, compute-efficient scratch training."
        ),
    })
    innovation_hypothesis: str = (
        "Relative to a globally causal decoder-only control, bidirectional attention within the "
        "legal prompt is expected to improve the model's use of parties, provisions, and procedural "
        "facts that occur later in the prompt, while causal continuation masking preserves valid "
        "next-token prediction. Full-head RoPE is expected to make relative token distances easier "
        "to represent across long legal contexts. The predeclared comparison will use held-out "
        "continuation loss/perplexity, BERTScore F1, ROUGE-L F1, BLEU-4, repetition diagnostics, "
        "throughput, and peak memory; this is a hypothesis rather than a conclusion."
    )
    ablation_control: str = (
        "Submission 01 decoder-only baseline rerun or interpreted under the same four-source "
        "bounded corpus, 25k tokenizer, scratch-training token budget, validation split, decoding "
        "protocol, and evaluator; the intended treatment is prefix-hybrid attention plus RoPE and "
        "the narrow/deep shape. Submission 02 remains a secondary encoder-decoder reference."
    )
    student_owned_architecture_attestation: bool = False
    student_configuration_confirmed: bool = False
    full_training_authorized: bool = True  # Explicitly requested for this 10k-step Run All
    trained_from_scratch_attestation: bool = False
    decoding: dict[str, Any] = field(default_factory=lambda: {
        "target_words": 128,
        "max_new_tokens": 448,
        "policies": [
            {
                "name": "primary_top_k_nucleus",
                "strategy": "top_k_top_p",
                "seed": 781,
                "temperature": 0.85,
                "top_k": 100,
                "top_p": 0.92,
                "repetition_penalty": 1.08,
                "no_repeat_ngram_size": 4,
                "min_unique_word_ratio": 0.55,
                "max_word_fraction": 0.08,
            },
            {
                "name": "retry_wider_nucleus",
                "strategy": "top_k_top_p",
                "seed": 1781,
                "temperature": 0.95,
                "top_k": 160,
                "top_p": 0.96,
                "repetition_penalty": 1.12,
                "no_repeat_ngram_size": 3,
                "min_unique_word_ratio": 0.50,
                "max_word_fraction": 0.10,
            },
        ],
    })

CFG = RunConfig()
if CFG.execution_mode not in {"prepare", "auto_train", "fresh_train", "resume_train", "inference_only"}:
    raise ValueError(f"Unsupported execution_mode: {CFG.execution_mode}")

ROOT = Path(CFG.workspace)
DIRS = {name: ROOT / name for name in (
    "model", "tokenizer", "data", "checkpoints", "histories", "evaluations",
    "predictions", "plots", "recovery", "manifests", "submission"
)}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

def bootstrap_attached_recovery(input_root: Path, destination: Path) -> Path | None:
    """Restore one verified attached archive before tokenizer/model setup."""
    if (destination / "checkpoints" / "checkpoint.latest.pt").exists():
        return None
    input_root = Path(input_root)
    if not input_root.exists(): return None
    final_archives = sorted(input_root.rglob("phase1_submission03_recovery.zip"))
    periodic_archives = sorted(input_root.rglob("training_recovery.latest.zip"))
    choices = final_archives if final_archives else periodic_archives
    if not choices: return None
    if len(choices) != 1:
        raise RuntimeError(f"Multiple Submission 03 recovery ZIPs found: {choices}")
    archive_path = choices[0]
    import stat
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        manifest_name = "manifests/recovery_integrity.json"
        if len(names) != len(set(names)) or manifest_name not in names:
            raise ValueError("Recovery ZIP has duplicate names or no integrity manifest")
        manifest = json.loads(archive.read(manifest_name))
        rows = {row["path"]: row for row in manifest["files"]}
        members = set(names) - {manifest_name}
        if len(rows) != len(manifest["files"]) or set(rows) != members:
            raise ValueError("Recovery ZIP member list differs from its manifest")
        verified = []
        for name in sorted(members):
            info = archive.getinfo(name)
            parts = Path(name).parts
            if (not name or name.startswith("/") or chr(92) in name or
                name.endswith("/") or ".." in parts or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f"Unsafe recovery member: {name}")
            target = (destination / name).resolve()
            if destination.resolve() not in target.parents:
                raise ValueError(f"Recovery member escapes workspace: {name}")
            if info.file_size != int(rows[name]["bytes"]):
                raise ValueError(f"Recovery member size mismatch: {name}")
            digest = hashlib.sha256()
            with archive.open(name) as source:
                for chunk in iter(lambda: source.read(1 << 20), b""):
                    digest.update(chunk)
            if digest.hexdigest() != rows[name]["sha256"]:
                raise ValueError(f"Recovery member SHA mismatch: {name}")
            if target.exists():
                if not target.is_file() or sha256_file(target) != digest.hexdigest():
                    raise RuntimeError(f"Recovery conflict: {target}")
            verified.append((name, target))
        for name, target in verified:
            if target.exists(): continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=target.parent, delete=False) as handle:
                temporary = Path(handle.name)
                try:
                    with archive.open(name) as source:
                        shutil.copyfileobj(source, handle, length=1 << 20)
                    handle.flush(); os.fsync(handle.fileno())
                except Exception:
                    temporary.unlink(missing_ok=True)
                    raise
            os.replace(temporary, target)
    return archive_path

RESTORED_RECOVERY = bootstrap_attached_recovery(Path(CFG.input_root), ROOT)
if RESTORED_RECOVERY is not None: print("Restored verified recovery:", RESTORED_RECOVERY)

RUN_CONFIG_PATH = DIRS["manifests"] / "run_config.json"
def register_run_config(cfg: RunConfig) -> dict[str, Any]:
    current = json.loads(json.dumps(asdict(cfg), ensure_ascii=False))
    mutable_runtime = {
        "execution_mode", "full_training_authorized", "trained_from_scratch_attestation",
        "completed_notebook_path", "tokenizer_import_path", "student_model_source_path",
        "student_owned_architecture_attestation", "student_configuration_confirmed",
    }
    mutable_before_checkpoint = {
        "corpus_document_cap_per_source", "architecture_record", "innovation_hypothesis",
        "ablation_control", "decoding",
    }
    if RUN_CONFIG_PATH.exists():
        existing = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
        checkpoints_exist = any(DIRS["checkpoints"].glob("*.pt"))
        conflicts, merged = {}, dict(existing)
        for key, new_value in current.items():
            old_value = existing.get(key)
            if old_value == new_value:
                continue
            if key in mutable_runtime or (key in mutable_before_checkpoint and not checkpoints_exist):
                merged[key] = new_value
            else:
                conflicts[key] = (old_value, new_value)
        if conflicts:
            raise RuntimeError(
                f"Existing Submission 03 configuration conflict: {conflicts}. "
                "Use a new experiment workspace instead of mutating a checkpointed run."
            )
        atomic_write_json(RUN_CONFIG_PATH, merged)
        return merged
    atomic_write_json(RUN_CONFIG_PATH, current)
    return current

random.seed(CFG.seed); np.random.seed(CFG.seed); torch.manual_seed(CFG.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG.seed)
register_run_config(CFG)
print({
    "workspace": str(ROOT), "mode": CFG.execution_mode,
    "corpus_cap_per_source": CFG.corpus_document_cap_per_source,
    "tokenizer_import_path": CFG.tokenizer_import_path,
    "student_model_source_path": CFG.student_model_source_path,
})


## 2. Kaggle GPU detection and optional authentication

In [ ]:
def gpu_inventory() -> list[dict[str, Any]]:
    rows = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        free_bytes, total_bytes = torch.cuda.mem_get_info(index)
        rows.append({
            "index": index,
            "device": f"cuda:{index}",
            "name": props.name,
            "total_memory_gib": round(total_bytes / 2**30, 2),
            "free_memory_gib": round(free_bytes / 2**30, 2),
            "capability": f"{props.major}.{props.minor}",
            "native_bf16_supported": bool(props.major >= 8 and torch.cuda.is_bf16_supported()),
        })
    return rows

RUNTIME = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "cuda_available": torch.cuda.is_available(),
}
GPUS = gpu_inventory()
print(json.dumps({"runtime": RUNTIME, "gpus": GPUS}, indent=2))
if not GPUS:
    print("WARNING: no CUDA GPU detected. This Run All configuration requires two T4 GPUs.")
elif len(GPUS) == 1:
    print("WARNING: one GPU detected. This Run All configuration requires two CUDA GPUs.")
else:
    print("Multiple GPUs detected. Submission 03 is configured for balanced single-process DataParallel on cuda:0 and cuda:1.")


In [ ]:
# Optional only when an attached artifact actually requires Hugging Face authentication.
# Do not paste a token into the notebook. Use a Kaggle secret named HF_TOKEN.
def optional_hf_login(secret_name: str = "HF_TOKEN") -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        token = UserSecretsClient().get_secret(secret_name)
        if not token:
            return False
        login(token=token, add_to_git_credential=False)
        return True
    except Exception as exc:
        print(f"HF login skipped: {type(exc).__name__}: {exc}")
        return False

# optional_hf_login()  # Uncomment only if required.

## 3. Official file discovery and schema inspection

In [ ]:
import subprocess

HF_DATA_ROOT=Path('/kaggle/working/capstone_data') if Path('/kaggle/working').exists() else ROOT/'downloaded_official_data'
INSTALL_RUNTIME_DEPENDENCIES=True
DOWNLOAD_PHASE1_FROM_HF=False

def ensure_runtime_dependency(import_name:str,pip_spec:str)->None:
    if importlib.util.find_spec(import_name) is not None:
        return
    if not INSTALL_RUNTIME_DEPENDENCIES:
        raise RuntimeError(
            f'Missing {import_name!r}. Set INSTALL_RUNTIME_DEPENDENCIES=True and rerun this cell '
            f'to install {pip_spec!r} into the Kaggle session.'
        )
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_spec])

def kaggle_hf_token(secret_name:str='HF_TOKEN')->str|None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or None
    except Exception:
        return None

def inspect_huggingface_phase1_files(repo_id:str)->list[str]:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    from huggingface_hub import HfApi
    api=HfApi(token=kaggle_hf_token())
    entries=list(api.list_repo_tree(repo_id=repo_id,repo_type='dataset',recursive=True))
    phase1=[];total_bytes=0
    for entry in entries:
        name=str(getattr(entry,'path',''))
        if 'phase_1' in name.lower() or 'phase-1' in name.lower():
            size=int(getattr(entry,'size',0) or 0);phase1.append(name);total_bytes+=size
            print(f'{name} ({size/2**30:.2f} GiB)' if size else name)
    if not phase1:
        raise RuntimeError(f'No Phase 1 files were visible in {repo_id}; check access/authentication')
    free=shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free
    print({'phase1_download_gib':round(total_bytes/2**30,2),'destination_free_gib':round(free/2**30,2)})
    if total_bytes and free<total_bytes+8*2**30:
        print('WARNING: direct download leaves less than 8 GiB for compressed preprocessing and checkpoints; prefer a private attached Kaggle dataset.')
    return phase1

def download_huggingface_phase1(cfg:RunConfig,destination:Path)->Path:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    ensure_runtime_dependency('ijson','ijson>=3.3,<4')
    from huggingface_hub import snapshot_download
    published=inspect_huggingface_phase1_files(cfg.hf_repo_id)
    required=set(cfg.train_files)
    available_names={Path(name).name for name in published}
    missing=sorted(required-available_names)
    if missing: raise RuntimeError(f'Official repository listing is missing required training files: {missing}')
    if not (set(cfg.test_filename_aliases)&available_names): raise RuntimeError(f'No official test filename matched {cfg.test_filename_aliases}')
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    resolved=snapshot_download(
        repo_id=cfg.hf_repo_id,repo_type='dataset',token=kaggle_hf_token(),
        allow_patterns=list(cfg.hf_allow_patterns),local_dir=str(destination),
    )
    print('Downloaded Phase 1 snapshot to',resolved)
    return Path(resolved)

print({'data_search_roots':[str(Path(CFG.input_root)),str(HF_DATA_ROOT)],'hf_destination_free_gib':round(shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free/2**30,2)})
if DOWNLOAD_PHASE1_FROM_HF:
    download_huggingface_phase1(CFG,HF_DATA_ROOT)

def discover_named_files(roots:Sequence[Path],cfg:RunConfig)->dict[str,list[Path]]:
    wanted=set(cfg.train_files)
    aliases=set(cfg.test_filename_aliases)
    found={name:[] for name in [*cfg.train_files,cfg.test_file]}
    for root in roots:
        root=Path(root)
        if not root.exists(): continue
        for path in root.rglob('*'):
            if not path.is_file(): continue
            if path.name in wanted:
                found[path.name].append(path.resolve())
            elif path.name in aliases:
                found[cfg.test_file].append(path.resolve())
    for name in found:
        found[name]=sorted(set(found[name]))
    return found

DATA_SEARCH_ROOTS=[]
for candidate in (Path(CFG.input_root),HF_DATA_ROOT):
    resolved=candidate.resolve() if candidate.exists() else candidate
    if resolved not in DATA_SEARCH_ROOTS:
        DATA_SEARCH_ROOTS.append(resolved)
DISCOVERED=discover_named_files(DATA_SEARCH_ROOTS,CFG)
for name,paths in sorted(DISCOVERED.items()):
    print(name,'->',[str(path) for path in paths])

def require_unique_file(discovered:Mapping[str,Sequence[Path]],name:str)->Path:
    override=CFG.official_file_overrides.get(name)
    if override:
        selected=Path(override).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(f'Configured override for {name} does not exist: {selected}')
        return selected
    matches=list(discovered.get(name,[]))
    if len(matches)!=1:
        raise RuntimeError(
            f'Expected exactly one {name!r}; found {len(matches)}: {matches}. '
            'Set CFG.official_file_overrides[name] to an inspected exact path; duplicates are never selected silently.'
        )
    return Path(matches[0])


In [ ]:
MAX_IN_MEMORY_JSON_BYTES = 256 * 2**20

def _ijson():
    ensure_runtime_dependency("ijson", "ijson>=3.3,<4")
    import ijson
    return ijson

def detect_json_layout(path: Path) -> tuple[str, str | None]:
    ijson = _ijson()
    with Path(path).open("rb") as handle:
        parser = ijson.parse(handle)
        try:
            _, event, _ = next(parser)
        except StopIteration:
            raise ValueError(f"Empty JSON file: {path}")
        if event == "start_array":
            return "array", None
        if event != "start_map":
            raise TypeError(f"Unsupported JSON root event {event}: {path}")
        pending_key = None
        for prefix, event, value in parser:
            if prefix == "" and event == "map_key":
                pending_key = str(value)
                continue
            if pending_key is not None:
                if event == "start_array":
                    return "wrapper_array", pending_key
                if event in ("start_map", "string", "number", "boolean", "null"):
                    return "id_keyed_object", None
        return "empty_object", None

def iter_json_records(path: Path, *, id_field: str | None = None, text_field: str | None = None):
    ijson = _ijson()
    layout, wrapper = detect_json_layout(path)
    with Path(path).open("rb") as handle:
        if layout == "array":
            for index, row in enumerate(ijson.items(handle, "item")):
                if not isinstance(row, dict):
                    raise TypeError("Array records must be JSON objects")
                result = dict(row)
                result.setdefault("_record_index", index)
                yield result
        elif layout == "wrapper_array":
            for index, row in enumerate(ijson.items(handle, f"{wrapper}.item")):
                if not isinstance(row, dict):
                    raise TypeError("Wrapped records must be JSON objects")
                result = dict(row)
                result.setdefault("_record_index", index)
                yield result
        elif layout == "id_keyed_object":
            for index, (key, value) in enumerate(ijson.kvitems(handle, "")):
                if isinstance(value, dict):
                    row = dict(value)
                    if id_field and id_field not in row:
                        row[id_field] = str(key)
                    row.setdefault("_container_key", str(key))
                    row.setdefault("_record_index", index)
                    yield row
                elif isinstance(value, str):
                    yield {
                        id_field or "_container_key": str(key),
                        text_field or "_container_value": value,
                        "_container_key": str(key),
                        "_record_index": index,
                    }
                else:
                    raise TypeError(f"Unsupported top-level value under {key!r}: {type(value).__name__}")
        elif layout != "empty_object":
            raise TypeError(f"Unsupported JSON layout: {layout}")

def sample_json_records(path: Path, limit: int = 256) -> list[dict[str, Any]]:
    import itertools
    return list(itertools.islice(iter_json_records(path), int(limit)))

def inspect_schema(path: Path, sample_size: int = 3) -> dict[str, Any]:
    rows = sample_json_records(path, max(256, sample_size))
    keys = Counter(key for row in rows for key in row)
    layout, wrapper = detect_json_layout(path)
    return {
        "path": str(path),
        "bytes": Path(path).stat().st_size,
        "layout": layout,
        "wrapper_key": wrapper,
        "sampled_records": len(rows),
        "sample_field_presence": dict(keys),
        "samples": [
            {key: {"type": type(value).__name__, "preview": str(value)[:140]} for key, value in row.items()}
            for row in rows[:sample_size]
        ],
    }

def infer_record_schema(path: Path, *, role: str) -> dict[str, Any]:
    rows = sample_json_records(path, 256)
    if not rows:
        raise ValueError(f"No records in {path}")
    text_candidates = (
        "_container_value", "text", "document", "content", "case_text",
        "judgment", "judgement", "prompt",
    )
    common_text = [
        name for name in text_candidates
        if all(isinstance(row.get(name), str) and row[name].strip() for row in rows)
    ]
    if len(common_text) != 1:
        raise RuntimeError(f"{role} text field is ambiguous in {path}: {common_text}")

    id_candidates = ("_container_key", "document_id", "doc_id", "case_id", "uid", "id")
    valid_ids = []
    for name in id_candidates:
        values = [row.get(name) for row in rows]
        if all(value is not None and str(value).strip() for value in values):
            if len({str(value) for value in values}) == len(values):
                valid_ids.append(name)
    if role == "test":
        if len(valid_ids) != 1:
            raise RuntimeError(f"Official test IDs are ambiguous in {path}: {valid_ids}")
        return {"id_field": valid_ids[0], "id_strategy": "field", "text_field": common_text[0]}
    if len(valid_ids) == 1:
        return {"id_field": valid_ids[0], "id_strategy": "field", "text_field": common_text[0]}
    if not valid_ids:
        return {"id_field": None, "id_strategy": "record_index", "text_field": common_text[0]}
    raise RuntimeError(f"Training IDs are ambiguous in {path}: {valid_ids}")

def resolve_official_schemas(cfg: RunConfig, discovered: Mapping[str, Sequence[Path]]) -> dict[str, Any]:
    training = {}
    for filename in cfg.train_files:
        inferred = infer_record_schema(require_unique_file(discovered, filename), role="training")
        override = cfg.training_schema_overrides.get(filename)
        if override is not None and dict(override) != inferred:
            raise RuntimeError(f"Schema override for {filename} conflicts with inspected data: {override} != {inferred}")
        training[filename] = inferred
    test = infer_record_schema(require_unique_file(discovered, cfg.test_file), role="test")
    if cfg.test_schema_override is not None and dict(cfg.test_schema_override) != test:
        raise RuntimeError(f"Test schema override conflicts with inspected data: {cfg.test_schema_override} != {test}")
    return {"training": training, "test": test}

_schema_names = [*CFG.train_files, CFG.test_file]
if all(len(DISCOVERED.get(name, [])) == 1 or name in CFG.official_file_overrides for name in _schema_names):
    SCHEMAS = {name: inspect_schema(require_unique_file(DISCOVERED, name)) for name in _schema_names}
    print(json.dumps(SCHEMAS, indent=2, ensure_ascii=False))
    RESOLVED_SCHEMA = resolve_official_schemas(CFG, DISCOVERED)
    atomic_write_json(DIRS["manifests"] / "resolved_json_schemas.json", RESOLVED_SCHEMA)
    print("Verified per-file schemas:", json.dumps(RESOLVED_SCHEMA, indent=2))
else:
    SCHEMAS = {}
    RESOLVED_SCHEMA = None
    unresolved = {
        name: len(DISCOVERED.get(name, [])) for name in _schema_names
        if len(DISCOVERED.get(name, [])) != 1 and name not in CFG.official_file_overrides
    }
    print("Schema inspection deferred until files resolve uniquely:", unresolved)


## 4. Bounded official-corpus cache, provenance, deduplication, and split


In [ ]:
import heapq
import zlib

@dataclass(frozen=True)
class Document:
    uid: str
    text: str | None
    source: str
    raw_id: str
    content_hash: str
    word_count: int
    store_path: str | None = None
    store_offset: int | None = None
    store_length: int | None = None

def normalize_text(value: Any) -> str:
    if not isinstance(value, str):
        raise TypeError(f"Text must be a string, got {type(value).__name__}")
    result = value.replace("\r\n", "\n").strip()
    if not result:
        raise ValueError("Empty text")
    return result

def document_text(document: Document) -> str:
    if document.text is not None:
        return document.text
    if document.store_path is None or document.store_offset is None or document.store_length is None:
        raise RuntimeError(f"Document has no text storage: {document.uid}")
    with Path(document.store_path).open("rb") as handle:
        handle.seek(document.store_offset)
        payload = handle.read(document.store_length)
    return zlib.decompress(payload).decode("utf-8")

def fast_file_identity(path: Path, sample_bytes: int = 4 * 2**20) -> dict[str, Any]:
    path = Path(path)
    size = path.stat().st_size
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        digest.update(handle.read(sample_bytes))
        if size > sample_bytes:
            handle.seek(max(0, size - sample_bytes))
            digest.update(handle.read(sample_bytes))
    return {"name": path.name, "bytes": size, "head_tail_sha256": digest.hexdigest()}

def stable_raw_id(row: Mapping[str, Any], index: int, schema: Mapping[str, Any]) -> str:
    if schema["id_strategy"] == "record_index":
        return f"row-{index:012d}"
    value = str(row[schema["id_field"]]).strip()
    if not value:
        raise ValueError("Empty identifier")
    return value

def deterministic_selected_indices(
    path: Path, source: str, schema: Mapping[str, Any], cap: int, seed: int
) -> tuple[set[int], int]:
    if cap <= 0:
        raise ValueError("corpus_document_cap_per_source must be positive")
    heap: list[tuple[int, int]] = []
    count = 0
    for index, row in enumerate(iter_json_records(path, id_field=schema.get("id_field"), text_field=schema["text_field"])):
        raw_id = stable_raw_id(row, index, schema)
        score = int(hashlib.sha256(f"{seed}:{source}:{raw_id}".encode()).hexdigest(), 16)
        item = (-score, index)
        if len(heap) < cap:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
        count += 1
    return {index for _, index in heap}, count

def _load_document_index(index_path: Path, store_path: Path) -> list[Document]:
    documents = []
    with Path(index_path).open("r", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            documents.append(Document(
                uid=row["uid"], text=None, source=row["source"], raw_id=row["raw_id"],
                content_hash=row["content_hash"], word_count=int(row["word_count"]),
                store_path=str(store_path), store_offset=int(row["store_offset"]),
                store_length=int(row["store_length"]),
            ))
    return documents

def load_bounded_documents(
    path: Path,
    source: str,
    schema: Mapping[str, Any],
    store_dir: Path,
    cap: int,
    seed: int,
) -> tuple[list[Document], list[dict[str, Any]], dict[str, Any]]:
    path = Path(path)
    identity = fast_file_identity(path)
    selection_config = {
        "source_identity": identity,
        "schema": dict(schema),
        "cap": int(cap),
        "seed": int(seed),
        "selection": "smallest SHA-256(seed:source:stable_raw_id)",
        "format_version": 2,
    }
    key = hashlib.sha256(json.dumps(selection_config, sort_keys=True).encode()).hexdigest()[:20]
    store_dir = Path(store_dir)
    store_dir.mkdir(parents=True, exist_ok=True)
    store_path = store_dir / f"{source}.{key}.zbin"
    index_path = store_dir / f"{source}.{key}.index.jsonl"
    meta_path = store_dir / f"{source}.{key}.meta.json"
    if store_path.exists() and index_path.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("selection_config") != selection_config:
            raise RuntimeError(f"Existing cache metadata mismatch: {meta_path}")
        return _load_document_index(index_path, store_path), list(meta.get("rejections", [])), meta

    selected, total_records = deterministic_selected_indices(path, source, schema, cap, seed)
    with tempfile.NamedTemporaryFile(dir=store_dir, delete=False) as handle:
        temporary_store = Path(handle.name)
    with tempfile.NamedTemporaryFile(dir=store_dir, delete=False) as handle:
        temporary_index = Path(handle.name)
    rejections = []
    accepted = 0
    try:
        with temporary_store.open("wb") as store_handle, temporary_index.open("w", encoding="utf-8") as index_handle:
            for index, row in enumerate(iter_json_records(path, id_field=schema.get("id_field"), text_field=schema["text_field"])):
                if index not in selected:
                    continue
                try:
                    raw_id = stable_raw_id(row, index, schema)
                    text = normalize_text(row[schema["text_field"]])
                    encoded = text.encode("utf-8")
                    compressed = zlib.compress(encoded, level=3)
                    offset = store_handle.tell()
                    store_handle.write(compressed)
                    record = {
                        "uid": f"{source}:{raw_id}", "source": source, "raw_id": raw_id,
                        "content_hash": hashlib.sha256(encoded).hexdigest(),
                        "word_count": len(text.split()), "store_offset": offset,
                        "store_length": len(compressed),
                    }
                    index_handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                    accepted += 1
                except Exception as exc:
                    rejections.append({"index": index, "error": f"{type(exc).__name__}: {exc}"})
            store_handle.flush(); os.fsync(store_handle.fileno())
            index_handle.flush(); os.fsync(index_handle.fileno())
        os.replace(temporary_store, store_path)
        os.replace(temporary_index, index_path)
    finally:
        temporary_store.unlink(missing_ok=True)
        temporary_index.unlink(missing_ok=True)
    meta = {
        "selection_config": selection_config,
        "total_source_records": total_records,
        "accepted": accepted,
        "rejected": len(rejections),
        "rejections": rejections,
        "created_utc": utc_now(),
    }
    atomic_write_json(meta_path, meta)
    return _load_document_index(index_path, store_path), rejections, meta

def corpus_audit(docs: Sequence[Document]) -> dict[str, Any]:
    ids = Counter(document.uid for document in docs)
    hashes = Counter(document.content_hash for document in docs)
    return {
        "documents": len(docs),
        "sources": dict(Counter(document.source for document in docs)),
        "whitespace_word_count": sum(document.word_count for document in docs),
        "words_by_source": {
            source: sum(document.word_count for document in docs if document.source == source)
            for source in sorted({document.source for document in docs})
        },
        "duplicate_uids": sorted(key for key, value in ids.items() if value > 1),
        "duplicate_content_hashes": sorted(key for key, value in hashes.items() if value > 1),
    }

def deduplicate_exact_documents(docs: Sequence[Document]) -> tuple[list[Document], list[dict[str, str]]]:
    kept, dropped, seen_ids, seen_hashes = [], [], {}, {}
    for document in sorted(docs, key=lambda value: (value.uid, value.content_hash)):
        reason = "duplicate_uid" if document.uid in seen_ids else "duplicate_exact_text" if document.content_hash in seen_hashes else None
        if reason:
            retained = seen_ids.get(document.uid) if reason == "duplicate_uid" else seen_hashes[document.content_hash]
            dropped.append({"uid": document.uid, "kept_uid": retained, "reason": reason})
            continue
        kept.append(document)
        seen_ids[document.uid] = document.uid
        seen_hashes[document.content_hash] = document.uid
    return kept, dropped

def dataset_fingerprint(docs: Sequence[Document]) -> str:
    digest = hashlib.sha256()
    for document in sorted(docs, key=lambda value: value.uid):
        digest.update(f"{document.uid}\t{document.content_hash}\n".encode())
    return digest.hexdigest()


In [ ]:
def deterministic_document_split(documents:Sequence[Document],validation_fraction:float,seed:int)->tuple[list[Document],list[Document],dict[str,Any]]:
    if not 0<validation_fraction<1: raise ValueError('Choose validation_fraction strictly between zero and one')
    audit=corpus_audit(documents)
    if audit['duplicate_uids'] or audit['duplicate_content_hashes']: raise ValueError('Deduplicate exact documents before splitting')
    train=[];validation=[];per_source={}
    for source in sorted({document.source for document in documents}):
        group=sorted((document for document in documents if document.source==source),key=lambda value:value.uid)
        random.Random(seed+int(hashlib.sha256(source.encode()).hexdigest()[:8],16)).shuffle(group)
        n_val=max(1,round(len(group)*validation_fraction))
        if n_val>=len(group): raise ValueError(f'Need at least one training document for {source}')
        validation.extend(group[:n_val]);train.extend(group[n_val:])
        per_source[source]={'total':len(group),'training':len(group)-n_val,'validation':n_val}
    if {document.uid for document in train}&{document.uid for document in validation}: raise AssertionError('Split leakage')
    metadata={
        'created_utc':utc_now(),'seed':seed,'validation_fraction':validation_fraction,
        'strategy':'source-stratified deterministic document split','per_source':per_source,
        'dataset_fingerprint':dataset_fingerprint(documents),
        'training_ids':sorted(document.uid for document in train),
        'validation_ids':sorted(document.uid for document in validation),
    }
    return train,validation,metadata

def save_or_verify_split(path:Path,meta:Mapping[str,Any])->None:
    path=Path(path)
    if path.exists():
        old=json.loads(path.read_text(encoding='utf-8'))
        keys=('seed','validation_fraction','strategy','per_source','dataset_fingerprint','training_ids','validation_ids')
        if any(old.get(key)!=meta.get(key) for key in keys): raise RuntimeError('Existing data split differs; refusing overwrite')
        print('Verified existing split',path)
    else: atomic_write_json(path,dict(meta))

def validate_source_mixture(ratios:Mapping[str,float],sources:Iterable[str])->dict[str,float]:
    allowed=set(sources)
    if set(ratios)!=allowed: raise ValueError(f'Mixture keys {set(ratios)} must equal {allowed}')
    values={key:float(value) for key,value in ratios.items()}
    if any(not math.isfinite(value) or value<=0 for value in values.values()): raise ValueError('All source proportions must be finite and > 0')
    if not math.isclose(sum(values.values()),1.0,rel_tol=1e-6,abs_tol=1e-6): raise ValueError('Source proportions must sum to 1.0')
    return values

def source_sampling_indices(documents:Sequence[Document],ratios:Mapping[str,float],samples:int,seed:int)->list[int]:
    if samples<=0: raise ValueError('samples must be >0')
    ratios=validate_source_mixture(ratios,(document.source for document in documents))
    pools={source:[index for index,document in enumerate(documents) if document.source==source] for source in ratios}
    if any(not pool for pool in pools.values()): raise ValueError('Missing source documents')
    rng=random.Random(seed);sources=list(sorted(ratios));weights=[ratios[source] for source in sources]
    return [rng.choice(pools[source]) for source in rng.choices(sources,weights=weights,k=samples)]

def _stable_document_order(documents:Sequence[Document],seed:int)->list[Document]:
    return sorted(documents,key=lambda document:hashlib.sha256(f'{seed}:{document.uid}'.encode()).hexdigest())

def make_continuation_examples(documents:Sequence[Document],prompt_words:int=512,continuation_words:int=128,max_per_source:int=16,seed:int=781)->list[dict[str,str]]:
    result=[];needed=prompt_words+continuation_words
    for source in sorted({document.source for document in documents}):
        collected=0
        for document in _stable_document_order([value for value in documents if value.source==source],seed):
            if document.word_count<needed: continue
            units=document_text(document).split()
            result.append({'id':document.uid,'source':source,'prompt':' '.join(units[:prompt_words]),'reference':' '.join(units[prompt_words:needed])})
            collected+=1
            if collected>=max_per_source: break
    return result

def sampled_cross_split_overlap_audit(train:Sequence[Document],validation:Sequence[Document],seed:int,max_per_split:int=10_000)->dict[str,Any]:
    def signature(document:Document)->str:
        units=document_text(document).lower().split()
        sample=' '.join([*units[:128],*units[-128:]])
        return hashlib.sha256(sample.encode()).hexdigest()
    sampled_train=_stable_document_order(train,seed)[:max_per_split]
    sampled_val=_stable_document_order(validation,seed+1)[:max_per_split]
    train_signatures={signature(document):document.uid for document in sampled_train}
    overlaps=[]
    for document in sampled_val:
        value=signature(document)
        if value in train_signatures: overlaps.append({'training_uid':train_signatures[value],'validation_uid':document.uid})
    return {'method':'first+last 128 normalized whitespace words SHA-256','training_sample':len(sampled_train),'validation_sample':len(sampled_val),'overlaps':overlaps}


In [ ]:
def load_official_training_corpus(cfg: RunConfig, discovered: Mapping[str, Sequence[Path]], schemas: Mapping[str, Any]):
    if cfg.corpus_document_cap_per_source is None:
        raise RuntimeError("Set CFG.corpus_document_cap_per_source explicitly before bounded preprocessing")
    ratios = validate_source_mixture(
        cfg.source_mixing_ratios,
        (Path(filename).stem.removeprefix("train_") for filename in cfg.train_files),
    )
    all_docs, rejections, provenance = [], {}, {}
    store_dir = DIRS["data"] / "bounded_document_store"
    for filename in cfg.train_files:
        path = require_unique_file(discovered, filename)
        source = Path(filename).stem.removeprefix("train_")
        docs, bad, meta = load_bounded_documents(
            path, source, schemas["training"][filename], store_dir,
            cfg.corpus_document_cap_per_source, cfg.seed,
        )
        all_docs.extend(docs)
        rejections[source] = bad
        provenance[source] = {
            "file": filename,
            "source_identity": meta["selection_config"]["source_identity"],
            "total_source_records": meta["total_source_records"],
            "selected": len(docs),
            "rejected_selected_records": len(bad),
            "selection": meta["selection_config"],
        }
    clean, dropped = deduplicate_exact_documents(all_docs)
    audit = corpus_audit(clean)
    audit.update({
        "selected_before_deduplication": len(all_docs),
        "removed_exact_duplicates": len(dropped),
        "requested_sampling_ratios": ratios,
        "sampling_unit": "document draws, not token percentages",
        "official_test_excluded": True,
    })
    train, validation, split = deterministic_document_split(clean, cfg.validation_fraction, cfg.seed)
    audit["training"] = corpus_audit(train)
    audit["validation"] = corpus_audit(validation)
    save_or_verify_split(DIRS["manifests"] / "document_split.json", split)
    overlap = sampled_cross_split_overlap_audit(train, validation, cfg.seed)
    artifacts = {
        "dataset_provenance.json": provenance,
        "dataset_audit.json": audit,
        "rejected_records.json": rejections,
        "deduplication_log.json": dropped,
        "source_sampling_config.json": {"ratios": ratios, "unit": "document draws"},
        "cross_split_overlap_audit.json": overlap,
    }
    for name, value in artifacts.items():
        path = DIRS["manifests"] / name
        if path.exists() and json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError(f"Persisted manifest changed: {path}")
        if not path.exists():
            atomic_write_json(path, value)
    examples = make_continuation_examples(validation, max_per_source=16, seed=cfg.seed)
    examples_path = DIRS["data"] / "local_continuation_examples.json"
    if examples_path.exists() and json.loads(examples_path.read_text(encoding="utf-8")) != examples:
        raise RuntimeError("Fixed development examples changed")
    if not examples_path.exists():
        atomic_write_json(examples_path, examples)
    atomic_write_json(DIRS["manifests"] / "local_generation_protocol.json", {
        "selection": "up to 16 eligible documents per source in deterministic hash order",
        "seed": cfg.seed, "prompt_words": 512, "reference_words": 128,
        "examples": len(examples), "ids": [row["id"] for row in examples],
    })
    return train, validation, examples, audit

RUN_CORPUS_PREPROCESSING = True
if CFG.execution_mode == "inference_only":
    TRAIN_DOCS = VAL_DOCS = LOCAL_DEV = DATA_AUDIT = None
    print("Inference-only mode: skipped every training-corpus load and preprocessing operation.")
elif RUN_CORPUS_PREPROCESSING:
    if RESOLVED_SCHEMA is None:
        raise RuntimeError("Official file schemas must be resolved before preprocessing")
    TRAIN_DOCS, VAL_DOCS, LOCAL_DEV, DATA_AUDIT = load_official_training_corpus(CFG, DISCOVERED, RESOLVED_SCHEMA)
    print("Loaded bounded official corpus cache:", DATA_AUDIT)
else:
    TRAIN_DOCS = VAL_DOCS = LOCAL_DEV = DATA_AUDIT = None
    print("Corpus preprocessing is opt-in. Review the cap and set RUN_CORPUS_PREPROCESSING=True when needed.")


## 5. Tokenizer persistence and compatibility

In [ ]:
def directory_fingerprint(path:Path)->str:
    path=Path(path);digest=hashlib.sha256()
    files=sorted(p for p in path.rglob('*') if p.is_file() and p.name!='tokenizer_metadata.json')
    if not files: raise ValueError(f'No tokenizer files found: {path}')
    for file in files: digest.update(file.relative_to(path).as_posix().encode()+bytes.fromhex(sha256_file(file)))
    return digest.hexdigest()
def tokenizer_encode(tok:Any,text:str)->list[int]:
    try: result=tok.encode(text,add_special_tokens=False)
    except TypeError: result=tok.encode(text)
    return list(result)
def tokenizer_decode(tok:Any,ids:Sequence[int])->str:
    try: return tok.decode(list(ids),skip_special_tokens=True)
    except TypeError: return tok.decode(list(ids))
def tokenizer_metadata(tok:Any,location:Path)->dict[str,Any]:
    probes=['The Court considered section 14 and dismissed the appeal.','§ 21—न्यायालय: AIR 2024 SC 17;  multiple   spaces\nand punctuation.']
    results=[]
    for probe in probes:
        ids=tokenizer_encode(tok,probe);decoded=tokenizer_decode(tok,ids)
        if not ids or decoded!=probe: raise ValueError(f'Tokenizer failed exact roundtrip: {probe!r} -> {decoded!r}')
        results.append({'input':probe,'ids':ids,'decoded':decoded})
    special={k:getattr(tok,k,None) for k in ('bos_token_id','eos_token_id','pad_token_id','unk_token_id')}
    if len({value for value in special.values() if value is not None})!=len([value for value in special.values() if value is not None]): raise ValueError('Special token IDs must be distinct')
    return {'created_utc':utc_now(),'vocab_size':len(tok),'special_token_ids':special,'fingerprint':directory_fingerprint(location),'round_trip_probes':results}
def register_tokenizer(tok:Any,destination:Path,save_function:Callable[[Any,Path],Any]|None=None)->dict[str,Any]:
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True);meta_path=destination/'tokenizer_metadata.json'
    if meta_path.exists():
        previous=json.loads(meta_path.read_text(encoding='utf-8'))
        if previous.get('fingerprint')!=directory_fingerprint(destination): raise RuntimeError('Tokenizer registered files have changed')
        if previous.get('vocab_size')!=len(tok): raise RuntimeError('Loaded tokenizer size differs')
        return previous
    if any(destination.iterdir()): raise RuntimeError('Nonempty tokenizer directory; choose explicit restore/adaptation')
    if save_function: save_function(tok,destination)
    elif hasattr(tok,'save_pretrained'): tok.save_pretrained(destination)
    else: raise TypeError('Course tokenizer requires an explicit save_function(tok,destination) adapter')
    meta=tokenizer_metadata(tok,destination);atomic_write_json(meta_path,meta);return meta
def tokenizer_token_audit(docs:Iterable[Document],tok:Any)->dict[str,Any]:
    by_source=Counter(); count=0
    for d in docs: by_source[d.source]+=len(tokenizer_encode(tok,document_text(d)));count+=1
    return {'documents':count,'tokens':sum(by_source.values()),'tokens_by_source':dict(by_source)}

COURSE_BPE_SOURCE ='"""Deterministic byte-level BPE tokenizer used by the Phase 1 notebook.\n\nThe implementation is deliberately small: it wraps the `tokenizers` BPE model,\nexposes the interface expected by the notebook, and persists every artifact\nneeded to recreate the token-to-ID mapping in a later Kaggle session.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\n\nSPECIAL_TOKENS = ("<pad>", "<unk>", "<bos>", "<eos>")\n\n\nclass CourseBPETokenizer:\n    """Minimal tokenizer adapter with a stable encode/decode/save contract."""\n\n    def __init__(self, backend):\n        self.backend = backend\n        self.pad_token_id = backend.token_to_id(SPECIAL_TOKENS[0])\n        self.unk_token_id = backend.token_to_id(SPECIAL_TOKENS[1])\n        self.bos_token_id = backend.token_to_id(SPECIAL_TOKENS[2])\n        self.eos_token_id = backend.token_to_id(SPECIAL_TOKENS[3])\n\n    def __len__(self) -> int:\n        return int(self.backend.get_vocab_size())\n\n    def encode(self, text: str, add_special_tokens: bool = False) -> list[int]:\n        if not isinstance(text, str):\n            raise TypeError("text must be a string")\n        ids = list(self.backend.encode(text, add_special_tokens=False).ids)\n        if add_special_tokens:\n            ids = [self.bos_token_id, *ids, self.eos_token_id]\n        return ids\n\n    def decode(self, ids: Sequence[int], skip_special_tokens: bool = True) -> str:\n        return self.backend.decode([int(value) for value in ids], skip_special_tokens=skip_special_tokens)\n\n    @classmethod\n    def train(\n        cls,\n        texts: Iterable[str],\n        *,\n        vocab_size: int = 32_000,\n        min_frequency: int = 2,\n    ) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers\n        except ImportError as exc:\n            raise RuntimeError(\n                "The `tokenizers` package is required. Kaggle normally includes it; "\n                "otherwise install a pinned version before this cell."\n            ) from exc\n\n        if vocab_size < 1_024:\n            raise ValueError("vocab_size must be at least 1024")\n        if min_frequency < 1:\n            raise ValueError("min_frequency must be positive")\n\n        backend = Tokenizer(models.BPE(unk_token=SPECIAL_TOKENS[1]))\n        backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)\n        backend.decoder = decoders.ByteLevel()\n        trainer = trainers.BpeTrainer(\n            vocab_size=int(vocab_size),\n            min_frequency=int(min_frequency),\n            special_tokens=list(SPECIAL_TOKENS),\n            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),\n            show_progress=True,\n        )\n        backend.train_from_iterator((str(text) for text in texts), trainer=trainer)\n        tokenizer = cls(backend)\n        if len(tokenizer) != vocab_size:\n            raise RuntimeError(\n                f"BPE training produced {len(tokenizer)} tokens, expected exactly {vocab_size}. "\n                "Inspect the corpus and do not begin model training with a changing vocabulary."\n            )\n        return tokenizer\n\n    def save_pretrained(self, destination: str | Path) -> tuple[str, ...]:\n        destination = Path(destination)\n        destination.mkdir(parents=True, exist_ok=True)\n        tokenizer_json = destination / "tokenizer.json"\n        self.backend.save(str(tokenizer_json))\n        # Save the explicit BPE vocabulary and merge table as well as the unified\n        # tokenizer.json, so the learned mapping is auditable without code.\n        model_files = tuple(self.backend.model.save(str(destination), "course_bpe"))\n        config = {\n            "tokenizer_class": type(self).__name__,\n            "tokenizer_file": tokenizer_json.name,\n            "model_type": "byte_level_bpe",\n            "special_tokens": list(SPECIAL_TOKENS),\n            "special_token_ids": {\n                "pad_token_id": self.pad_token_id,\n                "unk_token_id": self.unk_token_id,\n                "bos_token_id": self.bos_token_id,\n                "eos_token_id": self.eos_token_id,\n            },\n            "vocab_size": len(self),\n        }\n        (destination / "tokenizer_config.json").write_text(\n            json.dumps(config, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8"\n        )\n        return (str(tokenizer_json), *model_files, str(destination / "tokenizer_config.json"))\n\n    @classmethod\n    def from_pretrained(cls, location: str | Path) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer\n        except ImportError as exc:\n            raise RuntimeError("The `tokenizers` package is required to restore the BPE tokenizer") from exc\n\n        location = Path(location)\n        tokenizer_path = location / "tokenizer.json"\n        config_path = location / "tokenizer_config.json"\n        if not tokenizer_path.is_file() or not config_path.is_file():\n            raise FileNotFoundError(f"Incomplete tokenizer directory: {location}")\n        config = json.loads(config_path.read_text(encoding="utf-8"))\n        tokenizer = cls(Tokenizer.from_file(str(tokenizer_path)))\n        if len(tokenizer) != int(config["vocab_size"]):\n            raise RuntimeError("Saved tokenizer vocabulary size does not match tokenizer_config.json")\n        actual_ids = {\n            "pad_token_id": tokenizer.pad_token_id,\n            "unk_token_id": tokenizer.unk_token_id,\n            "bos_token_id": tokenizer.bos_token_id,\n            "eos_token_id": tokenizer.eos_token_id,\n        }\n        if actual_ids != config["special_token_ids"]:\n            raise RuntimeError("Saved tokenizer special-token mapping is inconsistent")\n        return tokenizer\n'

TOKENIZER_TRAINING_CONFIG={
    'type':'byte_level_bpe','vocab_size':25_000,'min_frequency':2,
    'training_documents':'TRAIN_DOCS only','special_tokens':['<pad>','<unk>','<bos>','<eos>'],
}
RUN_TOKENIZER_TRAINING=True

def install_python_source(path:Path,source:str)->None:
    path=Path(path)
    if path.exists() and path.read_text(encoding='utf-8')!=source and any(DIRS['checkpoints'].glob('*.pt')):
        raise RuntimeError(f'Refusing to change source after checkpoints exist: {path}')
    if not path.exists() or path.read_text(encoding='utf-8')!=source: atomic_write_bytes(path,source.encode('utf-8'))

def import_source_module(module_name:str,path:Path):
    spec=importlib.util.spec_from_file_location(module_name,path)
    if spec is None or spec.loader is None: raise ImportError(f'Cannot import {path}')
    module=importlib.util.module_from_spec(spec);sys.modules[module_name]=module;spec.loader.exec_module(module);return module

ensure_runtime_dependency('tokenizers','tokenizers>=0.20,<1')
BPE_SOURCE_PATH=DIRS['model']/'course_bpe.py';install_python_source(BPE_SOURCE_PATH,COURSE_BPE_SOURCE)
course_tokenizer_import = Path(CFG.tokenizer_import_path).expanduser() if CFG.tokenizer_import_path else None
if course_tokenizer_import is not None and not (DIRS['tokenizer']/'tokenizer.json').exists():
    if not course_tokenizer_import.is_dir(): raise FileNotFoundError(course_tokenizer_import)
    if any(DIRS['tokenizer'].iterdir()): raise RuntimeError('Tokenizer destination is nonempty')
    for source_file in course_tokenizer_import.iterdir():
        if source_file.is_file(): shutil.copy2(source_file, DIRS['tokenizer']/source_file.name)
    print('Imported the frozen Submission 01/course tokenizer from',course_tokenizer_import)
course_bpe=import_source_module('cs781_course_bpe',BPE_SOURCE_PATH)
if (DIRS['tokenizer']/'tokenizer.json').exists():
    tokenizer=course_bpe.CourseBPETokenizer.from_pretrained(DIRS['tokenizer'])
elif TRAIN_DOCS is not None and RUN_TOKENIZER_TRAINING:
    tokenizer=course_bpe.CourseBPETokenizer.train(
        (document_text(document) for document in TRAIN_DOCS),
        vocab_size=TOKENIZER_TRAINING_CONFIG['vocab_size'],min_frequency=TOKENIZER_TRAINING_CONFIG['min_frequency'],
    )
else: tokenizer=None

if tokenizer is not None:
    TOKENIZER_META=register_tokenizer(tokenizer,DIRS['tokenizer'])
    audit_docs=TRAIN_DOCS[:min(256,len(TRAIN_DOCS))] if TRAIN_DOCS else []
    TOKENIZER_AUDIT=tokenizer_token_audit(audit_docs,tokenizer) if audit_docs else None
    if TOKENIZER_AUDIT is not None:
        TOKENIZER_AUDIT['scope']='deterministic first 256 training documents (or all when fewer)'
        atomic_write_json(DIRS['manifests']/'tokenizer_audit.json',TOKENIZER_AUDIT)
    print('Restored/registered tokenizer:',TOKENIZER_META)
else:
    TOKENIZER_META=None;TOKENIZER_AUDIT=None
    print('No saved/imported tokenizer found. Set CFG.tokenizer_import_path, or explicitly authorize one tokenizer-training run.')


## 6. Prefix-hybrid model and enforced contract

Supply a standalone `prefix_hybrid_lm.py` that defines:

- `make_model_config(vocab_size) -> Mapping[str, Any]`
- `build_model(config) -> torch.nn.Module`
- either `model(input_ids=..., attention_mask=..., prefix_lengths=...)`, or a module-level
  `forward_adapter(model, input_ids, attention_mask, prefix_lengths)`.

For prefix length `p`, valid key `j`, and query `i`, the required visibility is:

- if `i < p`, allow `j < p` (bidirectional prompt representation);
- if `i >= p`, allow `j < p` or `p <= j <= i` (causal continuation conditioned on the prompt);
- padded keys are never visible.

The selected implementation is a **52,206,376-parameter narrow/deep Transformer**: 25k token
vocabulary, hidden size 384, 24 shared blocks, 6 attention heads, full-head RoPE, pre-RMSNorm,
and SwiGLU. Input/output embeddings are tied. It is trained from scratch and uses one visibility
mask rather than a separate encoder, decoder, or cross-attention module.

The module must be implemented from scratch, contain 50M-300M parameters, use no pretrained model
weights, and reconstruct exactly from its saved numerical configuration. The notebook tests tensor
shape, finite gradients, target causality, bidirectional-prefix behavior, prompt conditioning, context
limits, source hashing, and exact state reconstruction.


In [ ]:
STUDENT_MODEL_CONFIG_OVERRIDE = None
MODEL = MODEL_CONFIG = RECONSTRUCT_MODEL = MODEL_MODULE = None
MODEL_SOURCE_PATH = MODEL_SOURCE_SHA256 = None

def install_student_model_source(source_path: Path) -> Path:
    source_path = Path(source_path).expanduser().resolve()
    if not source_path.is_file(): raise FileNotFoundError(source_path)
    tree = ast.parse(source_path.read_text(encoding="utf-8"), filename=str(source_path))
    banned = {"transformers", "fairseq"}
    imported = {
        alias.name.split(".")[0]
        for node in ast.walk(tree) if isinstance(node, ast.Import)
        for alias in node.names
    }
    imported |= {
        (node.module or "").split(".")[0]
        for node in ast.walk(tree) if isinstance(node, ast.ImportFrom)
    }
    if imported & banned:
        raise RuntimeError(f"Forbidden pretrained-model libraries in model source: {imported & banned}")
    destination = DIRS["model"] / "prefix_hybrid_lm.py"
    payload = source_path.read_bytes()
    if destination.exists() and destination.read_bytes() != payload and any(DIRS["checkpoints"].glob("*.pt")):
        raise RuntimeError("Refusing to change model source after checkpoints exist")
    atomic_write_bytes(destination, payload)
    return destination

def hybrid_logits(model, input_ids, attention_mask, prefix_lengths):
    if MODEL_MODULE is not None and callable(getattr(MODEL_MODULE, "forward_adapter", None)):
        output = MODEL_MODULE.forward_adapter(model, input_ids, attention_mask, prefix_lengths)
    else:
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            prefix_lengths=prefix_lengths,
        )
    return output.logits if hasattr(output, "logits") else output[0] if isinstance(output, (tuple, list)) else output

def count_parameters(model: torch.nn.Module) -> dict[str, int]:
    return {
        "total": sum(parameter.numel() for parameter in model.parameters()),
        "trainable": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
    }

def validate_hybrid_contract(model, vocab_size: int, device: str, sequence_length: int = 13):
    model = model.to(device); model.train(); model.zero_grad(set_to_none=True)
    input_ids = torch.randint(0, vocab_size, (2, sequence_length), dtype=torch.long, device=device)
    attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
    prefix_lengths = torch.tensor([5, 7], dtype=torch.long, device=device)
    logits = hybrid_logits(model, input_ids, attention_mask, prefix_lengths)
    expected = (2, sequence_length, vocab_size)
    if tuple(logits.shape) != expected:
        raise ValueError(f"Expected logits shape {expected}, got {tuple(logits.shape)}")
    if not torch.isfinite(logits).all(): raise FloatingPointError("Non-finite logits")
    labels = torch.randint(0, vocab_size, (2, sequence_length), dtype=torch.long, device=device)
    for row, prefix in enumerate(prefix_lengths.tolist()): labels[row, :prefix] = -100
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, vocab_size), labels.reshape(-1), ignore_index=-100
    )
    loss.backward()
    gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not gradients or not all(torch.isfinite(g).all().item() for g in gradients):
        raise FloatingPointError("Missing or non-finite gradients")
    counts = count_parameters(model)
    if not 50_000_000 <= counts["total"] <= 300_000_000:
        raise ValueError(f"Official parameter bound violated: {counts['total']:,}")
    model.zero_grad(set_to_none=True)
    return {"logits_shape": list(logits.shape), "loss": float(loss.detach().cpu()), **counts, "device": device}

_restored_model_source = DIRS["model"] / "prefix_hybrid_lm.py"
_configured_model_source = (
    Path(CFG.student_model_source_path).expanduser()
    if CFG.student_model_source_path
    else _restored_model_source if _restored_model_source.exists() else None
)
if _configured_model_source is not None:
    if TOKENIZER_META is None:
        raise RuntimeError("Register/restore the tokenizer before importing the student model")
    MODEL_SOURCE_PATH = install_student_model_source(_configured_model_source)
    MODEL_SOURCE_SHA256 = sha256_file(MODEL_SOURCE_PATH)
    MODEL_MODULE = import_source_module(
        f"cs781_submission03_model_{MODEL_SOURCE_SHA256[:12]}", MODEL_SOURCE_PATH
    )
    if not callable(getattr(MODEL_MODULE, "build_model", None)):
        raise AttributeError("Student module must define build_model(config)")
    if STUDENT_MODEL_CONFIG_OVERRIDE is not None:
        MODEL_CONFIG = dict(STUDENT_MODEL_CONFIG_OVERRIDE)
    elif callable(getattr(MODEL_MODULE, "make_model_config", None)):
        MODEL_CONFIG = dict(MODEL_MODULE.make_model_config(int(TOKENIZER_META["vocab_size"])))
    else:
        candidate = getattr(MODEL_MODULE, "MODEL_CONFIG", None)
        if not isinstance(candidate, Mapping):
            raise RuntimeError("Provide STUDENT_MODEL_CONFIG_OVERRIDE, make_model_config, or MODEL_CONFIG")
        MODEL_CONFIG = dict(candidate)
    if int(MODEL_CONFIG.get("vocab_size", -1)) != int(TOKENIZER_META["vocab_size"]):
        raise ValueError("Model vocabulary does not match the frozen tokenizer")
    if int(MODEL_CONFIG.get("max_sequence_length", 0)) <= 2:
        raise ValueError("MODEL_CONFIG must declare max_sequence_length > 2")
    RECONSTRUCT_MODEL = MODEL_MODULE.build_model
    MODEL = RECONSTRUCT_MODEL(MODEL_CONFIG)
    MODEL_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    MODEL_REPORT = validate_hybrid_contract(MODEL, TOKENIZER_META["vocab_size"], MODEL_DEVICE)
    analytical_counter = getattr(MODEL_MODULE, "analytical_parameter_count", None)
    if not callable(analytical_counter):
        raise AttributeError("Student module must define analytical_parameter_count(config)")
    analytical_total = int(analytical_counter(MODEL_CONFIG))
    if analytical_total != int(MODEL_REPORT["total"]):
        raise AssertionError(
            f"Analytical parameter count {analytical_total:,} differs from runtime "
            f"{int(MODEL_REPORT['total']):,}"
        )
    MODEL_REPORT["analytical_parameter_count"] = analytical_total
    MODEL_REPORT["analytical_runtime_match"] = "PASS"
    MODEL_REPORT.update({
        "architecture_family": "single-stack prefix hybrid",
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "model_config": MODEL_CONFIG,
    })
    atomic_write_json(DIRS["manifests"] / "model_validation.json", MODEL_REPORT)
    MODEL.to("cpu")
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Student model validation:", json.dumps(MODEL_REPORT, indent=2))
else:
    MODEL_REPORT = None
    print("STUDENT ACTION REQUIRED: attach and select your independently authored prefix_hybrid_lm.py")


## 7. Stateful sampling, histories, checkpoints, and progress

In [ ]:
class StatefulRandomSampler(Sampler[int]):
    # Cursor is updated when DataLoader requests indices, possibly ahead of consumed batches if num_workers>0.
    # For exact checkpoint continuation use num_workers=0 or save a committed (consumed) batch cursor.
    def __init__(self,data_source:Sequence[Any],seed:int,epoch:int=0,cursor:int=0):
        self.length=len(data_source);self.seed=int(seed);self.epoch=int(epoch);self.cursor=int(cursor)
    def _order(self)->list[int]:
        return torch.randperm(self.length,generator=torch.Generator().manual_seed(self.seed+self.epoch)).tolist()
    def __iter__(self):
        for position in range(self.cursor,self.length):
            self.cursor=position+1
            yield self._order_cache[position] if hasattr(self,'_order_cache') else self._get_order_item(position)
    def _get_order_item(self,position):
        if not hasattr(self,'_order_cache'): self._order_cache=self._order()
        return self._order_cache[position]
    def __len__(self): return max(0,self.length-self.cursor)
    def next_epoch(self): self.epoch+=1;self.cursor=0;self.__dict__.pop('_order_cache',None)
    def state_dict(self): return {'length':self.length,'seed':self.seed,'epoch':self.epoch,'cursor':self.cursor}
    def load_state_dict(self,state:Mapping[str,Any]):
        if int(state['length'])!=self.length or int(state['seed'])!=self.seed: raise ValueError('Incompatible sampler')
        cursor=int(state['cursor'])
        if not 0<=cursor<=self.length: raise ValueError('Sampler cursor outside dataset')
        self.epoch=int(state['epoch']);self.cursor=cursor;self.__dict__.pop('_order_cache',None)

class JsonlHistory:
    def __init__(self,path:Path): self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
    def records(self)->list[dict[str,Any]]:
        if not self.path.exists(): return []
        lines=self.path.read_text(encoding='utf-8').splitlines();rows=[]
        for i,line in enumerate(lines):
            if not line.strip():continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i==len(lines)-1: raise RuntimeError('Incomplete last history line. Restore last checkpoint/history or explicitly repair; do not silently discard')
                raise
        steps=[r['step'] for r in rows]
        if len(steps)!=len(set(steps)) or steps!=sorted(steps): raise ValueError('Duplicate or nonmonotonic history steps')
        return rows
    def append(self,record:Mapping[str,Any]):
        existing=self.records()
        if existing and int(record['step'])<=int(existing[-1]['step']): raise ValueError('Refusing duplicated/nonmonotonic history step')
        with self.path.open('ab') as h:
            h.write((json.dumps(dict(record),ensure_ascii=False)+'\n').encode());h.flush();os.fsync(h.fileno())

In [ ]:
CHECKPOINT_VERSION=3
def unwrap_model(model:torch.nn.Module)->torch.nn.Module:
    if isinstance(model,torch.nn.DataParallel): model=model.module
    return getattr(model,"base_model",model)
def capture_rng_state(device:str|None=None)->dict[str,Any]:
    cuda_states=[]
    if torch.cuda.is_available():
        cuda_states=[{"index":index,"state":state} for index,state in enumerate(torch.cuda.get_rng_state_all())]
    return {
        "python":random.getstate(),"numpy":np.random.get_state(),
        "torch_cpu":torch.get_rng_state(),"torch_cuda_all":cuda_states,
    }
def restore_rng_state(state:Mapping[str,Any])->None:
    random.setstate(state["python"]);np.random.set_state(state["numpy"]);torch.set_rng_state(state["torch_cpu"])
    cuda_states=state.get("torch_cuda_all")
    if cuda_states is None:
        legacy=state.get("torch_cuda_device");cuda_states=[] if legacy is None else [legacy]
    if cuda_states:
        if not torch.cuda.is_available() or max(item["index"] for item in cuda_states)>=torch.cuda.device_count():
            raise RuntimeError("Saved CUDA RNG devices are unavailable")
        for item in cuda_states: torch.cuda.set_rng_state(item["state"],device=item["index"])
def save_checkpoint_atomic(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,scheduler:Any,scaler:Any,sampler:StatefulRandomSampler,progress:Mapping[str,Any],model_config:Mapping[str,Any],tokenizer_fingerprint:str,dataset_fingerprint_value:str,training_config:Mapping[str,Any],history_path:Path,validation_history:Sequence[Mapping[str,Any]],experiment_id:str,device:str|None=None,model_source_sha256:str|None=None)->dict[str,Any]:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    hist=sha256_file(history_path) if Path(history_path).exists() else None
    state={'checkpoint_version':CHECKPOINT_VERSION,'experiment_id':experiment_id,'created_utc':utc_now(),'model_state':unwrap_model(model).state_dict(),'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler is not None else None,'scaler_state':scaler.state_dict() if scaler is not None else None,'sampler_state':sampler.state_dict(),'rng_state':capture_rng_state(device),'progress':dict(progress),'model_config':dict(model_config),'tokenizer_fingerprint':tokenizer_fingerprint,'dataset_fingerprint':dataset_fingerprint_value,'training_config':dict(training_config),'history_sha256':hist,'validation_history':list(validation_history),'model_source_sha256':model_source_sha256}
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: tmp=Path(h.name)
    try:
        torch.save(state,tmp)
        check=torch.load(tmp,map_location='cpu',weights_only=False)
        if check.get('checkpoint_version')!=CHECKPOINT_VERSION or check.get('progress')!=dict(progress): raise RuntimeError('Checkpoint readback failed')
        digest=sha256_file(tmp)
        if path.exists():
            previous=path.with_name('previous_known_good.pt')
            with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: prev_tmp=Path(h.name)
            try:
                shutil.copyfile(path,prev_tmp)
                torch.load(prev_tmp,map_location='cpu',weights_only=False)
                os.replace(prev_tmp,previous)
            finally: prev_tmp.unlink(missing_ok=True)
        os.replace(tmp,path)
        return {'path':str(path),'sha256':digest,'step':progress.get('step')}
    finally: tmp.unlink(missing_ok=True)

In [ ]:
def restore_checkpoint(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,sampler:StatefulRandomSampler,expected_tokenizer_fingerprint:str,expected_dataset_fingerprint:str,scheduler:Any=None,scaler:Any=None,expected_experiment_id:str|None=None,expected_model_config:Mapping[str,Any]|None=None,history_path:Path|None=None,device:str|None=None,expected_model_source_sha256:str|None=None)->dict[str,Any]:
    state=torch.load(Path(path),map_location='cpu',weights_only=False) # Only load your own trusted checkpoint files.
    if state.get('checkpoint_version')!=CHECKPOINT_VERSION: raise ValueError('Checkpoint schema mismatch')
    if expected_experiment_id is not None and state['experiment_id']!=expected_experiment_id: raise ValueError('Experiment ID mismatch')
    if state['tokenizer_fingerprint']!=expected_tokenizer_fingerprint: raise ValueError('Tokenizer fingerprint mismatch')
    if state['dataset_fingerprint']!=expected_dataset_fingerprint: raise ValueError('Dataset fingerprint mismatch')
    if expected_model_config is not None and dict(expected_model_config)!=state['model_config']: raise ValueError('Architecture config mismatch')
    if expected_model_source_sha256 is not None and state.get('model_source_sha256')!=expected_model_source_sha256: raise ValueError('Model source fingerprint mismatch')
    if history_path is not None:
        actual=sha256_file(history_path) if Path(history_path).exists() else None
        if actual!=state['history_sha256']: raise ValueError('History file differs from checkpoint; restore matching history or explicitly reconcile after crash')
    if (scheduler is None)!=(state['scheduler_state'] is None): raise ValueError('Scheduler state incompatibility')
    if (scaler is None)!=(state['scaler_state'] is None): raise ValueError('AMP scaler state incompatibility')
    unwrap_model(model).load_state_dict(state['model_state'],strict=True);optimizer.load_state_dict(state['optimizer_state'])
    if device is not None:
        for opt_state in optimizer.state.values():
            for k,v in opt_state.items():
                if isinstance(v,torch.Tensor): opt_state[k]=v.to(device)
    if scheduler is not None: scheduler.load_state_dict(state['scheduler_state'])
    if scaler is not None: scaler.load_state_dict(state['scaler_state'])
    sampler.load_state_dict(state['sampler_state']);restore_rng_state(state['rng_state'])
    return state

@dataclass
class ProgressState:
    experiment_id:str
    device:str
    total_steps:int
    step:int=0
    epoch:int=0
    tokens_processed:int=0
    cumulative_seconds:float=0.0
    last_checkpoint_step:int=0

class ProgressReporter:
    def __init__(self,state:ProgressState):
        self.state=state;self.start=time.perf_counter();self.base_cumulative=state.cumulative_seconds;self.last=self.start;self.last_step=state.step;self.last_tokens=state.tokens_processed
        self.bar=tqdm(total=state.total_steps,initial=state.step,desc=f'{state.experiment_id} [{state.device}]')
    def update(self,*,step:int,epoch:int,loss:float,learning_rate:float,tokens_processed:int,gradient_norm:float|None):
        now=time.perf_counter();dt=max(now-self.last,1e-9);ds=step-self.last_step;dtoken=tokens_processed-self.last_tokens
        if ds<0 or dtoken<0: raise ValueError('Progress cannot go backwards')
        cumulative=self.base_cumulative+(now-self.start);rate=ds/dt
        self.bar.update(ds);self.bar.set_postfix({'epoch':epoch,'loss':f'{loss:.4f}','lr':f'{learning_rate:.3g}','tok':tokens_processed,'tok/s':f'{dtoken/dt:.0f}','step/s':f'{rate:.2f}','grad':'NA' if gradient_norm is None else f'{gradient_norm:.3f}','elapsed_s':f'{cumulative:.0f}','eta_s':f'{max(0,self.state.total_steps-step)/rate:.0f}' if rate>0 else 'inf','ckpt':self.state.last_checkpoint_step})
        self.state.step=step;self.state.epoch=epoch;self.state.tokens_processed=tokens_processed;self.state.cumulative_seconds=cumulative;self.last=now;self.last_step=step;self.last_tokens=tokens_processed
    def close(self): self.bar.close()

def reconcile_history_to_checkpoint(checkpoint_path:Path,history_path:Path,*,apply:bool=False)->dict[str,Any]:
    checkpoint=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    expected=checkpoint.get('history_sha256');history_path=Path(history_path)
    payload=history_path.read_bytes() if history_path.exists() else b''
    if (hashlib.sha256(payload).hexdigest() if payload else None)==expected:
        return {'status':'already_matching','kept_bytes':len(payload),'discarded_bytes':0}
    boundaries=[0];boundaries.extend(index+1 for index,value in enumerate(payload) if value==10)
    match=None
    for boundary in boundaries:
        prefix=payload[:boundary];digest=hashlib.sha256(prefix).hexdigest() if prefix else None
        if digest==expected: match=boundary
    if match is None: raise RuntimeError('History is not a strict extension of the checkpoint history; restore a matching recovery archive')
    report={'status':'rollback_available','kept_bytes':match,'discarded_bytes':len(payload)-match,'checkpoint_step':checkpoint['progress']['step']}
    if apply:
        backup=history_path.with_name(f"{history_path.name}.uncommitted.after_step_{checkpoint['progress']['step']}.{int(time.time())}")
        atomic_write_bytes(backup,payload);atomic_write_bytes(history_path,payload[:match]);report.update(status='rolled_back',backup=str(backup))
    return report


## 8. Recovery/bootstrap discovery

In [ ]:
def validate_zip(path:Path)->dict[str,Any]:
    with zipfile.ZipFile(path,'r') as z:
        bad=z.testzip()
        if bad is not None: raise zipfile.BadZipFile(f'CRC failure: {bad}')
        return {'path':str(path),'members':z.namelist(),'sha256':sha256_file(path)}
def discover_recovery_archives(input_root:Path)->list[Path]:
    return sorted({*Path(input_root).rglob('phase1_submission03_recovery.zip'), *Path(input_root).rglob('training_recovery.latest.zip')}) if Path(input_root).exists() else []
RECOVERY_CANDIDATES=discover_recovery_archives(Path(CFG.input_root))
print('Recovery candidates:',[str(p) for p in RECOVERY_CANDIDATES])
if len(RECOVERY_CANDIDATES)>1: print('Multiple recovery archives found; choose explicit path.')
def verify_recovery_archive(path:Path)->dict[str,Any]:
    import stat
    with zipfile.ZipFile(path) as z:
        names=z.namelist()
        result={'path':str(path),'members':names,'sha256':sha256_file(path)}
        if len(names)!=len(set(names)): raise ValueError('Duplicate ZIP member names')
        if 'manifests/recovery_integrity.json' not in names: raise ValueError('No recovery member SHA manifest')
        manifest=json.loads(z.read('manifests/recovery_integrity.json'))
        expected={r['path']:r for r in manifest['files']}
        actual=set(n for n in names if not n.endswith('/') and n!='manifests/recovery_integrity.json')
        if len(expected)!=len(manifest['files']) or set(expected)!=actual:
            raise ValueError(f'Manifest/archive mismatch: missing={set(expected)-actual}, extras={actual-set(expected)}')
        for member in z.infolist():
            p=Path(member.filename)
            if p.is_absolute() or '..' in p.parts or not member.filename or chr(92) in member.filename:
                raise ValueError('Unsafe member path')
            if stat.S_ISLNK(member.external_attr>>16): raise ValueError('Symlink in archive is forbidden')
        for name,item in expected.items():
            if z.getinfo(name).file_size!=item['bytes']: raise ValueError(f'Size mismatch: {name}')
            digest=hashlib.sha256()
            with z.open(name) as source:
                for chunk in iter(lambda: source.read(1<<20),b''): digest.update(chunk)
            if digest.hexdigest()!=item['sha256']: raise ValueError(f'SHA mismatch: {name}')
    return result
def restore_recovery_archive(archive_path:Path,destination:Path,*,allow_existing_identical:bool=True)->None:
    verify_recovery_archive(archive_path)
    dest=Path(destination).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as z:
        manifest=json.loads(z.read('manifests/recovery_integrity.json'))
        expected={row['path']:row['sha256'] for row in manifest['files']}
        # Preflight every path and conflict BEFORE extracting anything.
        for member in z.infolist():
            target=(dest/member.filename).resolve()
            if dest not in target.parents and target!=dest: raise ValueError(f'Unsafe path: {member.filename}')
            if member.is_dir(): continue
            if target.exists():
                if not target.is_file(): raise RuntimeError(f'Restore path is not a file: {target}')
                if not allow_existing_identical or expected.get(member.filename)!=sha256_file(target):
                    raise RuntimeError(f'Restore conflict; refusing to overwrite existing file: {target}')
        for member in z.infolist():
            if member.is_dir(): continue
            target=dest/member.filename
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=target.parent,delete=False) as handle:
                temporary=Path(handle.name)
                try:
                    with z.open(member) as source: shutil.copyfileobj(source,handle,length=1<<20)
                    handle.flush();os.fsync(handle.fileno())
                except Exception:
                    temporary.unlink(missing_ok=True);raise
            os.replace(temporary,target)

## 9. Automatic infrastructure and model-contract checks


In [ ]:
def run_infrastructure_smoke_tests() -> dict[str, str]:
    results: dict[str, str] = {}
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name)
        fixture_texts = [
            " ".join(["alpha"] * 700),
            " ".join(["beta"] * 700),
            " ".join(["gamma"] * 700),
        ]
        docs = [
            Document(
                uid=f"fixture:row-{index:012d}", text=text, source="fixture",
                raw_id=f"row-{index:012d}", content_hash=hashlib.sha256(text.encode()).hexdigest(),
                word_count=len(text.split()),
            )
            for index, text in enumerate(fixture_texts)
        ]
        assert stable_raw_id({"_record_index": 0}, 0, {"id_strategy": "record_index", "id_field": None}) == "row-000000000000"
        results["idless_training_schema_strategy"] = "PASS"
        train, validation, split = deterministic_document_split(docs, 1 / 3, 781)
        assert not ({doc.uid for doc in train} & {doc.uid for doc in validation})
        save_or_verify_split(temporary / "split.json", split)
        results["document_split_and_leakage"] = "PASS"
        examples = make_continuation_examples(validation)
        assert len(examples) == 1 and len(examples[0]["prompt"].split()) == 512
        assert len(examples[0]["reference"].split()) == 128
        results["local_continuation_examples"] = "PASS"
        sampler = StatefulRandomSampler(range(10), seed=9)
        iterator = iter(sampler); prefix = [next(iterator) for _ in range(4)]
        state = sampler.state_dict(); resumed = StatefulRandomSampler(range(10), seed=9)
        resumed.load_state_dict(state); suffix = list(resumed)
        assert len(prefix + suffix) == 10 and len(set(prefix + suffix)) == 10
        results["sampler_continuation"] = "PASS"
        history = JsonlHistory(temporary / "history.jsonl")
        history.append({"step": 1, "loss": 1.0})
        assert history.records() == [{"step": 1, "loss": 1.0}]
        results["history_continuity"] = "PASS"
        dummy_prediction = {"1": {"text": " ".join(["word"] * 128)}}
        prediction_path = temporary / "prediction.json"; atomic_write_json(prediction_path, dummy_prediction)
        zip_path = temporary / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        assert validate_zip(zip_path)["members"] == ["prediction.json"]
        results["zip_integrity"] = "PASS"
    return results

SMOKE_RESULTS = run_infrastructure_smoke_tests()
print(json.dumps(SMOKE_RESULTS, indent=2))


In [ ]:
def validate_continuation_causality(model, vocab_size: int, device: str) -> None:
    model.eval(); prefix = 5
    first = torch.randint(0, vocab_size, (1, 12), dtype=torch.long, device=device)
    second = first.clone(); second[:, 9:] = torch.randint(0, vocab_size, (1, 3), device=device)
    mask = torch.ones_like(first, dtype=torch.bool)
    lengths = torch.tensor([prefix], dtype=torch.long, device=device)
    with torch.no_grad():
        a = hybrid_logits(model, first, mask, lengths)[:, :9]
        b = hybrid_logits(model, second, mask, lengths)[:, :9]
    if not torch.allclose(a, b, atol=1e-5, rtol=1e-5):
        raise AssertionError("Future continuation tokens leaked into earlier logits")

def validate_bidirectional_prefix(model, vocab_size: int, device: str) -> None:
    model.eval(); prefix = 6
    first = torch.randint(0, vocab_size, (1, 11), dtype=torch.long, device=device)
    second = first.clone(); second[:, prefix - 1] = (second[:, prefix - 1] + 1) % vocab_size
    mask = torch.ones_like(first, dtype=torch.bool)
    lengths = torch.tensor([prefix], dtype=torch.long, device=device)
    with torch.no_grad():
        a = hybrid_logits(model, first, mask, lengths)[:, 0]
        b = hybrid_logits(model, second, mask, lengths)[:, 0]
    if torch.allclose(a, b, atol=1e-7, rtol=1e-7):
        raise AssertionError("Earlier prefix representation is insensitive to a later prefix token")

def validate_prefix_conditioning(model, vocab_size: int, device: str) -> None:
    model.eval(); prefix = 6
    first = torch.randint(0, vocab_size, (1, 11), dtype=torch.long, device=device)
    second = first.clone(); second[:, :prefix] = torch.randint(0, vocab_size, (1, prefix), device=device)
    mask = torch.ones_like(first, dtype=torch.bool)
    lengths = torch.tensor([prefix], dtype=torch.long, device=device)
    with torch.no_grad():
        a = hybrid_logits(model, first, mask, lengths)[:, prefix:]
        b = hybrid_logits(model, second, mask, lengths)[:, prefix:]
    if torch.allclose(a, b, atol=1e-7, rtol=1e-7):
        raise AssertionError("Continuation logits are insensitive to the prompt prefix")

def run_student_integration_smoke_test(model, tokenizer, model_config, reconstruct_model, device=None):
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    meta = tokenizer_metadata(tokenizer, DIRS["tokenizer"])
    report = validate_hybrid_contract(model, meta["vocab_size"], device)
    validate_continuation_causality(model, meta["vocab_size"], device)
    report["continuation_future_leakage"] = "PASS"
    validate_bidirectional_prefix(model, meta["vocab_size"], device)
    report["bidirectional_prefix"] = "PASS"
    validate_prefix_conditioning(model, meta["vocab_size"], device)
    report["prefix_conditioning"] = "PASS"
    probe = tokenizer_encode(tokenizer, "A constitutional court considered the appeal.")
    if not probe or min(probe) < 0 or max(probe) >= meta["vocab_size"]:
        raise AssertionError("Tokenizer emitted an out-of-vocabulary ID")
    report["token_id_range"] = "PASS"
    limit = int(model_config["max_sequence_length"])
    try:
        too_long = torch.zeros((1, limit + 1), dtype=torch.long, device=device)
        hybrid_logits(model, too_long, torch.ones_like(too_long, dtype=torch.bool),
                      torch.tensor([limit], dtype=torch.long, device=device))
        raise AssertionError("Context limit not enforced")
    except (ValueError, RuntimeError):
        report["context_limit"] = "PASS"
    clone = reconstruct_model(model_config)
    clone.load_state_dict(model.state_dict(), strict=True)
    report["reconstruction"] = "PASS"; del clone
    model.to("cpu")
    if str(device).startswith("cuda"): torch.cuda.empty_cache()
    atomic_write_json(DIRS["manifests"] / "student_integration_smoke_test.json", report)
    return report

if MODEL is not None and TOKENIZER_META is not None:
    INTEGRATION_SMOKE_RESULTS = run_student_integration_smoke_test(
        MODEL, tokenizer, MODEL_CONFIG, RECONSTRUCT_MODEL
    )
    print(json.dumps(INTEGRATION_SMOKE_RESULTS, indent=2))
else:
    INTEGRATION_SMOKE_RESULTS = None
    print("Student prefix-hybrid integration smoke test is waiting for model/tokenizer registration.")


## 10. Resumable prefix-hybrid continuation training


In [ ]:
# Frozen before checkpoint 1. DataParallel scatters 8 examples as 4 per T4;
# four accumulation passes preserve an effective batch of 32 sequences.
TRAINING_CONFIG = {
    "experiment_id": CFG.experiment_id,
    "objective": "target-token cross-entropy under a bidirectional-prefix/causal-continuation mask",
    "optimizer": "AdamW",
    "learning_rate": 2.5e-4,
    "betas": [0.9, 0.95],
    "epsilon": 1e-8,
    "weight_decay": 0.1,
    "schedule": "linear_warmup_cosine_decay",
    "warmup_steps": 500,
    "min_lr_ratio": 0.1,
    "batch_size": 8,
    "per_gpu_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "source_length": 512,
    "target_length": 256,
    "max_steps": 10_000,
    "samples_per_epoch": 32_000,
    "gradient_clip_norm": 1.0,
    "gradient_checkpointing": False,
    "validation_interval": 500,
    "validation_sequences_per_source": 16,
    "parallelism": "data_parallel",
    "gpu_ids": [0, 1],
    "checkpoint_interval": 500,
    "recovery_interval": 1000,
    "num_workers": 0,
    "source_mixture": dict(CFG.source_mixing_ratios),
    "source_mixture_unit": "document draws, resampled deterministically per epoch",
    "seed": CFG.seed,
    "precision_policy": "native bf16 on SM80+; fp16+GradScaler on T4/CUDA; fp32 on CPU",
}

def validate_student_training_config(config: Mapping[str, Any]) -> dict[str, Any]:
    required = (
        "learning_rate", "betas", "epsilon", "weight_decay", "warmup_steps", "min_lr_ratio",
        "batch_size", "gradient_accumulation_steps", "source_length", "target_length",
        "max_steps", "samples_per_epoch", "gradient_clip_norm", "gradient_checkpointing",
        "validation_interval", "validation_sequences_per_source", "checkpoint_interval",
        "recovery_interval",
    )
    missing = [key for key in required if config.get(key) is None]
    if missing: raise RuntimeError(f"Student training configuration is incomplete: {missing}")
    if config.get("optimizer") != "AdamW": raise ValueError("This infrastructure validates AdamW only")
    if config.get("schedule") != "linear_warmup_cosine_decay":
        raise ValueError("This infrastructure validates linear_warmup_cosine_decay only")
    positive = (
        "learning_rate", "epsilon", "batch_size", "gradient_accumulation_steps", "source_length",
        "target_length", "max_steps", "samples_per_epoch", "gradient_clip_norm",
        "validation_interval", "validation_sequences_per_source", "checkpoint_interval", "recovery_interval",
    )
    if any(float(config[key]) <= 0 for key in positive): raise ValueError("Positive quantities must be > 0")
    if int(config["num_workers"]) != 0:
        raise ValueError("num_workers must remain 0 for exact sampler restoration")
    if len(config["betas"]) != 2 or not all(0 <= float(value) < 1 for value in config["betas"]):
        raise ValueError("AdamW betas must contain two values in [0, 1)")
    if float(config["weight_decay"]) < 0: raise ValueError("weight_decay must be non-negative")
    if not 0 <= float(config["min_lr_ratio"]) <= 1: raise ValueError("min_lr_ratio must be in [0, 1]")
    if int(config["warmup_steps"]) >= int(config["max_steps"]):
        raise ValueError("warmup_steps must be smaller than max_steps")
    if config.get("parallelism") != "data_parallel":
        raise ValueError("Submission 03 is configured for data_parallel")
    gpu_ids = [int(value) for value in config.get("gpu_ids", [])]
    if gpu_ids != [0, 1]: raise ValueError("Dual-T4 configuration requires gpu_ids [0, 1]")
    if int(config["batch_size"]) != int(config["per_gpu_batch_size"]) * len(gpu_ids):
        raise ValueError("batch_size must equal per_gpu_batch_size times the GPU count")
    if MODEL_CONFIG is not None:
        total = int(config["source_length"]) + int(config["target_length"])
        if total > int(MODEL_CONFIG["max_sequence_length"]):
            raise ValueError("source_length + target_length exceeds model max_sequence_length")
        if int(CFG.decoding["max_new_tokens"]) + 2 > int(MODEL_CONFIG["max_sequence_length"]):
            raise ValueError("Decoding budget leaves no room for a prompt and BOS")
    return dict(config)

def save_or_verify_json(path: Path, value: Any) -> None:
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != value:
            raise RuntimeError(f"Existing immutable manifest differs: {path}")
    else:
        atomic_write_json(path, value)

def append_event(path: Path, event: Mapping[str, Any]) -> None:
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("ab") as handle:
        handle.write((json.dumps(dict(event), ensure_ascii=False) + "\n").encode("utf-8"))
        handle.flush(); os.fsync(handle.fileno())

class PrefixHybridDataset(Dataset):
    def __init__(self, documents, tokenizer, ratios, samples, source_length, target_length, seed, cache_documents=128):
        self.documents = list(documents); self.tokenizer = tokenizer; self.ratios = dict(ratios)
        self.samples = int(samples); self.source_length = int(source_length); self.target_length = int(target_length)
        self.seed = int(seed); self.epoch = 0; self.cache_documents = int(cache_documents); self.cache = {}
        self.pad_id = getattr(tokenizer, "pad_token_id", None)
        self.bos_id = getattr(tokenizer, "bos_token_id", None)
        self.eos_id = getattr(tokenizer, "eos_token_id", None)
        if None in (self.pad_id, self.bos_id, self.eos_id):
            raise ValueError("Prefix-hybrid training requires pad, bos, and eos IDs")
        if len({int(self.pad_id), int(self.bos_id), int(self.eos_id)}) != 3:
            raise ValueError("pad, bos, and eos IDs must be distinct")
        self.sequence_length = self.source_length + self.target_length
        self._resample()
    def _resample(self):
        self.document_indices = source_sampling_indices(
            self.documents, self.ratios, self.samples, self.seed + self.epoch
        )
    def __len__(self): return self.samples
    def set_epoch(self, epoch: int): self.epoch = int(epoch); self._resample()
    def _tokens(self, document_index: int) -> list[int]:
        if document_index not in self.cache:
            if len(self.cache) >= self.cache_documents: self.cache.pop(next(iter(self.cache)))
            self.cache[document_index] = tokenizer_encode(
                self.tokenizer, document_text(self.documents[document_index])
            )
        return self.cache[document_index]
    def __getitem__(self, index: int):
        document_index = self.document_indices[int(index)]
        document = self.documents[document_index]
        ids = self._tokens(document_index)
        needed = self.source_length + self.target_length
        start = 0
        if len(ids) >= needed:
            rng = random.Random(self.seed + self.epoch * len(self) + int(index))
            start = rng.randrange(0, len(ids) - needed + 1)
            source = list(ids[start:start + self.source_length])
            target = list(ids[start + self.source_length:start + needed])
        elif len(ids) >= 2:
            split = min(self.source_length, max(1, len(ids) - 1))
            source = list(ids[:split]); target = list(ids[split:split + self.target_length])
        else:
            source = list(ids) or [int(self.eos_id)]; target = []
        if len(target) < self.target_length: target = [*target, int(self.eos_id)]
        target = target[:self.target_length]
        prefix_length = len(source)
        valid_length = prefix_length + len(target)
        input_ids = [*source, int(self.bos_id), *target[:-1]]
        labels = [-100] * prefix_length + target
        input_ids = (input_ids + [int(self.pad_id)] * self.sequence_length)[:self.sequence_length]
        labels = (labels + [-100] * self.sequence_length)[:self.sequence_length]
        attention_mask = [True] * valid_length + [False] * (self.sequence_length - valid_length)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.bool),
            "prefix_lengths": torch.tensor(prefix_length, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "document_index": document_index,
            "document_source": document.source,
            "document_uid": document.uid,
            "window_start": start,
            "processed_tokens": prefix_length + len(target),
        }

def build_fixed_hybrid_validation_rows(documents, tokenizer, config):
    rows, manifest = [], []
    per_source = int(config["validation_sequences_per_source"])
    for source in sorted({document.source for document in documents}):
        source_documents = [document for document in documents if document.source == source]
        ordered = _stable_document_order(source_documents, int(config["seed"]) + 10_000)
        for local_index, document in enumerate(ordered):
            if len([row for row in manifest if row["source"] == source]) >= per_source: break
            dataset = PrefixHybridDataset(
                [document], tokenizer, {source: 1.0}, 1,
                int(config["source_length"]), int(config["target_length"]),
                int(config["seed"]) + 10_000 + local_index,
            )
            row = dataset[0]
            if int((row["labels"] != -100).sum()) == 0: continue
            rows.append(row)
            manifest.append({
                "source": source,
                "document_uid": document.uid,
                "window_start": int(row["window_start"]),
                "prefix_tokens": int(row["prefix_lengths"]),
                "target_tokens": int((row["labels"] != -100).sum()),
            })
    expected = per_source * len({document.source for document in documents})
    if len(rows) != expected:
        raise RuntimeError(f"Could create only {len(rows)} fixed validation rows; expected {expected}")
    return rows, manifest

def choose_precision(device: str):
    if not str(device).startswith("cuda"): return None, "fp32", False
    if torch.cuda.get_device_capability(torch.device(device))[0] >= 8 and torch.cuda.is_bf16_supported():
        return torch.bfloat16, "bf16", False
    return torch.float16, "fp16", True

def make_scaler(enabled: bool):
    try: return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError): return torch.cuda.amp.GradScaler(enabled=enabled)

def move_to_device(tensor:torch.Tensor,device:str)->torch.Tensor:
    return tensor.to(device,non_blocking=str(device).startswith("cuda"))

class ReplicaLocalTokenLoss(torch.nn.Module):
    """Compute summed token loss per GPU so DataParallel never gathers full logits."""
    def __init__(self,base_model:torch.nn.Module):
        super().__init__();self.base_model=base_model
    def forward(self,input_ids,attention_mask,prefix_lengths,labels=None):
        logits=hybrid_logits(self.base_model,input_ids,attention_mask,prefix_lengths)
        if labels is None: return logits
        loss_sum=torch.nn.functional.cross_entropy(
            logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),
            ignore_index=-100,reduction="sum",
        )
        target_count=(labels!=-100).sum()
        return loss_sum.reshape(1),target_count.reshape(1)

def prepare_parallel_training_model(model:torch.nn.Module,config:Mapping[str,Any],device:str):
    gpu_ids=[int(value) for value in config["gpu_ids"]]
    if not torch.cuda.is_available() or torch.cuda.device_count()<=max(gpu_ids):
        raise RuntimeError(
            f"Dual-T4 training requires CUDA devices {gpu_ids}; found {torch.cuda.device_count()}"
        )
    primary=f"cuda:{gpu_ids[0]}"
    if str(device)!=primary:
        raise ValueError(f"Primary device must be {primary}, got {device}")
    base=unwrap_model(model).to(primary)
    if hasattr(base,"set_gradient_checkpointing"):
        base.set_gradient_checkpointing(bool(config["gradient_checkpointing"]))
    training_replica=ReplicaLocalTokenLoss(base)
    wrapped=torch.nn.DataParallel(training_replica,device_ids=gpu_ids,output_device=gpu_ids[0],dim=0)
    return wrapped,gpu_ids

def make_adamw(parameters,config:Mapping[str,Any],device:str):
    kwargs={
        "lr":config["learning_rate"],"betas":tuple(config["betas"]),
        "eps":config["epsilon"],"weight_decay":config["weight_decay"],
    }
    if str(device).startswith("cuda"):
        try: return torch.optim.AdamW(parameters,**kwargs,fused=True),"fused"
        except (TypeError,RuntimeError): pass
    return torch.optim.AdamW(parameters,**kwargs),"standard"

def scaler_update_succeeded(before: float, after: float) -> bool:
    return float(after) >= float(before)

def learning_rate_multiplier(step: int, max_steps: int, warmup_steps: int, min_ratio: float) -> float:
    if step < warmup_steps: return max(step, 1) / max(warmup_steps, 1)
    progress = min(1.0, (step - warmup_steps) / max(1, max_steps - warmup_steps))
    return float(min_ratio) + (1.0 - float(min_ratio)) * 0.5 * (1.0 + math.cos(math.pi * progress))

def atomic_copy(source: Path, destination: Path) -> None:
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent, suffix=destination.suffix, delete=False) as handle:
        temporary = Path(handle.name)
    try:
        shutil.copyfile(source, temporary)
        if sha256_file(source) != sha256_file(temporary): raise IOError("Atomic copy verification failed")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

def save_inference_checkpoint(path, model, model_config, tokenizer_meta, dataset_fp, step, validation_loss):
    state = {
        "format": "phase1_submission03_prefix_hybrid_inference_v1",
        "created_utc": utc_now(), "model_state": unwrap_model(model).state_dict(),
        "model_config": dict(model_config), "model_source_sha256": MODEL_SOURCE_SHA256,
        "tokenizer_fingerprint": tokenizer_meta["fingerprint"], "dataset_fingerprint": dataset_fp,
        "step": int(step), "validation_loss": None if validation_loss is None else float(validation_loss),
    }
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".pt", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        torch.save(state, temporary)
        check = torch.load(temporary, map_location="cpu", weights_only=False)
        if check.get("format") != state["format"] or check.get("model_source_sha256") != MODEL_SOURCE_SHA256:
            raise RuntimeError("Inference checkpoint readback failed")
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return {"path": str(path), "sha256": sha256_file(path), "step": int(step)}

def export_training_recovery_snapshot(root: Path) -> Path:
    root = Path(root).resolve(); output = DIRS["recovery"] / "training_recovery.latest.zip"
    include = []
    for relative in ("model", "tokenizer", "manifests", "histories", "checkpoints"):
        include.extend(path for path in (root / relative).rglob("*") if path.is_file())
    include = sorted({path.resolve() for path in include if path.resolve() != output.resolve()})
    manifest = {
        "created_utc": utc_now(),
        "files": [
            {"path": path.relative_to(root).as_posix(), "bytes": path.stat().st_size,
             "sha256": sha256_file(path)}
            for path in include if path.name != "recovery_integrity.json"
        ],
    }
    with tempfile.NamedTemporaryFile(dir=output.parent, suffix=".zip", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_STORED, allowZip64=True) as archive:
            for item in manifest["files"]:
                compression = zipfile.ZIP_STORED if item["path"].endswith(".pt") else zipfile.ZIP_DEFLATED
                archive.write(root / item["path"], item["path"], compress_type=compression)
            archive.writestr("manifests/recovery_integrity.json", json.dumps(manifest, indent=2) + "\n")
        verify_recovery_archive(temporary)
        os.replace(temporary, output)
    finally:
        temporary.unlink(missing_ok=True)
    atomic_write_json(DIRS["recovery"] / "training_recovery.latest.json", {
        "created_utc": utc_now(), "path": str(output),
        "sha256": sha256_file(output), "bytes": output.stat().st_size,
    })
    return output

@torch.no_grad()
def evaluate_hybrid_loss(model, loader, device, amp_dtype):
    was_training = model.training; model.eval(); loss_sum = 0.0; token_count = 0
    for batch in loader:
        input_ids = move_to_device(batch["input_ids"], device)
        attention_mask = move_to_device(batch["attention_mask"], device)
        prefix_lengths = move_to_device(batch["prefix_lengths"], device)
        labels = move_to_device(batch["labels"], device)
        with torch.autocast(device_type=torch.device(device).type, dtype=amp_dtype, enabled=amp_dtype is not None):
            loss_parts,target_parts=model(
                input_ids=input_ids,attention_mask=attention_mask,
                prefix_lengths=prefix_lengths,labels=labels,
            )
            loss=loss_parts.sum()
        loss_sum += float(loss.cpu());token_count += int(target_parts.sum().cpu())
    model.train(was_training)
    if not token_count: raise ValueError("Validation loader contained no targets")
    return loss_sum / token_count

def train_submission03(model, tokenizer, train_docs, val_docs, model_config, tokenizer_meta,
                       *, device="cuda:0", resume=False):
    cfg = validate_student_training_config(TRAINING_CONFIG)
    if not CFG.full_training_authorized:
        raise RuntimeError("Full training has not been authorized")
    latest = DIRS["checkpoints"] / "checkpoint.latest.pt"
    if resume and not latest.exists():
        raise FileNotFoundError("resume_train requires checkpoint.latest.pt; refusing step-zero restart")
    if not resume and latest.exists():
        raise RuntimeError("fresh_train found checkpoint.latest.pt; use resume_train or a new workspace")
    save_or_verify_json(DIRS["manifests"] / "training_config.json", cfg)
    split_meta = json.loads((DIRS["manifests"] / "document_split.json").read_text(encoding="utf-8"))
    dataset_fp = split_meta["dataset_fingerprint"]
    ratios = validate_source_mixture(cfg["source_mixture"], (document.source for document in train_docs))
    dataset = PrefixHybridDataset(
        train_docs, tokenizer, ratios, cfg["samples_per_epoch"], cfg["source_length"],
        cfg["target_length"], cfg["seed"],
    )
    sampler = StatefulRandomSampler(dataset, seed=cfg["seed"])
    loader = DataLoader(
        dataset, batch_size=cfg["batch_size"], sampler=sampler, num_workers=0,
        pin_memory=str(device).startswith("cuda"), drop_last=True,
    )
    validation_rows, validation_manifest = build_fixed_hybrid_validation_rows(val_docs, tokenizer, cfg)
    save_or_verify_json(DIRS["manifests"] / "validation_sequence_manifest.json", {
        "protocol": "fixed held-out prefix/continuation windows", "rows": validation_manifest,
    })
    validation_loader = DataLoader(
        validation_rows,batch_size=cfg["batch_size"],shuffle=False,num_workers=0,
        pin_memory=True,
    )
    model,active_gpu_ids=prepare_parallel_training_model(model,cfg,device)
    optimizer,optimizer_backend=make_adamw(unwrap_model(model).parameters(),cfg,device)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: learning_rate_multiplier(
            step, cfg["max_steps"], cfg["warmup_steps"], cfg["min_lr_ratio"]
        ),
    )
    amp_dtype, actual_precision, use_scaler = choose_precision(device)
    scaler = make_scaler(use_scaler); checkpoint_scaler = scaler if use_scaler else None
    history_path = DIRS["histories"] / "training.jsonl"; history = JsonlHistory(history_path)
    progress = ProgressState(cfg["experiment_id"], device, cfg["max_steps"])
    validation_history = []
    if resume:
        preview = torch.load(latest, map_location="cpu", weights_only=False)
        if preview.get("training_config") != cfg: raise RuntimeError("Checkpoint training config differs")
        if preview.get("model_source_sha256") != MODEL_SOURCE_SHA256:
            raise RuntimeError("Checkpoint model source differs")
        reconcile_history_to_checkpoint(latest, history_path, apply=True)
        state = restore_checkpoint(
            latest, model=model, optimizer=optimizer, scheduler=scheduler,
            scaler=checkpoint_scaler, sampler=sampler,
            expected_tokenizer_fingerprint=tokenizer_meta["fingerprint"],
            expected_dataset_fingerprint=dataset_fp,
            expected_experiment_id=cfg["experiment_id"],
            expected_model_config=model_config, history_path=history_path,
            device=device, expected_model_source_sha256=MODEL_SOURCE_SHA256,
        )
        progress = ProgressState(**state["progress"])
        validation_history = list(state.get("validation_history", []))
        dataset.set_epoch(sampler.epoch)
        print("Restored optimizer step:", progress.step)
    elif history.records():
        raise RuntimeError("Training history exists without a matching fresh/resume decision")
    reporter = ProgressReporter(progress); iterator = iter(loader)
    best_loss = min((float(row["loss"]) for row in validation_history), default=math.inf)
    try:
        while reporter.state.step < cfg["max_steps"]:
            model.train(); optimizer.zero_grad(set_to_none=True)
            microbatches, valid_targets, processed_tokens = [], 0, 0
            for _ in range(cfg["gradient_accumulation_steps"]):
                try: batch = next(iterator)
                except StopIteration:
                    sampler.next_epoch(); dataset.set_epoch(sampler.epoch)
                    iterator = iter(loader); batch = next(iterator)
                valid_targets += int((batch["labels"] != -100).sum())
                processed_tokens += int(batch["processed_tokens"].sum())
                microbatches.append(batch)
            if valid_targets <= 0: raise ValueError("Accumulation window has no target tokens")
            numerator = 0.0
            for batch in microbatches:
                input_ids = move_to_device(batch["input_ids"], device)
                attention_mask = move_to_device(batch["attention_mask"], device)
                prefix_lengths = move_to_device(batch["prefix_lengths"], device)
                labels = move_to_device(batch["labels"], device)
                with torch.autocast(
                    device_type=torch.device(device).type, dtype=amp_dtype,
                    enabled=amp_dtype is not None,
                ):
                    loss_parts,target_parts=model(
                        input_ids=input_ids,attention_mask=attention_mask,
                        prefix_lengths=prefix_lengths,labels=labels,
                    )
                    loss_sum=loss_parts.sum()
                    backward_loss = loss_sum / valid_targets
                if not torch.isfinite(loss_sum): raise FloatingPointError("Non-finite training loss")
                (scaler.scale(backward_loss) if use_scaler else backward_loss).backward()
                numerator += float(loss_sum.detach().cpu())
            if use_scaler: scaler.unscale_(optimizer)
            gradient_norm = float(torch.nn.utils.clip_grad_norm_(
                model.parameters(), cfg["gradient_clip_norm"]
            ).cpu())
            learning_rate = float(optimizer.param_groups[0]["lr"])
            if use_scaler:
                before = float(scaler.get_scale()); scaler.step(optimizer); scaler.update()
                if not scaler_update_succeeded(before, scaler.get_scale()):
                    append_event(DIRS["histories"] / "overflow_events.jsonl", {
                        "attempted_step": reporter.state.step + 1, "epoch": sampler.epoch,
                        "scale_before": before, "scale_after": float(scaler.get_scale()),
                        "created_utc": utc_now(),
                    })
                    continue
            else:
                optimizer.step()
            scheduler.step(); step = reporter.state.step + 1
            total_tokens = reporter.state.tokens_processed + processed_tokens
            mean_loss = numerator / valid_targets
            validation_loss = None; improved = False
            if step % cfg["validation_interval"] == 0 or step == cfg["max_steps"]:
                validation_loss = evaluate_hybrid_loss(model, validation_loader, device, amp_dtype)
                validation_history.append({
                    "step": step, "loss": validation_loss,
                    "perplexity": math.exp(validation_loss) if validation_loss < 709 else math.inf,
                })
                improved = validation_loss < best_loss; best_loss = min(best_loss, validation_loss)
            reporter.update(
                step=step, epoch=sampler.epoch, loss=mean_loss, learning_rate=learning_rate,
                tokens_processed=total_tokens, gradient_norm=gradient_norm,
            )
            history.append({
                "step": step, "epoch": sampler.epoch, "loss": mean_loss,
                "validation_loss": validation_loss, "learning_rate": learning_rate,
                "gradient_norm": gradient_norm, "tokens_processed": total_tokens,
                "tokens_per_second": total_tokens / max(reporter.state.cumulative_seconds, 1e-9),
                "optimizer_step_seconds": reporter.state.cumulative_seconds / max(step, 1),
                "gpu_memory_gib": (
                    torch.cuda.max_memory_allocated(torch.device(device)) / 2**30
                    if str(device).startswith("cuda") else None
                ),
                "precision": actual_precision,
                "parallelism": cfg["parallelism"],
                "gpu_ids": active_gpu_ids,
                "optimizer_backend": optimizer_backend,
                "gpu_memory_gib_by_device": {
                    str(gpu): torch.cuda.max_memory_allocated(gpu) / 2**30
                    for gpu in active_gpu_ids
                },
                "checkpoint_step": reporter.state.last_checkpoint_step,
                "created_utc": utc_now(),
            })
            if step % cfg["checkpoint_interval"] == 0 or improved or step == cfg["max_steps"]:
                reporter.state.last_checkpoint_step = step
                save_checkpoint_atomic(
                    latest, model=model, optimizer=optimizer, scheduler=scheduler,
                    scaler=checkpoint_scaler, sampler=sampler, progress=asdict(reporter.state),
                    model_config=model_config, tokenizer_fingerprint=tokenizer_meta["fingerprint"],
                    dataset_fingerprint_value=dataset_fp, training_config=cfg,
                    history_path=history_path, validation_history=validation_history,
                    experiment_id=cfg["experiment_id"], device=device,
                    model_source_sha256=MODEL_SOURCE_SHA256,
                )
                if improved:
                    save_inference_checkpoint(
                        DIRS["checkpoints"] / "selected_model.pt", model, model_config,
                        tokenizer_meta, dataset_fp, step, validation_loss,
                    )
                if step % cfg["recovery_interval"] == 0:
                    export_training_recovery_snapshot(ROOT)
        final_loss = validation_history[-1]["loss"] if validation_history else None
        save_inference_checkpoint(
            DIRS["checkpoints"] / "model_final.pt", model, model_config,
            tokenizer_meta, dataset_fp, reporter.state.step, final_loss,
        )
        if not (DIRS["checkpoints"] / "selected_model.pt").exists():
            atomic_copy(DIRS["checkpoints"] / "model_final.pt", DIRS["checkpoints"] / "selected_model.pt")
        export_training_recovery_snapshot(ROOT)
    finally:
        reporter.close()
    result = {
        "completed_step": reporter.state.step,
        "tokens_processed": reporter.state.tokens_processed,
        "best_validation_loss": best_loss,
        "actual_precision": actual_precision,
        "parallelism": cfg["parallelism"],
        "gpu_ids": active_gpu_ids,
        "per_gpu_batch_size": cfg["per_gpu_batch_size"],
        "optimizer_backend": optimizer_backend,
        "latest_checkpoint": str(latest),
        "selected_checkpoint": str(DIRS["checkpoints"] / "selected_model.pt"),
    }
    atomic_write_json(DIRS["manifests"] / "training_result.json", result)
    return result

def run_gpu_training_smoke_test(device="cuda:0"):
    if not torch.cuda.is_available(): raise RuntimeError("Representative smoke test requires CUDA")
    cfg = validate_student_training_config(TRAINING_CONFIG)
    if MODEL is None or tokenizer is None or not TRAIN_DOCS or not VAL_DOCS:
        raise RuntimeError("Model, tokenizer, and bounded corpus cache are required")
    validation_rows, validation_manifest = build_fixed_hybrid_validation_rows(VAL_DOCS, tokenizer, cfg)
    save_or_verify_json(DIRS["manifests"] / "validation_sequence_manifest.json", {
        "protocol": "fixed held-out prefix/continuation windows", "rows": validation_manifest,
    })
    tiny = PrefixHybridDataset(
        TRAIN_DOCS, tokenizer, cfg["source_mixture"], max(2, cfg["batch_size"] * 2),
        cfg["source_length"], cfg["target_length"], cfg["seed"],
    )
    sampler = StatefulRandomSampler(tiny, seed=cfg["seed"])
    batch = next(iter(DataLoader(
        tiny,batch_size=cfg["batch_size"],sampler=sampler,num_workers=0,
        drop_last=True,pin_memory=True,
    )))
    smoke_model,active_gpu_ids=prepare_parallel_training_model(
        RECONSTRUCT_MODEL(MODEL_CONFIG),cfg,device
    )
    optimizer,optimizer_backend=make_adamw(
        unwrap_model(smoke_model).parameters(),cfg,device
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: learning_rate_multiplier(step, 10, 1, cfg["min_lr_ratio"])
    )
    amp_dtype, precision, use_scaler = choose_precision(device); scaler = make_scaler(use_scaler)
    input_ids = move_to_device(batch["input_ids"], device)
    attention_mask = move_to_device(batch["attention_mask"], device)
    prefix_lengths = move_to_device(batch["prefix_lengths"], device)
    labels = move_to_device(batch["labels"], device)
    for gpu in active_gpu_ids: torch.cuda.reset_peak_memory_stats(gpu)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
        loss_parts,target_parts=smoke_model(
            input_ids=input_ids,attention_mask=attention_mask,
            prefix_lengths=prefix_lengths,labels=labels,
        )
        if int(target_parts.sum().detach().cpu())!=int((labels!=-100).sum()):
            raise AssertionError("Replica-local smoke target count mismatch")
        loss=loss_parts.sum()/target_parts.sum()
    (scaler.scale(loss) if use_scaler else loss).backward()
    if use_scaler: scaler.unscale_(optimizer)
    gradient_norm = float(torch.nn.utils.clip_grad_norm_(
        smoke_model.parameters(), cfg["gradient_clip_norm"]
    ).cpu())
    if use_scaler:
        before = float(scaler.get_scale()); scaler.step(optimizer); scaler.update()
        if not scaler_update_succeeded(before, scaler.get_scale()):
            raise RuntimeError("FP16 smoke update overflowed")
    else:
        optimizer.step()
    scheduler.step(); loss_value = float(loss.detach().cpu())
    split_meta = json.loads((DIRS["manifests"] / "document_split.json").read_text(encoding="utf-8"))
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name); history = JsonlHistory(temporary / "history.jsonl")
        history.append({"step": 1, "loss": loss_value})
        checkpoint = temporary / "smoke.pt"
        progress = ProgressState(
            "submission03_gpu_smoke", device, 1, step=1,
            tokens_processed=int(batch["processed_tokens"].sum()), last_checkpoint_step=1,
        )
        save_checkpoint_atomic(
            checkpoint, model=smoke_model, optimizer=optimizer, scheduler=scheduler,
            scaler=scaler if use_scaler else None, sampler=sampler,
            progress=asdict(progress), model_config=MODEL_CONFIG,
            tokenizer_fingerprint=TOKENIZER_META["fingerprint"],
            dataset_fingerprint_value=split_meta["dataset_fingerprint"],
            training_config={"smoke": True}, history_path=history.path,
            validation_history=[], experiment_id="submission03_gpu_smoke",
            device=device, model_source_sha256=MODEL_SOURCE_SHA256,
        )
        del loss_parts,target_parts,loss;torch.cuda.empty_cache()
        clone = RECONSTRUCT_MODEL(MODEL_CONFIG)
        clone_optimizer,_=make_adamw(clone.parameters(),cfg,"cpu")
        clone_scheduler = torch.optim.lr_scheduler.LambdaLR(
            clone_optimizer, lambda step: learning_rate_multiplier(step, 10, 1, cfg["min_lr_ratio"])
        )
        clone_scaler = make_scaler(use_scaler)
        clone_sampler = StatefulRandomSampler(tiny, seed=cfg["seed"])
        restored = restore_checkpoint(
            checkpoint, model=clone, optimizer=clone_optimizer, scheduler=clone_scheduler,
            scaler=clone_scaler if use_scaler else None, sampler=clone_sampler,
            expected_tokenizer_fingerprint=TOKENIZER_META["fingerprint"],
            expected_dataset_fingerprint=split_meta["dataset_fingerprint"],
            expected_experiment_id="submission03_gpu_smoke", expected_model_config=MODEL_CONFIG,
            history_path=history.path, device=None,
            expected_model_source_sha256=MODEL_SOURCE_SHA256,
        )
        if restored["progress"]["step"] != 1 or not clone_optimizer.state:
            raise AssertionError("Checkpoint did not restore model, optimizer, or progress")
    report = {
        "status": "PASS", "loss": loss_value, "gradient_norm": gradient_norm,
        "precision": precision,
        "parallelism": cfg["parallelism"], "gpu_ids": active_gpu_ids,
        "per_gpu_batch_size": cfg["per_gpu_batch_size"],
        "optimizer_backend": optimizer_backend,
        "peak_gpu_memory_gib_by_device": {
            str(gpu): torch.cuda.max_memory_allocated(gpu) / 2**30
            for gpu in active_gpu_ids
        },
        "source_length": cfg["source_length"], "target_length": cfg["target_length"],
        "batch_size": cfg["batch_size"], "checkpoint_restore": "PASS", "created_utc": utc_now(),
    }
    atomic_write_json(DIRS["manifests"] / "gpu_training_smoke_test.json", report)
    del smoke_model, optimizer, clone, clone_optimizer; torch.cuda.empty_cache()
    return report

RUN_GPU_TRAINING_SMOKE_TEST = False
GPU_TRAINING_SMOKE_RESULT = (
    run_gpu_training_smoke_test("cuda:0") if RUN_GPU_TRAINING_SMOKE_TEST else None
)

def ensure_validation_sequence_manifest():
    """
    Create the deterministic validation sequence manifest before
    the full-training preflight checks for its existence.
    """
    if tokenizer is None:
        raise RuntimeError(
            "Tokenizer unavailable; cannot build validation manifest"
        )

    if not VAL_DOCS:
        raise RuntimeError(
            "Validation documents unavailable; cannot build validation manifest"
        )

    cfg = validate_student_training_config(TRAINING_CONFIG)

    _, validation_manifest = build_fixed_hybrid_validation_rows(
        VAL_DOCS,
        tokenizer,
        cfg,
    )

    manifest_value = {
        "protocol": "fixed held-out prefix/continuation windows",
        "rows": validation_manifest,
    }

    save_or_verify_json(
        DIRS["manifests"] / "validation_sequence_manifest.json",
        manifest_value,
    )

    print(
        "Validation sequence manifest ready:",
        DIRS["manifests"] / "validation_sequence_manifest.json",
    )


def preflight_full_training():
    problems = []

    if CFG.execution_mode not in {
        "auto_train",
        "fresh_train",
        "resume_train",
    }:
        problems.append(
            "execution_mode is not a training mode"
        )

    if not CFG.full_training_authorized:
        problems.append(
            "full_training_authorized is False"
        )

    architecture_text = json.dumps(
        CFG.architecture_record,
        ensure_ascii=False,
    ).upper()

    if "REQUIRED BEFORE TRAINING" in architecture_text:
        problems.append(
            "architecture_record still contains required student placeholders"
        )

    if CFG.innovation_hypothesis.startswith("STUDENT MUST"):
        problems.append(
            "innovation_hypothesis is still a placeholder"
        )

    if CFG.ablation_control.startswith("STUDENT MUST"):
        problems.append(
            "ablation_control is still a placeholder"
        )

    try:
        validate_student_training_config(
            TRAINING_CONFIG
        )
    except Exception as exc:
        problems.append(str(exc))

    if MODEL is None or tokenizer is None:
        problems.append(
            "model/tokenizer unavailable"
        )

    if not TRAIN_DOCS or not VAL_DOCS:
        problems.append(
            "bounded train/validation cache unavailable"
        )

    required_manifests = (
        "document_split.json",
        "model_validation.json",
        "student_integration_smoke_test.json",
        "validation_sequence_manifest.json",
    )

    for name in required_manifests:
        manifest_path = (
            DIRS["manifests"] / name
        )

        if not manifest_path.exists():
            problems.append(
                f"missing manifest {name}"
            )

    integration_path = (
        DIRS["manifests"]
        / "student_integration_smoke_test.json"
    )

    if integration_path.exists():
        integration = json.loads(
            integration_path.read_text(
                encoding="utf-8"
            )
        )

        required_contract_checks = (
            "continuation_future_leakage",
            "bidirectional_prefix",
            "prefix_conditioning",
            "reconstruction",
        )

        for key in required_contract_checks:
            if integration.get(key) != "PASS":
                problems.append(
                    f"model contract failed: {key}"
                )

    if problems:
        raise RuntimeError(
            "Full-training preflight failed:\n- "
            + "\n- ".join(problems)
        )

    print("Full-training preflight: PASS")


RUN_FULL_TRAINING = True


if RUN_FULL_TRAINING:
    # The preflight requires this manifest, so create/verify it first.
    ensure_validation_sequence_manifest()

    # Now all required artifacts exist and can be checked.
    preflight_full_training()

    resume_training = (
        CFG.execution_mode == "resume_train"
        or (
            CFG.execution_mode == "auto_train"
            and (
                DIRS["checkpoints"]
                / "checkpoint.latest.pt"
            ).exists()
        )
    )

    print(
        "Training mode:",
        "RESUME" if resume_training else "FRESH",
    )

    TRAINING_RESULT = train_submission03(
        MODEL,
        tokenizer,
        TRAIN_DOCS,
        VAL_DOCS,
        MODEL_CONFIG,
        TOKENIZER_META,
        device="cuda:0",
        resume=resume_training,
    )

    print(
        "Full training completed:",
        TRAINING_RESULT,
    )

else:
    TRAINING_RESULT = None
    print("Training disabled for this run.")

## 11. Local evaluation plumbing

In [ ]:
EXPECTED_METRIC_FIELDS=('bertscore_f1','rouge_l_f1','bleu_4')
METRIC_ALIASES={
    'bertscore_f1':'bertscore_f1','bert_score_f1':'bertscore_f1','bertscore':'bertscore_f1',
    'rouge_l_f1':'rouge_l_f1','rougel_f1':'rouge_l_f1','rouge-l_f1':'rouge_l_f1',
    'bleu_4':'bleu_4','bleu4':'bleu_4','bleu-4':'bleu_4',
}
def perplexity_from_loss(loss): return float(math.exp(loss)) if loss<709 else math.inf
def normalize_official_metrics(raw:Mapping[str,Any])->dict[str,float]:
    normalized={}
    for key,value in raw.items():
        canonical=METRIC_ALIASES.get(str(key).strip().lower().replace(' ','_'))
        if canonical: normalized[canonical]=float(value)
    missing=[key for key in EXPECTED_METRIC_FIELDS if key not in normalized]
    if missing: raise ValueError(f'Official evaluator result is missing required metrics {missing}; raw keys={list(raw)}')
    return {key:normalized[key] for key in EXPECTED_METRIC_FIELDS}
def run_official_evaluator(evaluator,predictions,references,checkpoint_id):
    if len(predictions)!=len(references) or not predictions: raise ValueError('Predictions/references must be nonempty and aligned')
    metrics=normalize_official_metrics(evaluator(predictions,references));record={'checkpoint_id':checkpoint_id,'created_utc':utc_now(),'metrics':metrics,'examples':len(predictions)}
    atomic_write_json(DIRS['evaluations']/f'{checkpoint_id}.json',record);csv_path=DIRS['evaluations']/'metrics.csv';write_header=not csv_path.exists()
    expected_header=['checkpoint_id','created_utc','examples',*EXPECTED_METRIC_FIELDS]
    if csv_path.exists():
        with csv_path.open('r',newline='',encoding='utf-8') as existing_handle:
            existing_header=next(csv.reader(existing_handle),None)
        if existing_header!=expected_header: raise RuntimeError(f'Existing metrics.csv schema differs: {existing_header}')
    with csv_path.open('a',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=expected_header)
        if write_header: writer.writeheader()
        writer.writerow({'checkpoint_id':checkpoint_id,'created_utc':record['created_utc'],'examples':len(predictions),**metrics})
    return record

def load_official_evaluator(path:Path,function_name:str,argument_order:str):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    if not function_name: raise ValueError('Set CS781_EVALUATOR_FUNCTION to the inspected official callable name')
    if argument_order not in ('predictions_references','references_predictions'): raise ValueError('Set CS781_EVALUATOR_ARGUMENT_ORDER explicitly')
    module=import_source_module(f'cs781_official_evaluator_{sha256_file(path)[:12]}',path);function=getattr(module,function_name,None)
    if not callable(function): raise AttributeError(f'{path} has no callable {function_name!r}')
    def adapted(predictions,references):
        result=function(predictions,references) if argument_order=='predictions_references' else function(references,predictions)
        if not isinstance(result,Mapping): raise TypeError(f'Official evaluator must return a mapping, got {type(result).__name__}')
        return result
    return adapted

def discover_evaluator_candidates(root:Path)->list[Path]:
    if not Path(root).exists(): return []
    return sorted(path for path in Path(root).rglob('*.py') if any(term in path.name.lower() for term in ('eval','metric','score')) and ROOT.resolve() not in path.resolve().parents)

OFFICIAL_EVALUATOR_PATH=os.environ.get('CS781_EVALUATOR_PATH');OFFICIAL_EVALUATOR_FUNCTION=os.environ.get('CS781_EVALUATOR_FUNCTION')
OFFICIAL_EVALUATOR_ARGUMENT_ORDER=os.environ.get('CS781_EVALUATOR_ARGUMENT_ORDER')
OFFICIAL_EVALUATOR=(load_official_evaluator(Path(OFFICIAL_EVALUATOR_PATH),OFFICIAL_EVALUATOR_FUNCTION,OFFICIAL_EVALUATOR_ARGUMENT_ORDER) if OFFICIAL_EVALUATOR_PATH else None)
if OFFICIAL_EVALUATOR is None:
    EVALUATOR_CANDIDATES=discover_evaluator_candidates(Path(CFG.input_root));print('Inspect official evaluator candidates:',[str(path) for path in EVALUATOR_CANDIDATES])
    print('Set CS781_EVALUATOR_PATH, CS781_EVALUATOR_FUNCTION and CS781_EVALUATOR_ARGUMENT_ORDER explicitly.')


In [ ]:
def plot_saved_history(history_path: Path, output_path: Path) -> None:
    import matplotlib.pyplot as plt
    records = JsonlHistory(history_path).records()
    if not records:
        raise ValueError("No history records to plot")
    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    fields = [("loss", "Training loss"), ("learning_rate", "Learning rate"), ("tokens_per_second", "Tokens/s"), ("gpu_memory_gib", "GPU memory GiB")]
    for axis, (field_name, title) in zip(axes.flat, fields):
        x, y = zip(*[(row["step"], row[field_name]) for row in records if row.get(field_name) is not None]) if any(row.get(field_name) is not None for row in records) else ([], [])
        axis.plot(x, y)
        axis.set_title(title)
        axis.set_xlabel("Optimizer step")
        axis.grid(alpha=0.25)
    figure.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=160)
    plt.close(figure)

if (DIRS["histories"] / "training.jsonl").exists():
    plot_saved_history(DIRS["histories"] / "training.jsonl", DIRS["plots"] / "training_history.png")

## 12. Resumable prefix-hybrid generation and prediction management


In [ ]:
WORD_PATTERN = re.compile(r"\S+")
def words(text: str) -> list[str]: return WORD_PATTERN.findall(text.strip())
def exact_word_prefix(text: str, count: int = 128) -> str:
    units = words(text)
    if len(units) < count: raise ValueError(f"Only {len(units)} words, need {count}")
    return " ".join(units[:count])

def _policy_value(policy: Mapping[str, Any], name: str, default=None):
    value = policy.get(name, default)
    return default if value is None else value

def select_next_token(logits: torch.Tensor, generated: Sequence[int], policy: Mapping[str, Any], generator):
    scores = logits.float().clone()
    repetition_penalty = float(_policy_value(policy, "repetition_penalty", 1.0))
    if repetition_penalty <= 0: raise ValueError("repetition_penalty must be positive")
    for token in set(int(value) for value in generated):
        scores[token] = scores[token] / repetition_penalty if scores[token] > 0 else scores[token] * repetition_penalty
    ngram = int(_policy_value(policy, "no_repeat_ngram_size", 0))
    if ngram > 0 and len(generated) >= ngram - 1:
        prefix = tuple(generated[-(ngram - 1):]) if ngram > 1 else tuple()
        banned = set()
        for index in range(len(generated) - ngram + 1):
            gram = tuple(generated[index:index + ngram])
            if gram[:-1] == prefix: banned.add(gram[-1])
        if banned: scores[list(banned)] = -torch.inf
    strategy = str(policy.get("strategy"))
    if strategy == "greedy": return int(scores.argmax())
    if strategy not in {"top_k", "top_p", "top_k_top_p"}:
        raise ValueError(f"Unsupported declared decoding strategy: {strategy}")
    temperature = float(_policy_value(policy, "temperature", 1.0))
    if temperature <= 0: raise ValueError("temperature must be positive")
    scores = scores / temperature
    top_k = int(_policy_value(policy, "top_k", 0))
    if top_k > 0:
        threshold = torch.topk(scores, min(top_k, scores.numel())).values[-1]
        scores[scores < threshold] = -torch.inf
    top_p = float(_policy_value(policy, "top_p", 1.0))
    if not 0 < top_p <= 1: raise ValueError("top_p must be in (0, 1]")
    if top_p < 1:
        sorted_scores, sorted_indices = torch.sort(scores, descending=True)
        probabilities = torch.softmax(sorted_scores, dim=-1)
        cumulative = probabilities.cumsum(dim=-1)
        remove = cumulative > top_p; remove[1:] = remove[:-1].clone(); remove[0] = False
        scores[sorted_indices[remove]] = -torch.inf
    probabilities = torch.softmax(scores, dim=-1)
    if not torch.isfinite(probabilities).all() or float(probabilities.sum()) <= 0:
        raise FloatingPointError("Invalid sampling distribution")
    return int(torch.multinomial(probabilities, 1, generator=generator))

def generation_quality_report(text: str) -> dict[str, Any]:
    units = words(text); lowered = [unit.lower() for unit in units]
    counts = Counter(lowered)
    return {
        "word_count": len(units),
        "unique_word_ratio": len(counts) / max(1, len(units)),
        "max_word_fraction": max(counts.values(), default=0) / max(1, len(units)),
    }


@torch.inference_mode()
def hybrid_continuation(model, tokenizer, prompt, device, *, policy, target_words=128, max_new_tokens=None):
    if max_new_tokens is None: raise ValueError("A declared max_new_tokens is required")
    max_sequence_length = int(MODEL_CONFIG.get("max_sequence_length", 0))
    if max_sequence_length <= 2: raise ValueError("MODEL_CONFIG must declare max_sequence_length")
    prompt_budget = max_sequence_length - int(max_new_tokens) - 1
    if prompt_budget < 1:
        raise ValueError("max_new_tokens leaves no room for a prompt prefix and BOS token")
    prompt_ids = tokenizer_encode(tokenizer, prompt)
    if not prompt_ids: raise ValueError("Prompt encoded to no tokens")
    prompt_ids = prompt_ids[-prompt_budget:]
    bos_id = getattr(tokenizer, "bos_token_id", None)
    eos_id = getattr(tokenizer, "eos_token_id", None)
    pad_id = getattr(tokenizer, "pad_token_id", None)
    if bos_id is None: raise ValueError("Prefix-hybrid inference requires bos_token_id")
    forbidden = {int(value) for value in (bos_id, pad_id) if value is not None}
    generated = []; generator = torch.Generator(device=device).manual_seed(int(policy["seed"]))
    model = model.to(device); model.eval()
    use_cache = callable(getattr(model, "prefill_prefix", None)) and callable(getattr(model, "decode_cached", None))
    if use_cache:
        prefix_tensor = torch.tensor([prompt_ids], dtype=torch.long, device=device)
        caches = model.prefill_prefix(prefix_tensor)
        current_id = int(bos_id)
    for _ in range(int(max_new_tokens)):
        if use_cache:
            current_tensor = torch.tensor([[current_id]], dtype=torch.long, device=device)
            next_logits, caches = model.decode_cached(current_tensor, caches)
            logits = next_logits[0].float()
        else:
            sequence = [*prompt_ids, int(bos_id), *generated]
            input_ids = torch.tensor([sequence], dtype=torch.long, device=device)
            attention_mask = torch.ones_like(input_ids, dtype=torch.bool)
            prefix_lengths = torch.tensor([len(prompt_ids)], dtype=torch.long, device=device)
            logits = hybrid_logits(model, input_ids, attention_mask, prefix_lengths)[0, -1].float()
        if forbidden: logits[list(forbidden)] = -torch.inf
        next_id = select_next_token(logits, generated, policy, generator)
        generated.append(next_id)
        if use_cache: current_id = next_id
        continuation = tokenizer_decode(tokenizer, generated)
        if len(words(continuation)) >= target_words + 1:
            result = exact_word_prefix(continuation, target_words)
            report = generation_quality_report(result)
            if report["unique_word_ratio"] < float(_policy_value(policy, "min_unique_word_ratio", 0.0)):
                raise RuntimeError(f"Declared uniqueness threshold failed: {report}")
            if report["max_word_fraction"] > float(_policy_value(policy, "max_word_fraction", 1.0)):
                raise RuntimeError(f"Declared repetition threshold failed: {report}")
            return result, report
        if eos_id is not None and next_id == int(eos_id):
            raise RuntimeError(f"Premature EOS after {len(words(continuation))} words")
    raise RuntimeError(f"Generation ended before {target_words} verified words")

def load_test_records(path: Path, schema: Mapping[str, Any]) -> list[dict[str, str]]:
    id_field, prompt_field = schema["id_field"], schema["text_field"]
    result, seen = [], set()
    for index, row in enumerate(iter_json_records(path, id_field=id_field, text_field=prompt_field)):
        identifier = str(row[id_field]); prompt = normalize_text(row[prompt_field])
        if identifier in seen: raise ValueError(f"Duplicate official ID: {identifier}")
        seen.add(identifier); result.append({"id": identifier, "prompt": prompt})
    return result

def test_records_fingerprint(records):
    digest = hashlib.sha256()
    for row in records:
        digest.update(f"{row['id']}\t{hashlib.sha256(row['prompt'].encode()).hexdigest()}\n".encode())
    return digest.hexdigest()


In [ ]:
def load_selected_model_for_inference(checkpoint_path, tokenizer_meta, reconstruct_model, device):
    checkpoint_path = Path(checkpoint_path)
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if state.get("tokenizer_fingerprint") != tokenizer_meta["fingerprint"]:
        raise RuntimeError("Selected checkpoint tokenizer mismatch")
    if state.get("model_source_sha256") != MODEL_SOURCE_SHA256:
        raise RuntimeError("Selected checkpoint model-source mismatch")
    restored = reconstruct_model(state["model_config"])
    restored.load_state_dict(state["model_state"], strict=True)
    return restored.to(device).eval(), state

def generate_official_predictions(model, tokenizer, test_records, device, *, checkpoint_id, tokenizer_meta):
    policies = list(CFG.decoding.get("policies") or [])
    max_new_tokens = CFG.decoding.get("max_new_tokens")
    if not policies or max_new_tokens is None:
        raise RuntimeError("Declare ordered decoding policies and max_new_tokens before inference")
    names = [str(policy.get("name", "")) for policy in policies]
    if not all(names) or len(names) != len(set(names)):
        raise ValueError("Every decoding policy requires a unique nonempty name")
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", checkpoint_id)
    partial_path = DIRS["predictions"] / f"official_predictions.{safe}.partial.json"
    meta_path = partial_path.with_suffix(".meta.json")
    attempts_path = DIRS["predictions"] / f"generation_attempts.{safe}.jsonl"
    expected_meta = {
        "checkpoint_id": checkpoint_id,
        "checkpoint_sha256": sha256_file(DIRS["checkpoints"] / "selected_model.pt"),
        "model_source_sha256": MODEL_SOURCE_SHA256,
        "tokenizer_fingerprint": tokenizer_meta["fingerprint"],
        "test_fingerprint": test_records_fingerprint(test_records),
        "decoding": dict(CFG.decoding),
    }
    predictions, failures = {}, []
    if partial_path.exists():
        if not meta_path.exists() or json.loads(meta_path.read_text(encoding="utf-8")) != expected_meta:
            raise RuntimeError("Partial predictions do not match checkpoint/tokenizer/test/decoding provenance")
        predictions = json.loads(partial_path.read_text(encoding="utf-8"))
        expected_ids = {row["id"] for row in test_records}
        if set(predictions) - expected_ids: raise RuntimeError("Partial predictions contain foreign IDs")
        for identifier, value in predictions.items():
            if not isinstance(value, dict) or set(value) != {"text"}:
                raise RuntimeError(f"Invalid cached prediction structure: {identifier}")
            if len(words(value["text"])) != int(CFG.decoding["target_words"]):
                raise RuntimeError(f"Invalid cached word count: {identifier}")
    else:
        atomic_write_json(meta_path, expected_meta)
    for row in tqdm(test_records, desc="Official test generation"):
        if row["id"] in predictions: continue
        errors = []
        for policy_index, policy in enumerate(policies):
            started = time.time()
            try:
                continuation, quality = hybrid_continuation(
                    model, tokenizer, row["prompt"], device, policy=policy,
                    target_words=int(CFG.decoding["target_words"]),
                    max_new_tokens=max_new_tokens,
                )
                append_event(attempts_path, {
                    "id": row["id"], "policy_index": policy_index, "policy": policy["name"],
                    "status": "success", "quality": quality,
                    "seconds": time.time() - started, "created_utc": utc_now(),
                })
                predictions[row["id"]] = {"text": continuation}
                atomic_write_json(partial_path, predictions)
                break
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"; errors.append(error)
                append_event(attempts_path, {
                    "id": row["id"], "policy_index": policy_index, "policy": policy["name"],
                    "status": "failure", "error": error,
                    "seconds": time.time() - started, "created_utc": utc_now(),
                })
        if row["id"] not in predictions: failures.append({"id": row["id"], "errors": errors})
    atomic_write_json(DIRS["predictions"] / "generation_failures.json", failures)
    if failures: raise RuntimeError(f"Generation failed for {len(failures)} examples; no ZIP will be created")
    provenance = {
        **expected_meta, "prediction_count": len(predictions),
        "partial_prediction_sha256": sha256_file(partial_path), "created_utc": utc_now(),
    }
    atomic_write_json(DIRS["predictions"] / "inference_provenance.json", provenance)
    return predictions, failures

def build_local_generation_examples(documents, examples_per_source=4):
    examples = []
    for source in sorted({document.source for document in documents}):
        collected = 0
        for document in _stable_document_order(
            [item for item in documents if item.source == source], CFG.seed + 20_000
        ):
            units = words(document_text(document))
            if len(units) < 640: continue
            examples.append({
                "id": f"{source}:{document.uid}",
                "source": source,
                "prompt": " ".join(units[:512]),
                "reference": " ".join(units[512:640]),
            })
            collected += 1
            if collected >= int(examples_per_source): break
        if collected < int(examples_per_source):
            raise RuntimeError(f"Insufficient long held-out documents for local evaluation source {source}")
    return examples

def evaluate_local_continuations(model, tokenizer, examples, evaluator, device, checkpoint_id):
    if evaluator is None: raise RuntimeError("Load and inspect the professor-supplied evaluator first")
    policies = list(CFG.decoding.get("policies") or [])
    if not policies: raise RuntimeError("Declare decoding policies before local evaluation")
    generated, references = [], []
    for row in tqdm(examples, desc="Held-out continuation evaluation"):
        last_error = None
        for policy in policies:
            try:
                text, _ = hybrid_continuation(
                    model, tokenizer, row["prompt"], device, policy=policy,
                    target_words=int(CFG.decoding["target_words"]),
                    max_new_tokens=CFG.decoding["max_new_tokens"],
                )
                generated.append(text); references.append(row["reference"]); break
            except Exception as exc:
                last_error = exc
        else:
            raise RuntimeError(f"All declared policies failed for {row['id']}: {last_error}")
    return run_official_evaluator(evaluator, generated, references, checkpoint_id)

@torch.inference_mode()
def validate_cached_decoding(model, tokenizer, prompt, device):
    if not callable(getattr(model, "prefill_prefix", None)) or not callable(getattr(model, "decode_cached", None)):
        raise RuntimeError("The selected model source has no cached-decoding interface")
    budget = int(MODEL_CONFIG["max_sequence_length"]) - int(CFG.decoding["max_new_tokens"]) - 1
    prompt_ids = tokenizer_encode(tokenizer, prompt)[-min(128, budget):]
    if not prompt_ids: raise ValueError("Cannot validate cached decoding on an empty prompt")
    model.eval()
    caches = model.prefill_prefix(torch.tensor([prompt_ids], dtype=torch.long, device=device))
    current = [int(tokenizer.bos_token_id), int(prompt_ids[-1])]
    sequence = list(prompt_ids)
    largest_error = 0.0
    for token_id in current:
        sequence.append(token_id)
        cached_logits, caches = model.decode_cached(
            torch.tensor([[token_id]], dtype=torch.long, device=device), caches
        )
        full_ids = torch.tensor([sequence], dtype=torch.long, device=device)
        full_logits = hybrid_logits(
            model, full_ids, torch.ones_like(full_ids, dtype=torch.bool),
            torch.tensor([len(prompt_ids)], dtype=torch.long, device=device),
        )[0, -1]
        error = float((cached_logits[0] - full_logits).abs().max().cpu())
        largest_error = max(largest_error, error)
        if not torch.allclose(cached_logits[0], full_logits, atol=2e-3, rtol=2e-3):
            raise AssertionError(f"Cached/full decoding diverged; max absolute error={error}")
    report = {"status": "PASS", "probe_tokens": len(current),
              "prompt_tokens": len(prompt_ids), "max_absolute_error": largest_error}
    atomic_write_json(DIRS["manifests"] / "cached_inference_validation.json", report)
    return report

def run_phase1_inference_pipeline(device="cuda:0", run_local_evaluation=False):
    if MODEL_SOURCE_SHA256 is None or RECONSTRUCT_MODEL is None or TOKENIZER_META is None:
        raise RuntimeError("Restore tokenizer and authenticated student model source before inference")
    checkpoint_path = DIRS["checkpoints"] / "selected_model.pt"
    if not checkpoint_path.exists(): raise FileNotFoundError("selected_model.pt is required")
    selected_model, state = load_selected_model_for_inference(
        checkpoint_path, TOKENIZER_META, RECONSTRUCT_MODEL, device
    )
    checkpoint_id = f"selected-{sha256_file(checkpoint_path)[:12]}"
    local_evaluation = None
    if run_local_evaluation:
        if not VAL_DOCS: raise RuntimeError("Optional local evaluation requires held-out validation documents")
        examples = build_local_generation_examples(VAL_DOCS)
        atomic_write_json(DIRS["evaluations"] / "fixed_local_examples.json", examples)
        local_evaluation = evaluate_local_continuations(
            selected_model, tokenizer, examples, OFFICIAL_EVALUATOR, device, checkpoint_id
        )
    test_records = load_test_records(
        require_unique_file(DISCOVERED, CFG.test_file), SCHEMAS["test"]
    )
    if not test_records: raise RuntimeError("The official test set is empty")
    print("Cached decoding validation:", validate_cached_decoding(
        selected_model, tokenizer, test_records[0]["prompt"], device))
    predictions, failures = generate_official_predictions(
        selected_model, tokenizer, test_records, device,
        checkpoint_id=checkpoint_id, tokenizer_meta=TOKENIZER_META,
    )
    return selected_model, local_evaluation, test_records, predictions, failures

RUN_INFERENCE = True
RUN_OPTIONAL_LOCAL_EVALUATION = False
if RUN_INFERENCE:
    MODEL, EVAL_RECORD, TEST_RECORDS, PREDICTIONS, FAILURES = run_phase1_inference_pipeline(
        "cuda:0", run_local_evaluation=RUN_OPTIONAL_LOCAL_EVALUATION
    )
else:
    print("Inference disabled for this run.")


In [ ]:
# ================================================================
# CS781 SUBMISSION 03
# FAST DUAL-T4 POST-RESTART INFERENCE + ALL DELIVERABLES
#
# Assumptions:
#   - Training already finished.
#   - /kaggle/working persisted after kernel restart.
#   - selected_model.pt / model_final.pt / tokenizer etc. exist.
#   - Official dataset is still attached under /kaggle/input.
#
# THIS CELL DOES NOT TRAIN.
# ================================================================

from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import hashlib
import importlib.util
import json
import os
import re
import sys
import tempfile
import time
import zipfile

import torch


# ================================================================
# SPEED SETTINGS
# ================================================================

ROOT = Path(
    "/kaggle/working/CS781_Capstone/phase1/submission_03"
)

# Keep the last N prompt tokens.
# Original model supports context 1280.
PROMPT_CAP = 768

# Prompts are grouped into equal-length buckets so they can be
# decoded together.
PROMPT_BUCKET = 64

# Number of sequences generated simultaneously PER GPU.
#
# 8 is conservative for T4.
# If GPU memory is comfortable you can later try 12.
BATCH_PER_GPU = 8

# Do NOT decode generated text after every token.
# Check every 16 generated tokens instead.
WORD_CHECK_EVERY = 16

TARGET_WORDS = 128

# Primary generation budget.
PRIMARY_MAX_NEW = 448

# Only failed samples use this larger retry budget.
RETRY_MAX_NEW = 512


# ================================================================
# WORKSPACE
# ================================================================

if not ROOT.exists():
    raise FileNotFoundError(
        f"Missing trained workspace:\n{ROOT}"
    )


DIRS = {
    name: ROOT / name
    for name in (
        "model",
        "tokenizer",
        "data",
        "checkpoints",
        "histories",
        "evaluations",
        "predictions",
        "plots",
        "recovery",
        "manifests",
        "submission",
    )
}


for path in DIRS.values():
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ================================================================
# BASIC HELPERS
# ================================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(
    path,
    chunk_size=1 << 20,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as handle:

        while True:
            block = handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def atomic_json(
    path,
    value,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        suffix=".tmp",
        delete=False,
    ) as handle:

        temporary = Path(
            handle.name
        )

        json.dump(
            value,
            handle,
            ensure_ascii=False,
            indent=2,
        )

        handle.write("\n")

        handle.flush()

        os.fsync(
            handle.fileno()
        )

    os.replace(
        temporary,
        path,
    )


def import_module(
    path,
):
    path = Path(path)

    spec = (
        importlib.util
        .spec_from_file_location(
            "fast_prefix_hybrid_lm",
            str(path),
        )
    )

    if (
        spec is None
        or spec.loader is None
    ):
        raise ImportError(
            f"Cannot import {path}"
        )

    module = (
        importlib.util
        .module_from_spec(
            spec
        )
    )

    sys.modules[
        spec.name
    ] = module

    spec.loader.exec_module(
        module
    )

    return module


# ================================================================
# REQUIRED TRAINED FILES
# ================================================================

selected_checkpoint = (
    DIRS["checkpoints"]
    / "selected_model.pt"
)

final_checkpoint = (
    DIRS["checkpoints"]
    / "model_final.pt"
)

tokenizer_json = (
    DIRS["tokenizer"]
    / "tokenizer.json"
)

tokenizer_meta_path = (
    DIRS["tokenizer"]
    / "tokenizer_metadata.json"
)

schema_path = (
    DIRS["manifests"]
    / "resolved_json_schemas.json"
)

model_source_path = (
    DIRS["model"]
    / "prefix_hybrid_lm.py"
)


# Fallback to attached source if it isn't
# stored inside /kaggle/working.
if not model_source_path.exists():

    candidates = list(
        Path("/kaggle/input").rglob(
            "prefix_hybrid_lm.py"
        )
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            "Expected exactly one "
            "prefix_hybrid_lm.py.\n"
            f"Found: {len(candidates)}"
        )

    model_source_path = (
        candidates[0]
    )


required_files = (
    selected_checkpoint,
    final_checkpoint,
    tokenizer_json,
    tokenizer_meta_path,
    schema_path,
    model_source_path,
)


for path in required_files:

    if not Path(path).exists():
        raise FileNotFoundError(
            f"Required file missing:\n"
            f"{path}"
        )


# ================================================================
# GPU CHECK
# ================================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required."
    )


GPU_COUNT = min(
    torch.cuda.device_count(),
    2,
)


if GPU_COUNT < 1:
    raise RuntimeError(
        "No CUDA GPU detected."
    )


print("=" * 72)

print(
    "FAST CS781 INFERENCE"
)

print("=" * 72)

print(
    "GPUs:",
    [
        torch.cuda.get_device_name(i)
        for i in range(
            GPU_COUNT
        )
    ],
)


# ================================================================
# TOKENIZER
# ================================================================

from tokenizers import Tokenizer


class CourseBPETokenizer:

    def __init__(
        self,
        backend,
    ):
        self.backend = backend

        self.pad_token_id = (
            backend.token_to_id(
                "<pad>"
            )
        )

        self.unk_token_id = (
            backend.token_to_id(
                "<unk>"
            )
        )

        self.bos_token_id = (
            backend.token_to_id(
                "<bos>"
            )
        )

        self.eos_token_id = (
            backend.token_to_id(
                "<eos>"
            )
        )

    def __len__(
        self,
    ):
        return int(
            self.backend.get_vocab_size()
        )

    def encode(
        self,
        text,
        add_special_tokens=False,
    ):
        ids = list(
            self.backend.encode(
                str(text),
                add_special_tokens=False,
            ).ids
        )

        if add_special_tokens:
            ids = [
                self.bos_token_id,
                *ids,
                self.eos_token_id,
            ]

        return ids

    def decode(
        self,
        ids,
        skip_special_tokens=True,
    ):
        return self.backend.decode(
            [
                int(value)
                for value
                in ids
            ],
            skip_special_tokens=(
                skip_special_tokens
            ),
        )


tokenizer = CourseBPETokenizer(
    Tokenizer.from_file(
        str(
            tokenizer_json
        )
    )
)


TOKENIZER_META = json.loads(
    tokenizer_meta_path.read_text(
        encoding="utf-8"
    )
)


if len(tokenizer) != int(
    TOKENIZER_META[
        "vocab_size"
    ]
):
    raise RuntimeError(
        "Tokenizer vocabulary size "
        "does not match metadata."
    )


print(
    "Tokenizer restored:",
    len(tokenizer),
)


# ================================================================
# LOAD TRAINED CHECKPOINT
# ================================================================

MODEL_SOURCE_SHA256 = (
    sha256_file(
        model_source_path
    )
)


MODEL_MODULE = import_module(
    model_source_path
)


checkpoint_state = torch.load(
    selected_checkpoint,
    map_location="cpu",
    weights_only=False,
)


if (
    checkpoint_state.get(
        "tokenizer_fingerprint"
    )
    != TOKENIZER_META.get(
        "fingerprint"
    )
):
    raise RuntimeError(
        "Checkpoint/tokenizer "
        "fingerprint mismatch."
    )


saved_model_hash = (
    checkpoint_state.get(
        "model_source_sha256"
    )
)


if (
    saved_model_hash
    and saved_model_hash
    != MODEL_SOURCE_SHA256
):
    raise RuntimeError(
        "Checkpoint was trained "
        "with a different model source."
    )


MODEL_CONFIG = dict(
    checkpoint_state[
        "model_config"
    ]
)


# ================================================================
# ONE FP16 MODEL COPY PER GPU
# ================================================================

MODELS = []


for gpu_index in range(
    GPU_COUNT
):

    device = (
        f"cuda:{gpu_index}"
    )

    model = (
        MODEL_MODULE
        .build_model(
            MODEL_CONFIG
        )
    )

    model.load_state_dict(
        checkpoint_state[
            "model_state"
        ],
        strict=True,
    )

    # Critical speed improvement on T4:
    model = (
        model
        .eval()
        .to(device)
        .half()
    )

    MODELS.append(
        model
    )


print(
    "Checkpoint step:",
    checkpoint_state.get(
        "step"
    ),
)

print(
    "FP16 model loaded on",
    GPU_COUNT,
    "GPU(s).",
)


# Release unnecessary CPU checkpoint tensors.
del checkpoint_state

torch.cuda.empty_cache()


# ================================================================
# TEST SCHEMA
# ================================================================

resolved_schema = json.loads(
    schema_path.read_text(
        encoding="utf-8"
    )
)


if "test" not in resolved_schema:
    raise KeyError(
        "resolved_json_schemas.json "
        "does not contain 'test'."
    )


test_schema = (
    resolved_schema[
        "test"
    ]
)


id_field = (
    test_schema[
        "id_field"
    ]
)

text_field = (
    test_schema[
        "text_field"
    ]
)


# ================================================================
# FIND OFFICIAL TEST FILE
# ================================================================

test_candidates = []


for filename in (
    "Phase1- Test.json",
    "test.json",
):

    test_candidates.extend(
        Path("/kaggle/input").rglob(
            filename
        )
    )


test_candidates = sorted(
    {
        path.resolve()
        for path
        in test_candidates
        if path.is_file()
    }
)


if len(test_candidates) != 1:

    raise RuntimeError(
        "Expected exactly one "
        "official test file.\n"
        f"Found {len(test_candidates)}:\n"
        + "\n".join(
            map(
                str,
                test_candidates,
            )
        )
    )


TEST_PATH = (
    test_candidates[0]
)


print(
    "Test file:",
    TEST_PATH,
)


# ================================================================
# LOAD TEST JSON
# ================================================================

raw_test = json.loads(
    TEST_PATH.read_text(
        encoding="utf-8"
    )
)


if isinstance(
    raw_test,
    list,
):

    test_sequence = (
        raw_test
    )


elif isinstance(
    raw_test,
    dict,
):

    wrapper_array = next(
        (
            value
            for value
            in raw_test.values()
            if isinstance(
                value,
                list,
            )
        ),
        None,
    )

    if wrapper_array is not None:

        test_sequence = (
            wrapper_array
        )

    else:

        test_sequence = []

        for key, value in (
            raw_test.items()
        ):

            if isinstance(
                value,
                dict,
            ):

                record = dict(
                    value
                )

                record.setdefault(
                    id_field,
                    str(key),
                )

                test_sequence.append(
                    record
                )

            elif isinstance(
                value,
                str,
            ):

                test_sequence.append(
                    {
                        id_field:
                            str(key),

                        text_field:
                            value,
                    }
                )

else:

    raise TypeError(
        "Unsupported test JSON root."
    )


def normalize_text(
    text,
):
    return " ".join(
        str(text).split()
    )


TEST_RECORDS = []

seen_ids = set()


for record in test_sequence:

    identifier = str(
        record[
            id_field
        ]
    )

    if identifier in seen_ids:
        raise ValueError(
            f"Duplicate ID: "
            f"{identifier}"
        )

    seen_ids.add(
        identifier
    )

    TEST_RECORDS.append(
        {
            "id":
                identifier,

            "prompt":
                normalize_text(
                    record[
                        text_field
                    ]
                ),
        }
    )


if not TEST_RECORDS:
    raise RuntimeError(
        "Official test set is empty."
    )


print(
    "Official test records:",
    len(TEST_RECORDS),
)


# ================================================================
# WORD HELPERS
# ================================================================

WORD_PATTERN = re.compile(
    r"\S+"
)


def words(
    text,
):
    return WORD_PATTERN.findall(
        str(text).strip()
    )


def exact_128_words(
    text,
):

    units = words(
        text
    )

    if len(units) < TARGET_WORDS:
        return None

    return " ".join(
        units[
            :TARGET_WORDS
        ]
    )


def quality_report(
    text,
):

    units = words(
        text
    )

    lower = [
        unit.lower()
        for unit
        in units
    ]

    counts = Counter(
        lower
    )

    return {
        "word_count":
            len(units),

        "unique_word_ratio":
            (
                len(counts)
                / max(
                    len(units),
                    1,
                )
            ),

        "max_word_fraction":
            (
                max(
                    counts.values(),
                    default=0,
                )
                / max(
                    len(units),
                    1,
                )
            ),
    }


# ================================================================
# PROMPT BUCKETING
#
# Equal prompt lengths allow multiple records to share
# one batched KV-cache decoding operation.
# ================================================================

def get_bucket_length(
    length,
):

    length = min(
        int(length),
        PROMPT_CAP,
    )

    if length < PROMPT_BUCKET:
        return length

    return max(
        PROMPT_BUCKET,
        (
            length
            // PROMPT_BUCKET
        )
        * PROMPT_BUCKET,
    )


PREPARED = []


for record in TEST_RECORDS:

    prompt_ids = (
        tokenizer.encode(
            record["prompt"],
            add_special_tokens=False,
        )
    )

    if not prompt_ids:

        prompt_ids = [
            int(
                tokenizer.eos_token_id
            )
        ]

    length = (
        get_bucket_length(
            len(prompt_ids)
        )
    )

    prompt_ids = (
        prompt_ids[
            -length:
        ]
    )

    PREPARED.append(
        {
            "id":
                record["id"],

            "prompt_ids":
                prompt_ids,

            "prompt_len":
                len(
                    prompt_ids
                ),
        }
    )


# ================================================================
# REUSE ALREADY-FINISHED OUTPUTS
#
# If the slow generator already completed some records,
# don't waste GPU time generating them again.
# ================================================================

PREDICTIONS = {}


for previous_path in sorted(
    DIRS[
        "predictions"
    ].glob(
        "official_predictions*.partial.json"
    )
):

    try:

        previous = json.loads(
            previous_path.read_text(
                encoding="utf-8"
            )
        )

        for identifier, value in (
            previous.items()
        ):

            if (
                identifier in seen_ids

                and isinstance(
                    value,
                    dict,
                )

                and set(
                    value
                ) == {"text"}

                and len(
                    words(
                        value["text"]
                    )
                ) == TARGET_WORDS
            ):

                PREDICTIONS[
                    identifier
                ] = value

    except Exception:
        pass


print(
    "Valid old predictions reused:",
    len(PREDICTIONS),
)


# ================================================================
# BUILD BATCHES
# ================================================================

remaining_records = [
    record
    for record
    in PREPARED
    if record["id"]
    not in PREDICTIONS
]


groups = defaultdict(
    list
)


for record in (
    remaining_records
):

    groups[
        record[
            "prompt_len"
        ]
    ].append(
        record
    )


batches = []


for prompt_length, items in (
    groups.items()
):

    for start in range(
        0,
        len(items),
        BATCH_PER_GPU,
    ):

        batches.append(
            (
                prompt_length,
                items[
                    start:
                    start
                    + BATCH_PER_GPU
                ],
            )
        )


# Largest batches first.
batches.sort(
    key=lambda item:
        len(
            item[1]
        ),
    reverse=True,
)


# ================================================================
# DISTRIBUTE WORK ACROSS BOTH T4s
# ================================================================

GPU_ASSIGNMENTS = [
    []
    for _ in range(
        GPU_COUNT
    )
]


GPU_LOADS = [
    0
    for _ in range(
        GPU_COUNT
    )
]


for batch in batches:

    gpu_index = min(
        range(
            GPU_COUNT
        ),
        key=lambda index:
            GPU_LOADS[
                index
            ],
    )

    GPU_ASSIGNMENTS[
        gpu_index
    ].append(
        batch
    )

    GPU_LOADS[
        gpu_index
    ] += len(
        batch[1]
    )


print(
    "New examples per GPU:",
    GPU_LOADS,
)


# ================================================================
# GENERATION POLICIES
# ================================================================

POLICIES = [

    {
        "name":
            "fast_primary",

        "seed":
            781,

        "temperature":
            0.85,

        "top_k":
            100,

        "top_p":
            0.92,

        "repetition_penalty":
            1.08,

        "no_repeat_ngram_size":
            4,

        "min_unique_word_ratio":
            0.55,

        "max_word_fraction":
            0.08,

        "max_new":
            PRIMARY_MAX_NEW,
    },

    {
        "name":
            "retry",

        "seed":
            1781,

        "temperature":
            0.95,

        "top_k":
            160,

        "top_p":
            0.96,

        "repetition_penalty":
            1.12,

        "no_repeat_ngram_size":
            3,

        "min_unique_word_ratio":
            0.50,

        "max_word_fraction":
            0.10,

        "max_new":
            RETRY_MAX_NEW,
    },
]


# ================================================================
# FAST SAMPLER
#
# Important optimization:
# Top-k is calculated FIRST.
# Nucleus sorting therefore works on only 100/160 values
# instead of all 25,000 vocabulary logits.
# ================================================================

def sample_next_token(
    logits,
    history,
    policy,
    generator,
    forbidden_tokens,
):

    scores = (
        logits
        .float()
        .clone()
    )


    # -------------------------
    # Special-token filtering
    # -------------------------

    if forbidden_tokens:

        indices = torch.tensor(
            list(
                forbidden_tokens
            ),
            dtype=torch.long,
            device=scores.device,
        )

        scores[
            indices
        ] = -torch.inf


    # -------------------------
    # Repetition penalty
    # -------------------------

    penalty = float(
        policy[
            "repetition_penalty"
        ]
    )


    if (
        history
        and penalty != 1.0
    ):

        unique_ids = torch.tensor(
            list(
                set(
                    history
                )
            ),
            dtype=torch.long,
            device=scores.device,
        )

        values = (
            scores.index_select(
                0,
                unique_ids,
            )
        )

        values = torch.where(
            values > 0,
            values / penalty,
            values * penalty,
        )

        scores.index_copy_(
            0,
            unique_ids,
            values,
        )


    # -------------------------
    # No-repeat ngram
    # -------------------------

    ngram_size = int(
        policy[
            "no_repeat_ngram_size"
        ]
    )


    if (
        ngram_size > 0
        and len(history)
        >= ngram_size - 1
    ):

        if ngram_size > 1:

            prefix = tuple(
                history[
                    -(ngram_size - 1):
                ]
            )

        else:

            prefix = tuple()


        banned = set()


        for index in range(
            len(history)
            - ngram_size
            + 1
        ):

            gram = tuple(
                history[
                    index:
                    index
                    + ngram_size
                ]
            )

            if (
                gram[:-1]
                == prefix
            ):

                banned.add(
                    gram[-1]
                )


        if banned:

            banned_tensor = torch.tensor(
                list(
                    banned
                ),
                dtype=torch.long,
                device=scores.device,
            )

            scores[
                banned_tensor
            ] = -torch.inf


    # -------------------------
    # FAST TOP-K FIRST
    # -------------------------

    k = min(
        int(
            policy[
                "top_k"
            ]
        ),
        scores.numel(),
    )


    top_values, top_indices = (
        torch.topk(
            scores,
            k,
            sorted=True,
        )
    )


    top_values = (
        top_values
        / float(
            policy[
                "temperature"
            ]
        )
    )


    probabilities = (
        torch.softmax(
            top_values,
            dim=-1,
        )
    )


    # -------------------------
    # Top-p inside top-k only
    # -------------------------

    top_p = float(
        policy[
            "top_p"
        ]
    )


    if top_p < 1.0:

        cumulative = (
            probabilities
            .cumsum(
                dim=0
            )
        )

        remove = (
            cumulative
            > top_p
        )

        if (
            remove.numel()
            > 1
        ):

            remove[1:] = (
                remove[:-1]
                .clone()
            )

        remove[0] = False


        probabilities = (
            probabilities
            .masked_fill(
                remove,
                0.0,
            )
        )


        probability_sum = (
            probabilities.sum()
        )


        if float(
            probability_sum
        ) <= 0:

            probabilities[0] = 1.0

        else:

            probabilities = (
                probabilities
                / probability_sum
            )


    chosen_position = (
        torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        )
    )


    return int(
        top_indices[
            chosen_position
        ].item()
    )


# ================================================================
# CACHE COMPACTION
#
# Once some sequences finish, remove them from the batch so the GPU
# stops wasting compute on completed sequences.
# ================================================================

def compact_caches(
    caches,
    keep_indices,
):

    return tuple(
        (
            key.index_select(
                0,
                keep_indices,
            ),

            value.index_select(
                0,
                keep_indices,
            ),
        )

        for key, value
        in caches
    )


# ================================================================
# BATCHED KV-CACHE GENERATION
# ================================================================

@torch.inference_mode()
def generate_batch(
    model,
    local_tokenizer,
    batch,
    device,
    policy,
):

    batch_size = len(
        batch
    )


    prompt_tensor = torch.tensor(
        [
            record[
                "prompt_ids"
            ]

            for record
            in batch
        ],
        dtype=torch.long,
        device=device,
    )


    # ONE prefix pass for the entire batch.
    caches = (
        model.prefill_prefix(
            prompt_tensor
        )
    )


    forbidden_tokens = {
        int(token)
        for token
        in (
            local_tokenizer.pad_token_id,
            local_tokenizer.bos_token_id,
            local_tokenizer.eos_token_id,
        )
        if token is not None
    }


    active = []


    for index, record in enumerate(
        batch
    ):

        generator = torch.Generator(
            device=device
        )


        # Stable seed per test ID.
        stable_id_seed = int(
            hashlib.sha256(
                record[
                    "id"
                ].encode(
                    "utf-8"
                )
            ).hexdigest()[
                :8
            ],
            16,
        )


        generator.manual_seed(
            int(
                policy[
                    "seed"
                ]
            )
            + stable_id_seed
        )


        active.append(
            {
                "id":
                    record["id"],

                "history":
                    [],

                "current":
                    int(
                        local_tokenizer
                        .bos_token_id
                    ),

                "generator":
                    generator,
            }
        )


    completed = {}

    failed = {}


    max_new_tokens = int(
        policy[
            "max_new"
        ]
    )


    for step in range(
        max_new_tokens
    ):


        current_tensor = torch.tensor(
            [
                [
                    item[
                        "current"
                    ]
                ]

                for item
                in active
            ],
            dtype=torch.long,
            device=device,
        )


        # ONE batched decoder call for all active samples.
        logits, caches = (
            model.decode_cached(
                current_tensor,
                caches,
            )
        )


        # Sample each row.
        for row_index, item in enumerate(
            active
        ):

            next_token = (
                sample_next_token(
                    logits[
                        row_index
                    ],
                    item[
                        "history"
                    ],
                    policy,
                    item[
                        "generator"
                    ],
                    forbidden_tokens,
                )
            )


            item[
                "history"
            ].append(
                next_token
            )


            item[
                "current"
            ] = next_token


        # --------------------------------------
        # DON'T decode strings every token.
        #
        # Check only periodically after at least
        # 128 generated tokens.
        # --------------------------------------

        should_check = (
            step + 1
            >= TARGET_WORDS

            and (
                (
                    step + 1
                )
                % WORD_CHECK_EVERY
                == 0

                or (
                    step + 1
                )
                == max_new_tokens
            )
        )


        if should_check:

            for item in list(
                active
            ):

                decoded = (
                    local_tokenizer.decode(
                        item[
                            "history"
                        ],
                        skip_special_tokens=True,
                    )
                )


                output = (
                    exact_128_words(
                        decoded
                    )
                )


                if output is None:
                    continue


                report = (
                    quality_report(
                        output
                    )
                )


                quality_passed = (
                    report[
                        "unique_word_ratio"
                    ]
                    >= float(
                        policy[
                            "min_unique_word_ratio"
                        ]
                    )

                    and

                    report[
                        "max_word_fraction"
                    ]
                    <= float(
                        policy[
                            "max_word_fraction"
                        ]
                    )
                )


                if quality_passed:

                    completed[
                        item["id"]
                    ] = {
                        "text":
                            output
                    }

                else:

                    failed[
                        item["id"]
                    ] = (
                        "quality threshold "
                        f"failed: {report}"
                    )


            # --------------------------------------
            # Remove completed/failed sequences
            # from the KV batch.
            # --------------------------------------

            keep_positions = [
                index

                for index, item
                in enumerate(
                    active
                )

                if (
                    item["id"]
                    not in completed

                    and

                    item["id"]
                    not in failed
                )
            ]


            if (
                len(
                    keep_positions
                )
                != len(
                    active
                )
            ):

                if not keep_positions:

                    active = []

                    break


                keep_tensor = torch.tensor(
                    keep_positions,
                    dtype=torch.long,
                    device=device,
                )


                caches = compact_caches(
                    caches,
                    keep_tensor,
                )


                active = [
                    active[index]

                    for index
                    in keep_positions
                ]


        if not active:
            break


    # ============================================================
    # FINAL CHECK FOR ANY STILL ACTIVE
    # ============================================================

    for item in active:

        decoded = (
            local_tokenizer.decode(
                item[
                    "history"
                ],
                skip_special_tokens=True,
            )
        )


        output = exact_128_words(
            decoded
        )


        if output is None:

            failed[
                item["id"]
            ] = (
                f"only "
                f"{len(words(decoded))} words "
                f"after "
                f"{len(item['history'])} tokens"
            )

            continue


        report = quality_report(
            output
        )


        if (
            report[
                "unique_word_ratio"
            ]
            >= float(
                policy[
                    "min_unique_word_ratio"
                ]
            )

            and

            report[
                "max_word_fraction"
            ]
            <= float(
                policy[
                    "max_word_fraction"
                ]
            )
        ):

            completed[
                item["id"]
            ] = {
                "text":
                    output
            }

        else:

            failed[
                item["id"]
            ] = (
                f"quality threshold "
                f"failed: {report}"
            )


    return (
        completed,
        failed,
    )


# ================================================================
# PER-GPU WORKER
# ================================================================

def gpu_worker(
    gpu_index,
    assigned_batches,
):

    device = (
        f"cuda:{gpu_index}"
    )


    model = MODELS[
        gpu_index
    ]


    # Separate tokenizer instance for the worker.
    local_tokenizer = (
        CourseBPETokenizer(
            Tokenizer.from_file(
                str(
                    tokenizer_json
                )
            )
        )
    )


    results = {}

    failures = {}


    total_examples = sum(
        len(
            batch[
                1
            ]
        )

        for batch
        in assigned_batches
    )


    processed_examples = 0


    print(
        f"[GPU {gpu_index}] "
        f"starting "
        f"{total_examples} examples"
    )


    for (
        prompt_length,
        batch,
    ) in assigned_batches:


        pending = list(
            batch
        )


        latest_errors = {}


        # Primary policy, followed by retry
        # only for failed examples.
        for policy in POLICIES:

            if not pending:
                break


            successes, errors = (
                generate_batch(
                    model,
                    local_tokenizer,
                    pending,
                    device,
                    policy,
                )
            )


            results.update(
                successes
            )


            latest_errors.update(
                errors
            )


            pending = [
                record

                for record
                in pending

                if record[
                    "id"
                ]
                not in successes
            ]


        # Anything still pending failed
        # both policies.
        for record in pending:

            identifier = (
                record[
                    "id"
                ]
            )

            failures[
                identifier
            ] = (
                latest_errors.get(
                    identifier,
                    "generation failed",
                )
            )


        processed_examples += len(
            batch
        )


        print(
            f"[GPU {gpu_index}] "
            f"{processed_examples}/"
            f"{total_examples} | "
            f"success={len(results)} | "
            f"failures={len(failures)}"
        )


    return (
        results,
        failures,
    )


# ================================================================
# RUN BOTH T4s IN PARALLEL
# ================================================================

generation_start = (
    time.time()
)


ALL_FAILURES = {}


if remaining_records:

    with ThreadPoolExecutor(
        max_workers=GPU_COUNT
    ) as executor:

        futures = [

            executor.submit(
                gpu_worker,
                gpu_index,
                GPU_ASSIGNMENTS[
                    gpu_index
                ],
            )

            for gpu_index
            in range(
                GPU_COUNT
            )
        ]


        for future in futures:

            generated, failures = (
                future.result()
            )

            PREDICTIONS.update(
                generated
            )

            ALL_FAILURES.update(
                failures
            )


generation_seconds = (
    time.time()
    - generation_start
)


# ================================================================
# SAVE PARTIAL OUTPUT
# ================================================================

fast_partial_path = (
    DIRS["predictions"]
    / "official_predictions.fast_batched.partial.json"
)


atomic_json(
    fast_partial_path,
    PREDICTIONS,
)


atomic_json(
    DIRS["predictions"]
    / "generation_failures.fast_batched.json",
    ALL_FAILURES,
)


if ALL_FAILURES:

    missing_ids = [

        record["id"]

        for record
        in TEST_RECORDS

        if record["id"]
        not in PREDICTIONS
    ]


    raise RuntimeError(
        f"{len(ALL_FAILURES)} "
        f"generations still failed.\n"
        f"Successful results were preserved at:\n"
        f"{fast_partial_path}\n\n"
        f"Failed IDs:\n"
        f"{missing_ids[:20]}"
    )


# ================================================================
# STRICT SUBMISSION VALIDATION
# ================================================================

official_ids = [
    record[
        "id"
    ]

    for record
    in TEST_RECORDS
]


expected_ids = set(
    official_ids
)


actual_ids = set(
    PREDICTIONS
)


validation_errors = []


missing_ids = (
    expected_ids
    - actual_ids
)


unknown_ids = (
    actual_ids
    - expected_ids
)


if missing_ids:

    validation_errors.append(
        "Missing IDs: "
        + str(
            sorted(
                missing_ids
            )[:20]
        )
    )


if unknown_ids:

    validation_errors.append(
        "Unknown IDs: "
        + str(
            sorted(
                unknown_ids
            )[:20]
        )
    )


for identifier in sorted(
    expected_ids
    & actual_ids
):

    value = (
        PREDICTIONS[
            identifier
        ]
    )


    if (
        not isinstance(
            value,
            dict,
        )

        or

        set(value)
        != {"text"}
    ):

        validation_errors.append(
            f"{identifier}: "
            "invalid entry"
        )

        continue


    word_count = len(
        words(
            value[
                "text"
            ]
        )
    )


    if word_count != TARGET_WORDS:

        validation_errors.append(
            f"{identifier}: "
            f"{word_count} words"
        )


if validation_errors:

    raise RuntimeError(
        "Prediction validation failed:\n"
        + "\n".join(
            validation_errors
        )
    )


print()
print(
    "Prediction validation: PASS"
)


# ================================================================
# PROVENANCE
# ================================================================

checkpoint_id = (
    "selected-"
    + sha256_file(
        selected_checkpoint
    )[:12]
)


provenance = {

    "checkpoint_id":
        checkpoint_id,

    "checkpoint_sha256":
        sha256_file(
            selected_checkpoint
        ),

    "model_source_sha256":
        MODEL_SOURCE_SHA256,

    "tokenizer_fingerprint":
        TOKENIZER_META[
            "fingerprint"
        ],

    "generation_backend":
        "dual_gpu_fp16_batched_kv_cache_v1",

    "gpu_count":
        GPU_COUNT,

    "batch_per_gpu":
        BATCH_PER_GPU,

    "prompt_cap":
        PROMPT_CAP,

    "prompt_bucket":
        PROMPT_BUCKET,

    "word_check_every":
        WORD_CHECK_EVERY,

    "policies":
        POLICIES,

    "prediction_count":
        len(
            PREDICTIONS
        ),

    "generation_seconds":
        generation_seconds,

    "created_utc":
        utc_now(),
}


atomic_json(
    DIRS["predictions"]
    / "inference_provenance.json",
    provenance,
)


# ================================================================
# CACHED INFERENCE SANITY CHECK
#
# If the original notebook already wrote a passing check,
# preserve it.
# Otherwise perform one small check.
# ================================================================

cached_validation_path = (
    DIRS["manifests"]
    / "cached_inference_validation.json"
)


existing_cached_pass = False


if cached_validation_path.exists():

    try:

        existing_cached_pass = (
            json.loads(
                cached_validation_path.read_text(
                    encoding="utf-8"
                )
            ).get(
                "status"
            )
            == "PASS"
        )

    except Exception:
        existing_cached_pass = False


if not existing_cached_pass:

    model = MODELS[0]

    device = "cuda:0"


    probe_ids = (
        PREPARED[0][
            "prompt_ids"
        ][-128:]
    )


    prompt_tensor = torch.tensor(
        [
            probe_ids
        ],
        dtype=torch.long,
        device=device,
    )


    with torch.inference_mode():

        caches = (
            model.prefill_prefix(
                prompt_tensor
            )
        )


        token_id = int(
            tokenizer.bos_token_id
        )


        cached_logits, _ = (
            model.decode_cached(
                torch.tensor(
                    [
                        [
                            token_id
                        ]
                    ],
                    dtype=torch.long,
                    device=device,
                ),
                caches,
            )
        )


        full_sequence = torch.tensor(
            [
                [
                    *probe_ids,
                    token_id,
                ]
            ],
            dtype=torch.long,
            device=device,
        )


        full_logits = model(
            full_sequence,

            attention_mask=(
                torch.ones_like(
                    full_sequence,
                    dtype=torch.bool,
                )
            ),

            prefix_lengths=(
                torch.tensor(
                    [
                        len(
                            probe_ids
                        )
                    ],
                    dtype=torch.long,
                    device=device,
                )
            ),
        )[
            0,
            -1,
        ]


        max_error = float(
            (
                cached_logits[
                    0
                ].float()
                - full_logits.float()
            )
            .abs()
            .max()
            .cpu()
        )


    # FP16 check gets a looser numerical tolerance.
    cached_status = (
        "PASS"
        if max_error < 0.05
        else "FAIL"
    )


    atomic_json(
        cached_validation_path,
        {
            "status":
                cached_status,

            "precision":
                "fp16",

            "max_absolute_error":
                max_error,

            "created_utc":
                utc_now(),
        },
    )


    if cached_status != "PASS":

        raise RuntimeError(
            "Cached/full inference "
            "sanity check failed. "
            f"Max error={max_error}"
        )


# ================================================================
# CREATE prediction.json
# ================================================================

submission_directory = (
    DIRS[
        "submission"
    ]
)


prediction_path = (
    submission_directory
    / "prediction.json"
)


atomic_json(
    prediction_path,
    PREDICTIONS,
)


# ================================================================
# CREATE submission.zip
# ================================================================

SUBMISSION_ZIP = (
    submission_directory
    / "submission.zip"
)


with tempfile.NamedTemporaryFile(
    dir=submission_directory,
    suffix=".zip",
    delete=False,
) as handle:

    temporary_zip = Path(
        handle.name
    )


try:

    with zipfile.ZipFile(
        temporary_zip,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:

        archive.write(
            prediction_path,
            "prediction.json",
        )


    with zipfile.ZipFile(
        temporary_zip,
        "r",
    ) as archive:

        if (
            archive.namelist()
            != [
                "prediction.json"
            ]
        ):

            raise RuntimeError(
                "Invalid submission "
                "ZIP structure."
            )


        bad_member = (
            archive.testzip()
        )


        if bad_member is not None:

            raise RuntimeError(
                "Submission ZIP "
                "CRC error: "
                + str(
                    bad_member
                )
            )


    os.replace(
        temporary_zip,
        SUBMISSION_ZIP,
    )


finally:

    temporary_zip.unlink(
        missing_ok=True
    )


submission_report = {

    "passed":
        True,

    "expected_ids":
        len(
            expected_ids
        ),

    "actual_ids":
        len(
            actual_ids
        ),

    "prediction_sha256":
        sha256_file(
            prediction_path
        ),

    "submission_zip_sha256":
        sha256_file(
            SUBMISSION_ZIP
        ),

    "inference_provenance":
        provenance,

    "quality_diagnostics":
        {
            identifier:
                quality_report(
                    value[
                        "text"
                    ]
                )

            for identifier, value
            in PREDICTIONS.items()
        },
}


atomic_json(
    submission_directory
    / "submission_validation.json",
    submission_report,
)


print(
    "Created submission:",
    SUBMISSION_ZIP,
)


# ================================================================
# CREATE FULL RECOVERY ZIP
# ================================================================

recovery_directory = (
    DIRS[
        "recovery"
    ]
)


RECOVERY_ZIP = (
    recovery_directory
    / "phase1_submission03_recovery.zip"
)


recovery_integrity_path = (
    DIRS[
        "manifests"
    ]
    / "recovery_integrity.json"
)


bounded_store = (
    DIRS["data"]
    / "bounded_document_store"
).resolve()


excluded_paths = {

    RECOVERY_ZIP.resolve(),

    recovery_integrity_path.resolve(),

    (
        recovery_directory
        / "training_recovery.latest.zip"
    ).resolve(),

    (
        recovery_directory
        / "training_recovery.latest.json"
    ).resolve(),

    (
        recovery_directory
        / "recovery_validation.json"
    ).resolve(),
}


files_to_archive = []


for path in sorted(
    ROOT.rglob("*")
):

    if (
        not path.is_file()
        or path.is_symlink()
    ):
        continue


    resolved = (
        path.resolve()
    )


    if resolved in excluded_paths:
        continue


    if (
        resolved
        == bounded_store

        or

        bounded_store
        in resolved.parents
    ):
        continue


    files_to_archive.append(
        path
    )


recovery_manifest = {

    "created_utc":
        utc_now(),

    "files":
        [

            {
                "path":
                    path.relative_to(
                        ROOT
                    ).as_posix(),

                "bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }

            for path
            in files_to_archive
        ],
}


atomic_json(
    recovery_integrity_path,
    recovery_manifest,
)


with tempfile.NamedTemporaryFile(
    dir=recovery_directory,
    suffix=".zip",
    delete=False,
) as handle:

    temporary_recovery = Path(
        handle.name
    )


try:

    with zipfile.ZipFile(
        temporary_recovery,
        "w",
        zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:


        for item in (
            recovery_manifest[
                "files"
            ]
        ):

            path = (
                ROOT
                / item[
                    "path"
                ]
            )


            if (
                item[
                    "path"
                ].endswith(
                    ".pt"
                )
            ):

                compression = (
                    zipfile.ZIP_STORED
                )

            else:

                compression = (
                    zipfile.ZIP_DEFLATED
                )


            archive.write(
                path,
                item[
                    "path"
                ],
                compress_type=(
                    compression
                ),
            )


        archive.write(
            recovery_integrity_path,
            "manifests/"
            "recovery_integrity.json",
            compress_type=(
                zipfile.ZIP_DEFLATED
            ),
        )


    with zipfile.ZipFile(
        temporary_recovery,
        "r",
    ) as archive:

        bad_member = (
            archive.testzip()
        )

        if bad_member is not None:

            raise RuntimeError(
                "Recovery ZIP CRC "
                "failure: "
                + str(
                    bad_member
                )
            )


    os.replace(
        temporary_recovery,
        RECOVERY_ZIP,
    )


finally:

    temporary_recovery.unlink(
        missing_ok=True
    )


atomic_json(
    recovery_directory
    / "recovery_validation.json",

    {
        "path":
            str(
                RECOVERY_ZIP
            ),

        "sha256":
            sha256_file(
                RECOVERY_ZIP
            ),

        "files":
            len(
                recovery_manifest[
                    "files"
                ]
            ),

        "created_utc":
            utc_now(),
    },
)


print(
    "Created recovery:",
    RECOVERY_ZIP,
)


# ================================================================
# FINAL DELIVERY CHECK
# ================================================================

training_result_path = (
    DIRS["manifests"]
    / "training_result.json"
)


training_completed = False


if training_result_path.exists():

    try:

        training_result = json.loads(
            training_result_path.read_text(
                encoding="utf-8"
            )
        )


        training_completed = (
            int(
                training_result.get(
                    "completed_step",
                    -1,
                )
            )
            == 10_000
        )

    except Exception:
        training_completed = False


final_checks = {

    "training_completed_10000_steps":
        training_completed,

    "selected_model_present":
        selected_checkpoint.exists(),

    "final_model_present":
        final_checkpoint.exists(),

    "prediction_count_matches_test":
        (
            len(
                PREDICTIONS
            )
            == len(
                TEST_RECORDS
            )
        ),

    "all_predictions_128_words":
        all(
            len(
                words(
                    value[
                        "text"
                    ]
                )
            )
            == TARGET_WORDS

            for value
            in PREDICTIONS.values()
        ),

    "submission_zip_present":
        (
            SUBMISSION_ZIP.exists()
            and
            SUBMISSION_ZIP.stat().st_size
            > 0
        ),

    "recovery_zip_present":
        (
            RECOVERY_ZIP.exists()
            and
            RECOVERY_ZIP.stat().st_size
            > 0
        ),
}


FINAL_STATUS = (
    "PASS"
    if all(
        final_checks.values()
    )
    else "FAIL"
)


atomic_json(
    DIRS["manifests"]
    / "post_restart_delivery_report.json",

    {
        "status":
            FINAL_STATUS,

        "checks":
            final_checks,

        "submission_zip":
            str(
                SUBMISSION_ZIP
            ),

        "recovery_zip":
            str(
                RECOVERY_ZIP
            ),

        "prediction_count":
            len(
                PREDICTIONS
            ),

        "generation_seconds":
            generation_seconds,

        "created_utc":
            utc_now(),
    },
)


# ================================================================
# FINISHED
# ================================================================

print()
print("=" * 72)

print(
    "FAST POST-RESTART PIPELINE:",
    FINAL_STATUS,
)

print(
    "Generation time:",
    round(
        generation_seconds,
        1,
    ),
    "seconds",
)

print(
    "Predictions:",
    len(
        PREDICTIONS
    ),
    "/",
    len(
        TEST_RECORDS
    ),
)

print()

print(
    "CODABENCH SUBMISSION:"
)

print(
    SUBMISSION_ZIP
)

print()

print(
    "COMPLETE RECOVERY ZIP:"
)

print(
    RECOVERY_ZIP
)

print("=" * 72)

In [ ]:
# =====================================================================
# CS781 SUBMISSION 03
# REPAIR FAILED GENERATIONS + COMPLETE ALL DELIVERABLES
#
# IMPORTANT:
# Run this DIRECTLY AFTER the previous fast-generation cell failed.
# Do NOT restart the kernel before running this cell.
#
# Existing objects reused:
#   MODELS
#   tokenizer
#   TEST_RECORDS
#   PREPARED
#   MODEL_CONFIG
#   TOKENIZER_META
#   MODEL_SOURCE_SHA256
#   selected_checkpoint
#   final_checkpoint
#   DIRS
#   ROOT
#   CourseBPETokenizer
#   Tokenizer
#   generate_batch
#   words
#   quality_report
#   sha256_file
#   atomic_json
# =====================================================================

from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import hashlib
import json
import os
import shutil
import tempfile
import time
import zipfile

import torch


# =====================================================================
# 0. BASIC CHECKS
# =====================================================================

print("=" * 76)
print("CS781 REPAIR + FINAL DELIVERY")
print("=" * 76)


REQUIRED_GLOBALS = [
    "MODELS",
    "tokenizer",
    "TEST_RECORDS",
    "PREPARED",
    "MODEL_CONFIG",
    "TOKENIZER_META",
    "MODEL_SOURCE_SHA256",
    "selected_checkpoint",
    "final_checkpoint",
    "DIRS",
    "ROOT",
    "CourseBPETokenizer",
    "Tokenizer",
    "generate_batch",
    "words",
    "quality_report",
    "sha256_file",
    "atomic_json",
]


missing_globals = [
    name
    for name in REQUIRED_GLOBALS
    if name not in globals()
]


if missing_globals:
    raise RuntimeError(
        "The previous fast-generation cell is no longer loaded.\n"
        "Missing objects:\n"
        + "\n".join(missing_globals)
        + "\n\nDo not use this cell after a kernel restart."
    )


GPU_COUNT = min(
    len(MODELS),
    torch.cuda.device_count(),
    2,
)


if GPU_COUNT < 1:
    raise RuntimeError(
        "No loaded CUDA model is available."
    )


print(
    "GPUs:",
    [
        torch.cuda.get_device_name(i)
        for i in range(GPU_COUNT)
    ],
)


print(
    "Official test records:",
    len(TEST_RECORDS),
)


# =====================================================================
# 1. LOAD EVERY VALID PREDICTION ALREADY PRODUCED
# =====================================================================

TARGET_WORDS = 128

VALID_IDS = {
    str(row["id"])
    for row in TEST_RECORDS
}


PREDICTIONS = {}


def absorb_prediction_file(path):
    global PREDICTIONS

    path = Path(path)

    if not path.exists():
        return 0

    try:
        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return 0

    if not isinstance(payload, dict):
        return 0

    added = 0

    for identifier, value in payload.items():

        identifier = str(identifier)

        if identifier not in VALID_IDS:
            continue

        if not isinstance(value, dict):
            continue

        if set(value.keys()) != {"text"}:
            continue

        text = value["text"]

        if not isinstance(text, str):
            continue

        if len(words(text)) != TARGET_WORDS:
            continue

        if identifier not in PREDICTIONS:
            PREDICTIONS[identifier] = {
                "text": text
            }

            added += 1

    return added


# Fast batch output from the cell that just failed.
absorb_prediction_file(
    DIRS["predictions"]
    / "official_predictions.fast_batched.partial.json"
)


# Also salvage anything produced by older inference attempts.
for path in sorted(
    DIRS["predictions"].glob(
        "official_predictions*.partial.json"
    )
):
    absorb_prediction_file(path)


# Also salvage an already-written prediction.json if one exists.
absorb_prediction_file(
    DIRS["submission"]
    / "prediction.json"
)


print(
    "Valid predictions recovered:",
    len(PREDICTIONS),
    "/",
    len(TEST_RECORDS),
)


# =====================================================================
# 2. PREPARED RECORD MAP
# =====================================================================

PREPARED_BY_ID = {
    str(record["id"]): record
    for record in PREPARED
}


missing_prepared = [
    identifier
    for identifier in VALID_IDS
    if identifier not in PREPARED_BY_ID
]


if missing_prepared:
    raise RuntimeError(
        "Some official IDs have no prepared prompt:\n"
        + str(
            missing_prepared[:20]
        )
    )


# =====================================================================
# 3. SAVE REPAIR PROGRESS AFTER EVERY BATCH
# =====================================================================

REPAIR_PARTIAL = (
    DIRS["predictions"]
    / "official_predictions.repaired.partial.json"
)


def save_progress():
    ordered = {}

    for row in TEST_RECORDS:

        identifier = str(
            row["id"]
        )

        if identifier in PREDICTIONS:
            ordered[identifier] = (
                PREDICTIONS[
                    identifier
                ]
            )

    atomic_json(
        REPAIR_PARTIAL,
        ordered,
    )


save_progress()


# =====================================================================
# 4. BUILD GPU BATCHES
# =====================================================================

REPAIR_BATCH_PER_GPU = 8


def build_batches(records):
    groups = defaultdict(list)

    for record in records:
        groups[
            int(record["prompt_len"])
        ].append(record)

    result = []

    for prompt_len in sorted(
        groups
    ):
        items = groups[
            prompt_len
        ]

        for start in range(
            0,
            len(items),
            REPAIR_BATCH_PER_GPU,
        ):
            result.append(
                (
                    prompt_len,
                    items[
                        start:
                        start
                        + REPAIR_BATCH_PER_GPU
                    ],
                )
            )

    # Largest batches first.
    result.sort(
        key=lambda item:
            len(item[1]),
        reverse=True,
    )

    return result


def distribute_batches(
    batches,
):
    assignments = [
        []
        for _ in range(
            GPU_COUNT
        )
    ]

    loads = [
        0
        for _ in range(
            GPU_COUNT
        )
    ]

    for batch in batches:

        gpu = min(
            range(
                GPU_COUNT
            ),
            key=lambda index:
                loads[index],
        )

        assignments[gpu].append(
            batch
        )

        loads[gpu] += len(
            batch[1]
        )

    return (
        assignments,
        loads,
    )


# =====================================================================
# 5. ONE REPAIR POLICY ACROSS BOTH GPUs
# =====================================================================

def run_policy_on_missing(
    policy,
    prepared_records,
):

    if not prepared_records:
        return {}, {}

    batches = build_batches(
        prepared_records
    )

    assignments, loads = (
        distribute_batches(
            batches
        )
    )

    print()
    print("-" * 76)

    print(
        "Policy:",
        policy["name"],
    )

    print(
        "Examples assigned:",
        loads,
    )

    print("-" * 76)


    def worker(
        gpu_index,
        gpu_batches,
    ):

        device = (
            f"cuda:{gpu_index}"
        )

        model = (
            MODELS[
                gpu_index
            ]
        )

        # Independent tokenizer object for each thread.
        local_tokenizer = (
            CourseBPETokenizer(
                Tokenizer.from_file(
                    str(
                        DIRS[
                            "tokenizer"
                        ]
                        / "tokenizer.json"
                    )
                )
            )
        )

        successes = {}
        failures = {}

        total = sum(
            len(batch)
            for _, batch
            in gpu_batches
        )

        done = 0

        for (
            prompt_length,
            batch,
        ) in gpu_batches:

            generated, failed = (
                generate_batch(
                    model,
                    local_tokenizer,
                    batch,
                    device,
                    policy,
                )
            )

            successes.update(
                generated
            )

            failures.update(
                failed
            )

            done += len(
                batch
            )

            print(
                f"[GPU {gpu_index}] "
                f"{done}/{total} | "
                f"new success={len(successes)} | "
                f"remaining={len(failures)}"
            )

        return (
            successes,
            failures,
        )


    all_successes = {}
    all_failures = {}


    with ThreadPoolExecutor(
        max_workers=GPU_COUNT
    ) as executor:

        futures = [
            executor.submit(
                worker,
                gpu_index,
                assignments[
                    gpu_index
                ],
            )
            for gpu_index
            in range(
                GPU_COUNT
            )
        ]


        for future in futures:

            successes, failures = (
                future.result()
            )

            all_successes.update(
                successes
            )

            all_failures.update(
                failures
            )


    return (
        all_successes,
        all_failures,
    )


# =====================================================================
# 6. RETRY POLICIES
#
# First retries still demand reasonable diversity.
# The LAST policy only enforces the actual submission requirement:
# a valid nonempty continuation of exactly 128 words.
# =====================================================================

REPAIR_POLICIES = [

    # ----------------------------------------------------------
    # Retry 1: slightly more diverse than original.
    # ----------------------------------------------------------
    {
        "name":
            "repair_diverse_1",

        "seed":
            2781,

        "temperature":
            0.92,

        "top_k":
            180,

        "top_p":
            0.96,

        "repetition_penalty":
            1.14,

        "no_repeat_ngram_size":
            4,

        "min_unique_word_ratio":
            0.48,

        "max_word_fraction":
            0.12,

        "max_new":
            511,
    },


    # ----------------------------------------------------------
    # Retry 2
    # ----------------------------------------------------------
    {
        "name":
            "repair_diverse_2",

        "seed":
            3781,

        "temperature":
            1.00,

        "top_k":
            220,

        "top_p":
            0.97,

        "repetition_penalty":
            1.18,

        "no_repeat_ngram_size":
            4,

        "min_unique_word_ratio":
            0.44,

        "max_word_fraction":
            0.14,

        "max_new":
            511,
    },


    # ----------------------------------------------------------
    # Retry 3
    # ----------------------------------------------------------
    {
        "name":
            "repair_diverse_3",

        "seed":
            4781,

        "temperature":
            1.05,

        "top_k":
            256,

        "top_p":
            0.98,

        "repetition_penalty":
            1.20,

        "no_repeat_ngram_size":
            3,

        "min_unique_word_ratio":
            0.40,

        "max_word_fraction":
            0.16,

        "max_new":
            511,
    },


    # ----------------------------------------------------------
    # Retry 4
    # ----------------------------------------------------------
    {
        "name":
            "repair_diverse_4",

        "seed":
            5781,

        "temperature":
            1.10,

        "top_k":
            320,

        "top_p":
            0.985,

        "repetition_penalty":
            1.22,

        "no_repeat_ngram_size":
            3,

        "min_unique_word_ratio":
            0.35,

        "max_word_fraction":
            0.20,

        "max_new":
            511,
    },


    # ----------------------------------------------------------
    # FINAL FALLBACK
    #
    # Submission validation in the notebook requires:
    #   - every official ID
    #   - {"text": ...}
    #   - exactly 128 words
    #
    # This final pass therefore accepts any exact 128-word
    # generation rather than throwing away otherwise valid output.
    # ----------------------------------------------------------
    {
        "name":
            "final_exact_128_fallback",

        "seed":
            9781,

        "temperature":
            1.10,

        "top_k":
            384,

        "top_p":
            0.99,

        "repetition_penalty":
            1.25,

        "no_repeat_ngram_size":
            3,

        "min_unique_word_ratio":
            0.0,

        "max_word_fraction":
            1.0,

        "max_new":
            511,
    },
]


# =====================================================================
# 7. RUN RETRIES ONLY ON MISSING IDs
# =====================================================================

repair_start = time.time()


for policy in REPAIR_POLICIES:

    missing_ids = [
        str(row["id"])
        for row in TEST_RECORDS
        if str(row["id"])
        not in PREDICTIONS
    ]


    print()
    print(
        "Missing before",
        policy["name"],
        ":",
        len(missing_ids),
    )


    if not missing_ids:
        break


    pending_records = [
        PREPARED_BY_ID[
            identifier
        ]
        for identifier
        in missing_ids
    ]


    successes, failures = (
        run_policy_on_missing(
            policy,
            pending_records,
        )
    )


    # Validate successes before accepting.
    accepted = 0

    for identifier, value in (
        successes.items()
    ):

        identifier = str(
            identifier
        )

        if identifier not in VALID_IDS:
            continue

        if (
            isinstance(
                value,
                dict,
            )
            and
            set(
                value.keys()
            )
            == {"text"}
            and
            isinstance(
                value["text"],
                str,
            )
            and
            len(
                words(
                    value["text"]
                )
            )
            == TARGET_WORDS
        ):

            PREDICTIONS[
                identifier
            ] = {
                "text":
                    value[
                        "text"
                    ]
            }

            accepted += 1


    save_progress()


    remaining = (
        len(TEST_RECORDS)
        - len(PREDICTIONS)
    )


    print(
        "Accepted this policy:",
        accepted,
    )

    print(
        "Total valid:",
        len(PREDICTIONS),
        "/",
        len(TEST_RECORDS),
    )

    print(
        "Still missing:",
        remaining,
    )


    if remaining == 0:
        break


# =====================================================================
# 8. EXTRA LAST-RESORT PASSES IF NECESSARY
#
# Different random seeds only.
# Exact 128 words required.
# =====================================================================

EXTRA_SEEDS = [
    11781,
    13781,
    15781,
    17781,
]


for extra_seed in EXTRA_SEEDS:

    missing_ids = [
        str(row["id"])
        for row in TEST_RECORDS
        if str(row["id"])
        not in PREDICTIONS
    ]


    if not missing_ids:
        break


    print()
    print(
        "Emergency exact-128 retry, seed",
        extra_seed,
        "| missing:",
        len(missing_ids),
    )


    emergency_policy = {

        "name":
            f"emergency_{extra_seed}",

        "seed":
            extra_seed,

        "temperature":
            1.15,

        "top_k":
            512,

        "top_p":
            0.995,

        "repetition_penalty":
            1.28,

        "no_repeat_ngram_size":
            3,

        "min_unique_word_ratio":
            0.0,

        "max_word_fraction":
            1.0,

        "max_new":
            511,
    }


    pending_records = [
        PREPARED_BY_ID[
            identifier
        ]
        for identifier
        in missing_ids
    ]


    successes, failures = (
        run_policy_on_missing(
            emergency_policy,
            pending_records,
        )
    )


    for identifier, value in (
        successes.items()
    ):

        identifier = str(
            identifier
        )

        if (
            identifier in VALID_IDS
            and
            isinstance(
                value,
                dict,
            )
            and
            set(
                value.keys()
            )
            == {"text"}
            and
            len(
                words(
                    value["text"]
                )
            )
            == TARGET_WORDS
        ):

            PREDICTIONS[
                identifier
            ] = {
                "text":
                    value[
                        "text"
                    ]
            }


    save_progress()


    print(
        "Now valid:",
        len(PREDICTIONS),
        "/",
        len(TEST_RECORDS),
    )


# =====================================================================
# 9. IF ANY STILL FAIL:
#    SHORTEN THEIR PROMPTS TO PROVIDE MORE GENERATION CONTEXT
#
# max context = 1280.
# Prompt <= 512 leaves >700 token positions for continuation.
# =====================================================================

missing_ids = [
    str(row["id"])
    for row in TEST_RECORDS
    if str(row["id"])
    not in PREDICTIONS
]


if missing_ids:

    print()
    print("=" * 76)

    print(
        "Final long-generation rescue for",
        len(missing_ids),
        "examples"
    )

    print("=" * 76)


    prompt_by_id = {
        str(row["id"]):
            row["prompt"]
        for row in TEST_RECORDS
    }


    rescue_records = []


    for identifier in missing_ids:

        ids = tokenizer.encode(
            prompt_by_id[
                identifier
            ],
            add_special_tokens=False,
        )


        # Keep only last 512 prompt tokens.
        ids = ids[
            -512:
        ]


        # Bucket lengths downward to 64.
        if len(ids) >= 64:

            length = max(
                64,
                (
                    len(ids)
                    // 64
                )
                * 64,
            )

        else:
            length = len(ids)


        ids = ids[
            -length:
        ]


        rescue_records.append(
            {
                "id":
                    identifier,

                "prompt_ids":
                    ids,

                "prompt_len":
                    len(ids),
            }
        )


    rescue_policies = [

        {
            "name":
                "short_prompt_rescue_1",

            "seed":
                21781,

            "temperature":
                1.10,

            "top_k":
                384,

            "top_p":
                0.99,

            "repetition_penalty":
                1.25,

            "no_repeat_ngram_size":
                3,

            "min_unique_word_ratio":
                0.0,

            "max_word_fraction":
                1.0,

            # 512 prompt + BOS + 700 <= 1280
            "max_new":
                700,
        },

        {
            "name":
                "short_prompt_rescue_2",

            "seed":
                31781,

            "temperature":
                1.18,

            "top_k":
                512,

            "top_p":
                0.995,

            "repetition_penalty":
                1.30,

            "no_repeat_ngram_size":
                3,

            "min_unique_word_ratio":
                0.0,

            "max_word_fraction":
                1.0,

            "max_new":
                700,
        },
    ]


    current_rescue = (
        rescue_records
    )


    for policy in rescue_policies:

        if not current_rescue:
            break


        successes, failures = (
            run_policy_on_missing(
                policy,
                current_rescue,
            )
        )


        for identifier, value in (
            successes.items()
        ):

            identifier = str(
                identifier
            )

            if (
                identifier in VALID_IDS
                and
                isinstance(
                    value,
                    dict,
                )
                and
                set(
                    value.keys()
                )
                == {"text"}
                and
                len(
                    words(
                        value["text"]
                    )
                )
                == TARGET_WORDS
            ):

                PREDICTIONS[
                    identifier
                ] = {
                    "text":
                        value[
                            "text"
                        ]
                }


        save_progress()


        current_rescue = [
            record
            for record
            in current_rescue
            if record[
                "id"
            ]
            not in PREDICTIONS
        ]


        print(
            "After rescue:",
            len(PREDICTIONS),
            "/",
            len(TEST_RECORDS),
        )


repair_seconds = (
    time.time()
    - repair_start
)


# =====================================================================
# 10. STRICTLY VERIFY ALL 1000 OUTPUTS
# =====================================================================

official_ids = [
    str(row["id"])
    for row in TEST_RECORDS
]


expected = set(
    official_ids
)


actual = set(
    PREDICTIONS.keys()
)


errors = []


missing = sorted(
    expected - actual
)


unknown = sorted(
    actual - expected
)


if missing:
    errors.append(
        "Missing IDs: "
        + str(
            missing[:50]
        )
    )


if unknown:
    errors.append(
        "Unknown IDs: "
        + str(
            unknown[:50]
        )
    )


for identifier in official_ids:

    if identifier not in PREDICTIONS:
        continue


    value = PREDICTIONS[
        identifier
    ]


    if not isinstance(
        value,
        dict,
    ):

        errors.append(
            f"{identifier}: "
            "prediction is not an object"
        )

        continue


    if set(
        value.keys()
    ) != {"text"}:

        errors.append(
            f"{identifier}: "
            "entry must contain only 'text'"
        )

        continue


    text = value[
        "text"
    ]


    if not isinstance(
        text,
        str,
    ):

        errors.append(
            f"{identifier}: "
            "text is not a string"
        )

        continue


    count = len(
        words(text)
    )


    if count != TARGET_WORDS:

        errors.append(
            f"{identifier}: "
            f"{count} words instead of "
            f"{TARGET_WORDS}"
        )


if errors:

    save_progress()

    raise RuntimeError(
        "FINAL PREDICTION VALIDATION FAILED.\n"
        + "\n".join(
            errors[:100]
        )
        + "\n\nProgress is safely stored at:\n"
        + str(
            REPAIR_PARTIAL
        )
    )


print()
print("=" * 76)

print(
    "PREDICTION VALIDATION: PASS"
)

print(
    "Valid predictions:",
    len(PREDICTIONS),
    "/",
    len(TEST_RECORDS),
)

print("=" * 76)


# =====================================================================
# 11. PUT PREDICTIONS IN OFFICIAL TEST ORDER
# =====================================================================

PREDICTIONS = {
    identifier:
        PREDICTIONS[
            identifier
        ]
    for identifier
    in official_ids
}


# =====================================================================
# 12. INFERENCE PROVENANCE
# =====================================================================

checkpoint_id = (
    "selected-"
    + sha256_file(
        selected_checkpoint
    )[:12]
)


inference_provenance = {

    "checkpoint_id":
        checkpoint_id,

    "checkpoint_sha256":
        sha256_file(
            selected_checkpoint
        ),

    "model_source_sha256":
        MODEL_SOURCE_SHA256,

    "tokenizer_fingerprint":
        TOKENIZER_META[
            "fingerprint"
        ],

    "test_count":
        len(
            TEST_RECORDS
        ),

    "prediction_count":
        len(
            PREDICTIONS
        ),

    "generation_backend":
        "dual_t4_batched_kv_cache_with_repair_v2",

    "reused_valid_predictions":
        (
            len(TEST_RECORDS)
            - len(
                [
                    row
                    for row in TEST_RECORDS
                    if str(row["id"])
                    not in PREDICTIONS
                ]
            )
        ),

    "repair_policies":
        [
            policy["name"]
            for policy
            in REPAIR_POLICIES
        ],

    "repair_seconds":
        repair_seconds,

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_json(
    DIRS["predictions"]
    / "inference_provenance.json",
    inference_provenance,
)


# =====================================================================
# 13. CACHED DECODING VALIDATION
# =====================================================================

cached_validation_path = (
    DIRS["manifests"]
    / "cached_inference_validation.json"
)


cached_already_passed = False


if cached_validation_path.exists():

    try:
        existing = json.loads(
            cached_validation_path.read_text(
                encoding="utf-8"
            )
        )

        cached_already_passed = (
            existing.get(
                "status"
            )
            == "PASS"
        )

    except Exception:
        cached_already_passed = False


if not cached_already_passed:

    model = MODELS[0]

    device = "cuda:0"


    probe_ids = tokenizer.encode(
        TEST_RECORDS[0][
            "prompt"
        ],
        add_special_tokens=False,
    )[-128:]


    if not probe_ids:
        raise RuntimeError(
            "Cached-decoding probe "
            "has no prompt tokens."
        )


    with torch.inference_mode():

        prefix = torch.tensor(
            [
                probe_ids
            ],
            dtype=torch.long,
            device=device,
        )


        caches = (
            model.prefill_prefix(
                prefix
            )
        )


        bos_id = int(
            tokenizer.bos_token_id
        )


        cached_logits, _ = (
            model.decode_cached(
                torch.tensor(
                    [
                        [
                            bos_id
                        ]
                    ],
                    dtype=torch.long,
                    device=device,
                ),
                caches,
            )
        )


        full_ids = torch.tensor(
            [
                [
                    *probe_ids,
                    bos_id,
                ]
            ],
            dtype=torch.long,
            device=device,
        )


        full_logits = model(
            full_ids,

            attention_mask=(
                torch.ones_like(
                    full_ids,
                    dtype=torch.bool,
                )
            ),

            prefix_lengths=(
                torch.tensor(
                    [
                        len(
                            probe_ids
                        )
                    ],
                    dtype=torch.long,
                    device=device,
                )
            ),
        )[
            0,
            -1,
        ]


        max_error = float(
            (
                cached_logits[
                    0
                ].float()
                -
                full_logits.float()
            )
            .abs()
            .max()
            .cpu()
        )


    # FP16 numerical tolerance.
    cached_status = (
        "PASS"
        if max_error <= 0.05
        else "FAIL"
    )


    atomic_json(
        cached_validation_path,

        {
            "status":
                cached_status,

            "precision":
                "fp16",

            "prompt_tokens":
                len(
                    probe_ids
                ),

            "max_absolute_error":
                max_error,

            "created_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        },
    )


    if cached_status != "PASS":

        raise RuntimeError(
            "Cached/full decoding "
            "validation failed. "
            f"max_error={max_error}"
        )


print(
    "Cached decoding validation: PASS"
)


# =====================================================================
# 14. WRITE OFFICIAL prediction.json
# =====================================================================

submission_dir = (
    DIRS[
        "submission"
    ]
)


submission_dir.mkdir(
    parents=True,
    exist_ok=True,
)


prediction_path = (
    submission_dir
    / "prediction.json"
)


atomic_json(
    prediction_path,
    PREDICTIONS,
)


print(
    "prediction.json written:"
)

print(
    prediction_path
)


# =====================================================================
# 15. BUILD submission.zip
# =====================================================================

SUBMISSION_ZIP = (
    submission_dir
    / "submission.zip"
)


with tempfile.NamedTemporaryFile(
    dir=submission_dir,
    suffix=".zip",
    delete=False,
) as handle:

    temporary_zip = Path(
        handle.name
    )


try:

    with zipfile.ZipFile(
        temporary_zip,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:

        archive.write(
            prediction_path,
            "prediction.json",
        )


    # Verify ZIP before replacing final file.
    with zipfile.ZipFile(
        temporary_zip,
        "r",
    ) as archive:

        members = (
            archive.namelist()
        )


        if members != [
            "prediction.json"
        ]:

            raise RuntimeError(
                "Invalid submission ZIP "
                f"members: {members}"
            )


        bad = archive.testzip()


        if bad is not None:

            raise RuntimeError(
                "Submission ZIP CRC "
                f"failure: {bad}"
            )


        zipped_predictions = (
            json.loads(
                archive.read(
                    "prediction.json"
                )
            )
        )


        if set(
            zipped_predictions.keys()
        ) != expected:

            raise RuntimeError(
                "ZIP prediction IDs "
                "do not match official IDs."
            )


        for identifier, value in (
            zipped_predictions.items()
        ):

            if (
                not isinstance(
                    value,
                    dict,
                )
                or
                set(
                    value.keys()
                )
                != {"text"}
                or
                len(
                    words(
                        value["text"]
                    )
                )
                != TARGET_WORDS
            ):

                raise RuntimeError(
                    f"ZIP contains invalid "
                    f"prediction: {identifier}"
                )


    os.replace(
        temporary_zip,
        SUBMISSION_ZIP,
    )


finally:

    temporary_zip.unlink(
        missing_ok=True
    )


# =====================================================================
# 16. SUBMISSION VALIDATION REPORT
# =====================================================================

quality_diagnostics = {
    identifier:
        quality_report(
            value[
                "text"
            ]
        )
    for identifier, value
    in PREDICTIONS.items()
}


submission_report = {

    "passed":
        True,

    "errors":
        [],

    "expected_ids":
        len(
            expected
        ),

    "actual_ids":
        len(
            PREDICTIONS
        ),

    "prediction_sha256":
        sha256_file(
            prediction_path
        ),

    "submission_zip_sha256":
        sha256_file(
            SUBMISSION_ZIP
        ),

    "inference_provenance":
        inference_provenance,

    "quality_diagnostics":
        quality_diagnostics,

    "zip":
        {
            "members":
                [
                    "prediction.json"
                ],

            "sha256":
                sha256_file(
                    SUBMISSION_ZIP
                ),

            "bytes":
                SUBMISSION_ZIP.stat().st_size,
        },
}


atomic_json(
    submission_dir
    / "submission_validation.json",
    submission_report,
)


print()
print(
    "SUBMISSION ZIP VALIDATED:"
)

print(
    SUBMISSION_ZIP
)


# =====================================================================
# 17. BUILD COMPLETE RECOVERY ZIP
# =====================================================================

recovery_dir = (
    DIRS[
        "recovery"
    ]
)


recovery_dir.mkdir(
    parents=True,
    exist_ok=True,
)


RECOVERY_ZIP = (
    recovery_dir
    / "phase1_submission03_recovery.zip"
)


PREVIOUS_RECOVERY = (
    recovery_dir
    / "phase1_submission03_recovery.previous.zip"
)


recovery_manifest_path = (
    DIRS[
        "manifests"
    ]
    / "recovery_integrity.json"
)


bounded_store = (
    DIRS["data"]
    / "bounded_document_store"
).resolve()


excluded_exact = {

    RECOVERY_ZIP.resolve(),

    PREVIOUS_RECOVERY.resolve(),

    recovery_manifest_path.resolve(),

    (
        recovery_dir
        / "recovery_validation.json"
    ).resolve(),

    (
        recovery_dir
        / "training_recovery.latest.zip"
    ).resolve(),

    (
        recovery_dir
        / "training_recovery.latest.json"
    ).resolve(),
}


files_to_archive = []


for path in sorted(
    ROOT.rglob("*")
):

    if not path.is_file():
        continue


    if path.is_symlink():
        continue


    resolved = (
        path.resolve()
    )


    if resolved in excluded_exact:
        continue


    if (
        resolved == bounded_store
        or
        bounded_store
        in resolved.parents
    ):
        continue


    files_to_archive.append(
        path
    )


recovery_manifest = {

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "files":
        [
            {
                "path":
                    path.relative_to(
                        ROOT
                    ).as_posix(),

                "bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
            for path
            in files_to_archive
        ],
}


atomic_json(
    recovery_manifest_path,
    recovery_manifest,
)


with tempfile.NamedTemporaryFile(
    dir=recovery_dir,
    suffix=".zip",
    delete=False,
) as handle:

    temporary_recovery = Path(
        handle.name
    )


try:

    with zipfile.ZipFile(
        temporary_recovery,
        "w",
        zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:


        for item in (
            recovery_manifest[
                "files"
            ]
        ):

            path = (
                ROOT
                / item[
                    "path"
                ]
            )


            compression = (
                zipfile.ZIP_STORED

                if item[
                    "path"
                ].endswith(
                    ".pt"
                )

                else

                zipfile.ZIP_DEFLATED
            )


            archive.write(
                path,
                item[
                    "path"
                ],
                compress_type=(
                    compression
                ),
            )


        archive.write(
            recovery_manifest_path,
            "manifests/"
            "recovery_integrity.json",
            compress_type=(
                zipfile.ZIP_DEFLATED
            ),
        )


    # -------------------------------------------------------------
    # Verify the temporary recovery ZIP thoroughly.
    # -------------------------------------------------------------

    with zipfile.ZipFile(
        temporary_recovery,
        "r",
    ) as archive:

        bad = archive.testzip()


        if bad is not None:

            raise RuntimeError(
                "Recovery ZIP CRC "
                f"failure: {bad}"
            )


        zipped_manifest = (
            json.loads(
                archive.read(
                    "manifests/"
                    "recovery_integrity.json"
                )
            )
        )


        expected_members = {
            item["path"]
            for item
            in zipped_manifest[
                "files"
            ]
        }


        actual_members = {
            name
            for name
            in archive.namelist()
            if name
            != (
                "manifests/"
                "recovery_integrity.json"
            )
        }


        if (
            expected_members
            != actual_members
        ):

            raise RuntimeError(
                "Recovery ZIP member "
                "list does not match "
                "integrity manifest."
            )


        # Verify every archived member hash.
        for item in (
            zipped_manifest[
                "files"
            ]
        ):

            payload = (
                archive.read(
                    item[
                        "path"
                    ]
                )
            )


            digest = (
                hashlib.sha256(
                    payload
                ).hexdigest()
            )


            if (
                digest
                != item[
                    "sha256"
                ]
            ):

                raise RuntimeError(
                    "Recovery ZIP hash "
                    "mismatch:\n"
                    + item[
                        "path"
                    ]
                )


    # Keep previous final recovery, if one exists.
    if RECOVERY_ZIP.exists():

        shutil.copy2(
            RECOVERY_ZIP,
            PREVIOUS_RECOVERY,
        )


    os.replace(
        temporary_recovery,
        RECOVERY_ZIP,
    )


finally:

    temporary_recovery.unlink(
        missing_ok=True
    )


recovery_report = {

    "passed":
        True,

    "path":
        str(
            RECOVERY_ZIP
        ),

    "sha256":
        sha256_file(
            RECOVERY_ZIP
        ),

    "bytes":
        RECOVERY_ZIP.stat().st_size,

    "file_count":
        len(
            recovery_manifest[
                "files"
            ]
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_json(
    recovery_dir
    / "recovery_validation.json",
    recovery_report,
)


print()
print(
    "RECOVERY ZIP VALIDATED:"
)

print(
    RECOVERY_ZIP
)


# =====================================================================
# 18. FINAL TRAINING + ARTIFACT CHECK
# =====================================================================

training_result_path = (
    DIRS["manifests"]
    / "training_result.json"
)


training_config_path = (
    DIRS["manifests"]
    / "training_config.json"
)


training_complete = False


if (
    training_result_path.exists()
    and
    training_config_path.exists()
):

    training_result = json.loads(
        training_result_path.read_text(
            encoding="utf-8"
        )
    )

    training_config = json.loads(
        training_config_path.read_text(
            encoding="utf-8"
        )
    )


    training_complete = (
        int(
            training_result.get(
                "completed_step",
                -1,
            )
        )
        == 10_000

        and

        int(
            training_config.get(
                "max_steps",
                -1,
            )
        )
        == 10_000
    )


final_checks = {

    "training_completed_10000_steps":
        training_complete,

    "selected_model_present":
        Path(
            selected_checkpoint
        ).exists(),

    "final_model_present":
        Path(
            final_checkpoint
        ).exists(),

    "cached_decoding_validated":
        (
            cached_validation_path.exists()
            and
            json.loads(
                cached_validation_path.read_text(
                    encoding="utf-8"
                )
            ).get(
                "status"
            )
            == "PASS"
        ),

    "all_official_ids_present":
        set(
            PREDICTIONS.keys()
        )
        == expected,

    "all_outputs_exactly_128_words":
        all(
            len(
                words(
                    value[
                        "text"
                    ]
                )
            )
            == TARGET_WORDS

            for value
            in PREDICTIONS.values()
        ),

    "prediction_json_present":
        prediction_path.exists(),

    "submission_zip_present":
        (
            SUBMISSION_ZIP.exists()
            and
            SUBMISSION_ZIP.stat().st_size
            > 0
        ),

    "recovery_zip_present":
        (
            RECOVERY_ZIP.exists()
            and
            RECOVERY_ZIP.stat().st_size
            > 0
        ),
}


FINAL_STATUS = (
    "PASS"
    if all(
        final_checks.values()
    )
    else "FAIL"
)


final_report = {

    "artifact_status":
        FINAL_STATUS,

    "checks":
        final_checks,

    "prediction_count":
        len(
            PREDICTIONS
        ),

    "expected_prediction_count":
        len(
            TEST_RECORDS
        ),

    "repair_seconds":
        repair_seconds,

    "submission_zip":
        str(
            SUBMISSION_ZIP
        ),

    "submission_zip_sha256":
        sha256_file(
            SUBMISSION_ZIP
        ),

    "recovery_zip":
        str(
            RECOVERY_ZIP
        ),

    "recovery_zip_sha256":
        sha256_file(
            RECOVERY_ZIP
        ),

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


atomic_json(
    DIRS["manifests"]
    / "final_verification.json",
    final_report,
)


# =====================================================================
# 19. FINISHED
# =====================================================================

print()
print("=" * 76)

print(
    "FINAL ARTIFACT STATUS:",
    FINAL_STATUS,
)

print("=" * 76)

print(
    "Predictions:",
    len(PREDICTIONS),
    "/",
    len(TEST_RECORDS),
)

print(
    "Repair generation time:",
    round(
        repair_seconds,
        1,
    ),
    "seconds",
)

print()

print(
    "CODABENCH FILE:"
)

print(
    SUBMISSION_ZIP
)

print()

print(
    "COMPLETE RECOVERY FILE:"
)

print(
    RECOVERY_ZIP
)

print()

print(
    "Final checks:"
)

for key, value in (
    final_checks.items()
):
    print(
        " ",
        key,
        "=",
        value,
    )

print("=" * 76)


if FINAL_STATUS != "PASS":

    failed_checks = [
        key
        for key, value
        in final_checks.items()
        if not value
    ]

    raise RuntimeError(
        "Final artifact verification "
        "did not fully pass:\n- "
        + "\n- ".join(
            failed_checks
        )
    )


print()
print(
    "SUCCESS: all required local "
    "artifacts were generated and verified."
)

## 13. Codabench validation and packaging

In [ ]:
def validate_predictions(predictions: Any, official_ids: Sequence[str], target_words: int = 128) -> dict[str, Any]:
    errors = []
    if not isinstance(predictions, dict):
        return {"passed": False, "errors": ["prediction root must be an object"]}
    expected, actual = set(map(str, official_ids)), set(map(str, predictions))
    if missing := sorted(expected - actual):
        errors.append(f"missing IDs: {missing[:20]}")
    if unknown := sorted(actual - expected):
        errors.append(f"unknown IDs: {unknown[:20]}")
    for identifier in sorted(expected & actual):
        entry = predictions[identifier]
        if not isinstance(entry, dict) or set(entry) != {"text"}:
            errors.append(f"{identifier}: entry must contain only a text field")
            continue
        text = entry["text"]
        if not isinstance(text, str) or not text.strip():
            errors.append(f"{identifier}: empty/non-string text")
        elif len(words(text)) != target_words:
            errors.append(f"{identifier}: {len(words(text))} words, expected {target_words}")
    return {"passed": not errors, "errors": errors, "expected_ids": len(expected), "actual_ids": len(actual)}

def create_submission_zip(predictions: Mapping[str, Any], official_ids: Sequence[str], output_dir: Path) -> tuple[Path, dict[str, Any]]:
    report = validate_predictions(predictions, official_ids, CFG.decoding["target_words"])
    if not report["passed"]:
        raise ValueError("Submission validation failed:\n" + "\n".join(report["errors"]))
    provenance_path=DIRS['predictions']/'inference_provenance.json'
    if not provenance_path.exists(): raise RuntimeError('Missing inference provenance')
    provenance=json.loads(provenance_path.read_text(encoding='utf-8'))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / "prediction.json"
    atomic_write_json(prediction_path, predictions)
    zip_path = output_dir / "submission.zip"
    with tempfile.NamedTemporaryFile(dir=output_dir, suffix=".zip", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        zip_report = validate_zip(temporary)
        if zip_report["members"] != ["prediction.json"]:
            raise ValueError(f"Invalid ZIP root structure: {zip_report['members']}")
        os.replace(temporary, zip_path)
    finally:
        temporary.unlink(missing_ok=True)
    report['prediction_sha256']=sha256_file(prediction_path)
    report['quality_diagnostics']={key:generation_quality_report(value['text']) for key,value in predictions.items()}
    report['inference_provenance']=provenance
    report["zip"] = validate_zip(zip_path)
    atomic_write_json(output_dir / "submission_validation.json", report)
    return zip_path, report

if 'PREDICTIONS' in globals() and 'TEST_RECORDS' in globals():
    SUBMISSION_ZIP,SUBMISSION_REPORT=create_submission_zip(PREDICTIONS,[row['id'] for row in TEST_RECORDS],DIRS['submission']);print('Created validated Codabench package:',SUBMISSION_ZIP)
else: SUBMISSION_ZIP=SUBMISSION_REPORT=None;print('Submission packaging deferred until authenticated official predictions exist.')


## 14. Recovery archive and integrity manifest

In [ ]:
def build_integrity_manifest(root:Path,excluded:set[Path]|None=None)->dict[str,Any]:
    root=Path(root).resolve();excluded={Path(p).resolve() for p in (excluded or set())}
    files=[]
    for p in sorted(item for item in root.rglob('*') if item.is_file()):
        resolved=p.resolve()
        if resolved in excluded or any(item.is_dir() and (item==resolved or item in resolved.parents) for item in excluded) or p.is_symlink():continue
        files.append({'path':p.relative_to(root).as_posix(),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    return {'created_utc':utc_now(),'files':files}
def create_recovery_zip(root:Path)->tuple[Path,dict[str,Any]]:
    root=Path(root).resolve();outdir=root/'recovery';outdir.mkdir(parents=True,exist_ok=True)
    out=outdir/'phase1_submission03_recovery.zip';previous=outdir/'phase1_submission03_recovery.previous.zip'
    manifest_path=root/'manifests'/'recovery_integrity.json'
    # Exclude previous archives, validation reports, and manifest from its own hash list.
    excluded={out,previous,manifest_path,outdir/'recovery_validation.json',
              outdir/'training_recovery.latest.zip',outdir/'training_recovery.latest.json',
              root/'data'/'bounded_document_store'}
    notebook_backup=Path(CFG.completed_notebook_path).expanduser() if CFG.completed_notebook_path else None
    if notebook_backup is not None:
        if not notebook_backup.is_file(): raise FileNotFoundError(notebook_backup)
        shutil.copy2(notebook_backup,root/notebook_backup.name)
    manifest=build_integrity_manifest(root,excluded)
    atomic_write_json(manifest_path,manifest)
    with tempfile.NamedTemporaryFile(dir=outdir,suffix='.zip',delete=False) as h: tmp=Path(h.name)
    excluded.add(tmp)
    try:
        with zipfile.ZipFile(tmp,'w',zipfile.ZIP_STORED,allowZip64=True) as z:
            for item in manifest['files']:
                compression=zipfile.ZIP_STORED if item['path'].endswith('.pt') else zipfile.ZIP_DEFLATED
                z.write(root/item['path'],item['path'],compress_type=compression)
            z.write(manifest_path,'manifests/recovery_integrity.json')
        report=verify_recovery_archive(tmp)
        if out.exists():
            verify_recovery_archive(out)
            with tempfile.NamedTemporaryFile(dir=outdir,suffix='.zip',delete=False) as h: prev_tmp=Path(h.name)
            try: shutil.copyfile(out,prev_tmp);os.replace(prev_tmp,previous)
            finally: prev_tmp.unlink(missing_ok=True)
        os.replace(tmp,out);report['path']=str(out)
        atomic_write_json(outdir/'recovery_validation.json',report)
        return out,report
    finally: tmp.unlink(missing_ok=True)
RECOVERY_ZIP,RECOVERY_REPORT=create_recovery_zip(ROOT)
print('Complete model/recovery ZIP:',RECOVERY_ZIP)

## 15. Final PASS/FAIL verification

In [ ]:
def final_verification(cfg, model=None, tokenizer_meta=None, official_ids=None):
    declarations = {
        "student_owned_architecture_attested": bool(cfg.student_owned_architecture_attestation),
        "student_configuration_confirmed": bool(cfg.student_configuration_confirmed),
        "trained_from_scratch_attested": bool(cfg.trained_from_scratch_attestation),
    }
    checks = {}
    model_validation = DIRS["manifests"] / "model_validation.json"
    report = json.loads(model_validation.read_text(encoding="utf-8")) if model_validation.exists() else {}
    checks["parameter_count_50m_to_300m"] = 50_000_000 <= int(report.get("total", -1)) <= 300_000_000
    checks["model_source_frozen"] = bool(
        MODEL_SOURCE_PATH and Path(MODEL_SOURCE_PATH).exists()
        and report.get("model_source_sha256") == sha256_file(MODEL_SOURCE_PATH)
    )
    training_result = DIRS["manifests"] / "training_result.json"
    training_config = DIRS["manifests"] / "training_config.json"
    checks["training_completed_10000_steps"] = bool(
        training_result.exists() and training_config.exists()
        and int(json.loads(training_result.read_text()).get("completed_step", -1)) == 10_000
        and int(json.loads(training_config.read_text()).get("max_steps", -1)) == 10_000
    )
    cached_report = DIRS["manifests"] / "cached_inference_validation.json"
    checks["cached_decoding_validated"] = bool(
        cached_report.exists() and json.loads(cached_report.read_text()).get("status") == "PASS"
    )
    tokenizer_path = DIRS["tokenizer"] / "tokenizer_metadata.json"
    checks["tokenizer_registered"] = tokenizer_path.exists()
    checks["tokenizer_integrity"] = bool(
        tokenizer_path.exists()
        and json.loads(tokenizer_path.read_text()).get("fingerprint") == directory_fingerprint(DIRS["tokenizer"])
    )
    checks["split_manifest_present"] = (DIRS["manifests"] / "document_split.json").exists()
    checks["validation_protocol_present"] = (DIRS["manifests"] / "validation_sequence_manifest.json").exists()
    checks["training_history_present"] = (DIRS["histories"] / "training.jsonl").exists()
    selected = DIRS["checkpoints"] / "selected_model.pt"
    final = DIRS["checkpoints"] / "model_final.pt"
    checks["selected_model_present"] = selected.exists()
    checks["final_model_present"] = final.exists()
    submission = DIRS["submission"] / "submission.zip"
    checks["submission_zip_valid"] = False
    checks["artifact_provenance_fresh"] = False
    if submission.exists() and official_ids is not None:
        try:
            with zipfile.ZipFile(submission) as archive:
                if archive.namelist() == ["prediction.json"]:
                    predictions = json.loads(archive.read("prediction.json"))
                    checks["submission_zip_valid"] = validate_predictions(predictions, official_ids, 128)["passed"]
            validation = json.loads((DIRS["submission"] / "submission_validation.json").read_text())
            provenance = validation.get("inference_provenance", {})
            checks["artifact_provenance_fresh"] = bool(
                selected.exists()
                and provenance.get("checkpoint_id") == f"selected-{sha256_file(selected)[:12]}"
                and provenance.get("model_source_sha256") == MODEL_SOURCE_SHA256
                and validation.get("prediction_sha256") == sha256_file(DIRS["submission"] / "prediction.json")
            )
        except Exception:
            pass
    recovery = DIRS["recovery"] / "phase1_submission03_recovery.zip"
    checks["recovery_zip_valid"] = False
    checks["recovery_is_fresh"] = False
    if recovery.exists():
        try:
            verify_recovery_archive(recovery); checks["recovery_zip_valid"] = True
            inputs = [path for path in (selected, submission) if path.exists()]
            checks["recovery_is_fresh"] = bool(inputs and recovery.stat().st_mtime >= max(path.stat().st_mtime for path in inputs))
        except Exception:
            pass
    artifact_status = "PASS" if all(checks.values()) else "FAIL"
    codabench_record = DIRS["manifests"] / "codabench_acceptance.json"
    accepted = False
    if codabench_record.exists():
        record = json.loads(codabench_record.read_text(encoding="utf-8"))
        accepted = bool(record.get("accepted") and record.get("submission_id"))
    outcome = {
        "artifact_status": artifact_status,
        "codabench_accepted": accepted,
        "completion_status": "COMPLETE" if artifact_status == "PASS" and accepted and all(declarations.values()) else "INCOMPLETE",
        "checks": checks,
        "declarations": declarations,
        "optional_local_evaluation_present": any(DIRS["evaluations"].glob("*.json")),
        "created_utc": utc_now(),
    }
    atomic_write_json(DIRS["manifests"] / "final_verification.json", outcome)
    print(json.dumps(outcome, indent=2))
    return outcome

# After Codabench accepts the upload, record truthful evidence, for example:
# atomic_write_json(DIRS['manifests']/'codabench_acceptance.json', {
#     'accepted': True, 'submission_id': 'REAL_ID', 'submitted_utc': utc_now(),
#     'result': {'bertscore_f1': 'REAL_VALUE', 'rouge_l_f1': 'REAL_VALUE', 'bleu_4': 'REAL_VALUE'}
# })
# Rebuild the recovery archive, then run:
FINAL_REPORT = final_verification(CFG, MODEL, TOKENIZER_META, [row['id'] for row in TEST_RECORDS])
print('Codabench submission ZIP:',SUBMISSION_ZIP)
print('Complete model/recovery ZIP:',RECOVERY_ZIP)


## Kaggle execution checklist for Submission 03

1. Keep Submissions 01 and 02 backed up. This notebook writes only under `.../submission_03`.
2. Attach the official four-source Phase 1 corpus/test data under `/kaggle/input`; confirm the schema
   report and that the official test file is excluded from training.
3. The run uses at most 8,000 documents per source and automatically builds or reopens the
   deterministic bounded cache; the original multi-GB corpus is not copied.
4. `CFG.tokenizer_import_path=None` and `RUN_TOKENIZER_TRAINING=True` train a fresh
   25,000-token byte-level BPE on `TRAIN_DOCS` if none is restored, then save it
   under the Submission 03 workspace, and automatically reuses that exact mapping on resume/inference.
   Never delete or change the tokenizer directory after checkpointing begins.
5. Attach `prefix_hybrid_lm.py`; the supplied implementation uses a 384-wide, 24-layer, 6-head RoPE
   prefix-hybrid Transformer with 52,206,376 trainable parameters at a 25k vocabulary. The runtime count
   and analytical count must match exactly before training.
6. Attach the official corpus/test data and the current `prefix_hybrid_lm.py`, select Kaggle T4 x2,
   and use Run All. The 10,000-step run starts fresh or resumes a matching checkpoint automatically.
   Training uses four examples per GPU and four accumulation passes (effective batch 32).
7. The separate GPU training smoke test is disabled. Automatic model checks still verify
   finite gradients, mask behavior, source hashing, and exact reconstruction before training.
8. Periodic `training_recovery.latest.zip` archives are written every 1,000 steps. Download one
   before a Kaggle session ends; attach it as input and rerun the same notebook to resume. The
   restore bootstrap validates per-member SHA-256 hashes and refuses conflicting files.
9. After training, a cached-decoding equivalence check runs before official generation. The
   notebook then writes validated `submission.zip` and `phase1_submission03_recovery.zip` automatically.
10. Upload `submission.zip` to Codabench and record the real acceptance ID. The course
    attestations remain false until personally reviewed and truthfully declared.

Submission 03 is complete only after the trained checkpoint is verified, artifact validation passes,
every official ID is valid, and Codabench acceptance is truthfully recorded.
